In [ ]:
import astropy
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from itables import show
import pprint

import itertools
from multiprocess import Pool
import multiprocess
from multiprocess import Manager
from threading import Thread

import os
import re
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map  # or thread_map
import numpy as np
from scipy.interpolate import interp1d
# # from scipy.optimize import UnivariateSpline
from scipy.interpolate import interp1d, UnivariateSpline
import matplotlib.pyplot as plt
from astropy.io import fits
from FitsClass import FITSFile as myfits
from ObservationClass import ObservationManager as obsm
from SimulationClass import Simulations as sc

import specs as specs
from CCF import CCFclass

obs = obsm()

In [ ]:
for i, star_name in enumerate(specs.star_names):
    print(f'{i} is {star_name}')

# Plotting spectra

## Plotting X-Shooter full spectra

In [ ]:
obs = obsm()
star_name = specs.star_names[12]
print(f'I am working on {star_name}')
star = obs.load_star_instance(star_name)

In [ ]:
%matplotlib widget
# star.plot_spectra([3],['UVB','VIS','NIR'],normalize=False,Rest_frame=False)
star.plot_spectra([3],['COMBINED'],normalize=False,Rest_frame=False,color_combined=True,log=True,add_continuum=True,add_RV_emission_lines=True)


## Plotting normalizied spectra

In [ ]:
import matplotlib
matplotlib.get_backend()


In [ ]:
%matplotlib widget
star_name = specs.star_names[20]
print(star_name)
star = obs.load_star_instance(star_name)
epoch_nums = [1,2,3,5,6]
epoch_nums = [1]
epoch_nums = None
EL_annotate = [5808]
# EL_annotate=True
star.plot_normalized_spectra(bands = 'COMBINED',compare=False,separation=0,bin_window=0,bary_correction=False,epoch_nums=epoch_nums,add_RV_emission_lines=True,
                             color_combined=True, add_continuum=True,for_presentation=False,EL_annotate=EL_annotate,separate=False)
# star.plot_normalized_spectrum(3,['UVB','VIS','NIR'])

In [ ]:
star_name = specs.star_names[1]
print(star_name)
star = obs.load_star_instance(star_name)
flux1 = star.load_property(epoch_num=1,band='COMBINED',property_name='cleaned_normalized_flux')['normalized_flux']
flux2 = star.load_property(epoch_num=2,band='COMBINED',property_name='cleaned_normalized_flux')['normalized_flux']
wave = star.load_property(epoch_num=1,band='COMBINED',property_name='cleaned_normalized_flux')['wavelengths']


window1 = flux1[(575 < wave)& (wave <595)]
window2 = flux2[(575 < wave)& (wave <595)]

plt.scatter(window1, window2)
plt.show()

In [ ]:
%matplotlib widget
star_name = specs.star_names[0]
print(star_name)
star = obs.load_star_instance(star_name)
star.plot_normalized_spectra(bands = 'COMBINED',compare=False,separation=0,bin_window=0,bary_correction=False,epoch_nums=[1])
# star.plot_normalized_spectrum(3,['UVB','VIS','NIR'])

In [ ]:
emission_lines = {
    'O V 3100-3175': [310., 317.5],
    'O IV 3350-3480': [335,348],
    'C IV 3650-3900': [365.,390],
    'He II 4686':  [456., 480.],
    'C IV 5808-5812': [570., 590],
    'O VI 5210-5340': [521,534],
    'C IV 5350-5540': [535,554],
    'C III 6700-6800': [667.,684.],
    'C III 7000- 7100': [700,714],
    # 'idk1': [1715,1755],
    # 'idk2': [2050,2110],
    # 'telurics1': [1340,1495],
    # 'telurics2': [1795,1985],
    'He I 17100-17650': [1710,1765],
    'Mystery 20500-21000': [2050,2100]
}

for i, star_name in enumerate(specs.star_names):
    print(f'\'{star_name}\':')
    print('{')
    for key in emission_lines.keys():
    # print(f'{i} is {star_name}')
        print(f'"{key}": ,')
    print('}')

In [ ]:
for i, star_name in enumerate(specs.star_names):
    print(f'{i} is {star_name}')

### Plot highest and lowest RVs fluxes

In [ ]:
# Air wavelengths in Å (μm shown in labels only for readability)
# Sources noted below; optical values are the standard air rest wavelengths.
O_STAR_ABS_LINES_AIR = [
    # ── Optical (≈3800–7100 Å) ───────────────────────────────────────────
    ("Hδ 4101.74", 4101.74),
    ("He II 4200", 4199.83),
    ("Hγ 4340.47", 4340.47),
    ("He I 4387.93", 4387.93),
    ("He I 4471.48", 4471.48),
    ("He II 4541.59", 4541.59),
    ("N III 4640", 4640.64),   # blend (4634–4641)
    ("He II 4686", 4685.68),
    ("He I 4713.15", 4713.15),
    ("Hβ 4861.33", 4861.33),
    ("He I 4921.93", 4921.93),
    ("He II 5411.52", 5411.52),
    ("O III 5592", 5592.26),
    # C IV “λ5808” = 5801.33 + 5811.98 (kept as split doublet for RV work)
    ("C IV 5801.33", 5801.33),
    ("C IV 5811.98", 5811.98),
    ("He I 5875.62", 5875.62),
    ("Hα 6562.80", 6562.80),
    ("He I 6678.15", 6678.15),
    ("He I 7065.18", 7065.18),

    # ── J band (≈1.0–1.35 μm) ────────────────────────────────────────────
    ("He II 1.0124 μm", 10123.6),
    ("He I 1.0830 μm (comp.)", 10829.09),  # triplet fine-structure components
    ("He I 1.0830 μm (comp.)", 10830.25),
    ("He I 1.0830 μm (comp.)", 10830.34),
    ("Paγ 1.0938 μm", 10938.1),
    ("Paβ 1.2818 μm", 12818.1),

    # ── H band (≈1.5–1.8 μm) ─────────────────────────────────────────────
    ("Br12 1.6412 μm", 16412.0),
    ("Br11 1.6811 μm", 16811.0),
    ("He II 1.693 μm", 16930.0),
    ("He I 1.700 μm", 17002.47),  # precise NIST value

    # ── K band (≈2.0–2.4 μm) ─────────────────────────────────────────────
    ("He I 2.0581 μm", 20581.3),
    ("C IV 2.069 μm", 20690.0),
    ("C IV 2.078 μm", 20780.0),
    ("C IV 2.083 μm", 20830.0),
    ("He I 2.1127 μm", 21127.0),
    ("He I 2.1138 μm", 21138.0),
    ("Brγ 2.166 μm", 21655.0),   # air; vacuum is 21661 Å
    ("He II 2.1885 μm", 21885.0),
    # Optional Paschen-α in the gap between H/K if you cover it:
    ("Paα 1.8751 μm", 18751.0),
]

O_STAR_UVB_EXTRA_LINES = [
    # ── Balmer upper members (blueward of Hγ) ─────────────────────
    ("Hκ 3750.15", 3750.15),
    ("Hι 3770.63", 3770.63),
    ("Hθ 3797.90", 3797.90),
    ("Hη 3835.38", 3835.38),
    ("Hζ 3889.05", 3889.05),     # blends with He I 3889
    ("Hε 3970.07", 3970.07),     # blends with Ca II H 3968.47

    # ── He I (UVB-rich multiplets) ─────────────────────────────────
    ("He I 3819.61", 3819.61),
    ("He I 3889.75", 3889.75),   # (triplet) near Hζ
    ("He I 3926.54", 3926.54),
    ("He I 3964.73", 3964.73),
    ("He I 4009.26", 4009.26),
    ("He I 4026.19", 4026.19),
    ("He I 4120.82", 4120.82),
    ("He I 4143.76", 4143.76),
    ("He I 4168.97", 4168.97),
    ("He I 4387.93", 4387.93),
    ("He I 4437.55", 4437.55),
    ("He I 4471.48", 4471.48),
    ("He I 4713.15", 4713.15),
    ("He I 4921.93", 4921.93),
    ("He I 5015.68", 5015.68),
    ("He I 5047.74", 5047.74),

    # ── He II (blue/UVB + one Pickering near Hβ) ──────────────────
    ("He II 4200.47", 4200.47),
    ("He II 4541.59", 4541.59),
    ("He II 4685.68", 4685.68),
    ("He II 4859.32 (Pick.)", 4859.32),  # blends with Hβ 4861.33

    # ── Silicon ────────────────────────────────────────────────────
    ("Si IV 4088.86", 4088.86),
    ("Si IV 4116.10", 4116.10),
    ("Si II 3856.02", 3856.02),
    ("Si II 3862.60", 3862.60),
    ("Si II 4128.05", 4128.05),
    ("Si II 4130.89", 4130.89),
    ("Si III 4552.62", 4552.62),
    ("Si III 4567.84", 4567.84),
    ("Si III 4574.76", 4574.76),
    ("Si III 4813.33", 4813.33),
    ("Si III 4819.72", 4819.72),
    ("Si III 4829.07", 4829.07),

    # ── Oxygen (O II forest) ───────────────────────────────────────
    ("O II 4072.16", 4072.16),
    ("O II 4075.86", 4075.86),
    ("O II 4317.14", 4317.14),
    ("O II 4319.63", 4319.63),
    ("O II 4349.43", 4349.43),
    ("O II 4351.26", 4351.26),
    ("O II 4366.89", 4366.89),
    ("O II 4414.90", 4414.90),
    ("O II 4416.98", 4416.98),
    ("O II 4661.63", 4661.63),
    ("O II 4676.24", 4676.24),

    # ── Nitrogen ───────────────────────────────────────────────────
    ("N II 3995.00", 3995.00),
    ("N II 4041.31", 4041.31),
    ("N II 4043.53", 4043.53),
    ("N III 4097.33", 4097.33),
    ("N III 4510.89", 4510.89),
    ("N III 4514.85", 4514.85),
    ("N III 4518.15", 4518.15),
    ("N II 4601.48", 4601.48),
    ("N II 4607.16", 4607.16),
    ("N II 4613.87", 4613.87),
    ("N II 4621.39", 4621.39),
    ("N II 4630.54", 4630.54),
    ("N II 4643.08", 4643.08),

    # ── Carbon ─────────────────────────────────────────────────────
    ("C II 3918.97", 3918.97),
    ("C II 3920.69", 3920.69),
    ("C II 4267.00", 4267.00),
    ("C III 4070.26", 4070.26),
    ("C III 4152.52", 4152.52),
    ("C III 4162.88", 4162.88),
    ("C III 4186.90", 4186.90),
    ("C III 4516.77", 4516.77),
    ("C III 4647.42", 4647.42),
    ("C III 4650.25", 4650.25),
    ("C III 4651.47", 4651.47),

    # ── Magnesium (late-O / early-B discriminator) ─────────────────
    ("Mg II 4481.13", 4481.13),

    # ── Optional ISM anchors in the blue end ───────────────────────
    ("Ca II K 3933.66", 3933.66),
    ("Ca II H 3968.47", 3968.47),
]

TARGET_UVB_LINES = [
    # 3890 region: Balmer Hζ + He I blend
    ("H I Hζ 3889.05", 3889.05),
    ("He I 3889.75",   3889.75),

    # 4026 He I
    ("He I 4026.19",   4026.19),

    # 4100 region: Hδ (and a nearby helper if you want it)
    ("H I Hδ 4101.74", 4101.74),
    ("N III 4097.33",  4097.33),    # optional context line

    # 4200 He II
    ("He II 4200.47",  4200.47),

    # 4341 region: Hγ (air + vacuum alias if you like to see both)
    ("H I Hγ 4340.47", 4340.47),
    ("H I Hγ (vac) 4341.68", 4341.68),  # optional alias label

    # --- ADDED: H-beta is often the red-end anchor for the UVB arm ---
    ("H I Hβ 4861.33", 4861.33),
]


# Updated species_filter
species_filter=("H I", "Hα", "Hβ", "Hγ", "Hδ", "Hε", 
                "He I", "He II", "O II", "O III", "C II", 
                "C III", "C IV", "N II", "N III", "Si II", 
                "Si III", "Si IV", "Mg II", "Ca II")

In [ ]:
%matplotlib widget
models = [
    'BG20000g300v2.vis.rectvmac30vsini25.dat',
    # 'G32500g325v10.uv.rectvmac20vsini100.dat',
    'G32500g325v10.vis.rect',
    'G40000g450v10.vis.rectvmac30vsini200.dat']
O_STAR_ABS_LINES_AIR_EXT = (O_STAR_ABS_LINES_AIR + O_STAR_UVB_EXTRA_LINES + TARGET_UVB_LINES)
star_name = specs.star_names[5]
print(star_name)
star = obs.load_star_instance(star_name)
emission_line = 'C III 6700-6800'
emission_line = 'C IV 5808-5812'
# emission_line = 'He II 4686'
star.plot_extreme_rv_spectra(emission_line = emission_line,to_plot=True,models = models,correct_lmc=True,species_filter=species_filter, auto_annotate=True,even_emission_line=False,line_list = O_STAR_ABS_LINES_AIR_EXT)

In [ ]:
%matplotlib widget
models = [
    'BG20000g300v2.vis.rectvmac30vsini25.dat',
    # 'G32500g325v10.uv.rectvmac20vsini100.dat',
    'G32500g325v10.vis.rect',
    'G40000g450v10.vis.rectvmac30vsini200.dat']
O_STAR_ABS_LINES_AIR_EXT = (O_STAR_ABS_LINES_AIR + O_STAR_UVB_EXTRA_LINES + TARGET_UVB_LINES)
star_name = specs.star_names[24]
print(star_name)
star = obs.load_star_instance(star_name)
emission_line = 'C III 6700-6800'
emission_line = 'C IV 5808-5812'
# emission_line = 'He II 4686'
star.plot_extreme_rv_spectra(emission_line = emission_line,to_plot=True,models = models,correct_lmc=True,species_filter=species_filter, auto_annotate=True,even_emission_line=False,line_list = O_STAR_ABS_LINES_AIR_EXT)

### Plotting backuped up spectra vs new spectra

In [ ]:
%matplotlib widget
plt.clf()
star_name = specs.star_names[2]
print(star_name)
star = obs.load_star_instance(star_name)
spectra = star.load_property(epoch_num=1,band='COMBINED',property_name='cleaned_normalized_flux')
new_flux = (spectra['normalized_flux']-1)*16/7 +1
new_wave = star.load_property(epoch_num=1,band='COMBINED',property_name='cleaned_normalized_flux')['wavelengths']
old_spectra = star.load_property(epoch_num=1,band='COMBINED',property_name='cleaned_normalized_flux',from_backup=True)
old_flux = old_spectra['normalized_flux']
old_wave = old_spectra['wavelengths']
plt.plot(old_wave,old_flux,label='old')
plt.plot(new_wave,new_flux,label='new',alpha=0.7)
plt.legend()
plt.show()

### Preview SNR snitch for cleaned normalized flux

In [ ]:
%matplotlib widget
star.preview_snr_stitch_cleaned_normalized(1)
# data = star.load_property('cleaned_normalized_flux',4,'NIR')
# flux = data['normalized_flux']
# wave = data['wavelengths']
# print(len(flux))
# print(len(wave))

## Plotting NRES spectra

### plotting normalized spectra

In [ ]:
%matplotlib widget
star_names = ['WR 52','WR17']
star_name = star_names[1]
star = obs.load_star_instance(star_name)
spectra_list = [1,2,3,4,5,6,7,8,9,10,11,12,13,14]
# spectra_list = [1,2,3]
star.plot_normalized_spectra(1,spectra_list,bin_window=20,clean=True,initial_separation=0)
# star_name = star_names[0]
# star = obs.load_star_instance(star_name)
# star.plot_normalized_spectra([1,2,3],[1,3])
# star.plot_normalized_spectra2(1,1)

### Plotting raw spectra

In [ ]:
%matplotlib widget
star = obs.load_star_instance('WR17')
plt.clf()
normalized_data = star.load_property('nornormalized_data')
wave = normalized_data

### plotting SNR

In [ ]:
%matplotlib widget

# star.list_available_properties()
star = obs.load_star_instance(star_names[0])
combined_flux = star.load_property('combined_flux',1,1)
wave = combined_flux['wavelength']
flux = combined_flux['flux']
print(combined_flux)
SNR = combined_flux['SNR']
plt.clf()
plt.plot(wave,SNR)
plt.title(f'SNR of {star_names[0]}, epoch 1 spectra 1')
plt.xlabel('Wavelengh (Angstrom)')
plt.ylabel(f'SNR')
plt.show()

In [ ]:
%matplotlib widget
wave,flux,snr = star.plot_stitched_spectra(1,12,my_SNR = True,plot_SNR=True,window_size = 20)

In [ ]:
%matplotlib widget
data = star.load_observation(1,12).data
flux = np.flip(data['flux'])
blaze = np.flip(data['blaze'])
wave = np.flip(data['wavelength'])
print(wave[1][10:19])
start = 33
plt.clf()
plt.plot(wave[start + 1],flux[start + 1]/blaze[start + 1])
plt.plot(wave[start + 3],flux[start + 3]/blaze[start + 3])
plt.plot(wave[start + 5],flux[start + 5]/blaze[start + 5])
plt.plot(wave[start + 7],flux[start + 7]/blaze[start + 7])
plt.plot(wave[start + 17],flux[start + 17]/blaze[start + 17])
plt.plot(wave[start + 27],flux[start + 27]/blaze[start + 27])
plt.plot(wave[start + 37],flux[start + 37]/blaze[start + 37])
plt.plot(wave[start + 38],flux[start + 38]/blaze[start + 38])
plt.plot(wave[start + 39],flux[start + 39]/blaze[start + 39])
plt.plot(wave[start + 40],flux[start + 40]/blaze[start + 40])
plt.plot(wave[start + 41],flux[start + 41]/blaze[start + 41])
plt.plot(wave[start + 42],flux[start + 42]/blaze[start + 42])
plt.plot(wave[start + 43],flux[start + 43]/blaze[start + 43])
# plt.plot(wave[start + 44],flux[start + 44]/blaze[start + 44])
plt.plot(wave[start + 45],flux[start + 45]/blaze[start + 45])
# plt.plot(wave[start + 46],flux[start + 46]/blaze[start + 46])
plt.plot(wave[start + 47],flux[start + 47]/blaze[start + 47])
plt.plot(wave[start + 49],flux[start + 49]/blaze[start + 49])
# plt.plot(wave[1],flux[1])
# plt.plot(wave[3],flux[3])
plt.show()

In [ ]:
%matplotlib widget

for_presentation = True  # <-- turn on/off big text for slides

import numpy as np
import matplotlib.pyplot as plt

if for_presentation:
    plt.rcParams.update({
        "figure.figsize": (7, 6.5),
        "font.size": 16,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "lines.linewidth": 2.0,
    })

data = star.load_observation(1, 12).data
flux = np.flip(data["flux"])
blaze = np.flip(data["blaze"])
wave = np.flip(data["wavelength"])

print(wave[1][10:19])

start = 33
plt.clf()

order_offsets = [1, 3, 5, 7, 17, 27, 37, 38, 39, 40, 41, 42, 43, 45, 47, 49]
for off in order_offsets:
    i = start + off
    plt.plot(wave[i], flux[i] / blaze[i])

plt.xlabel(r"Wavelength [$\AA$]")
plt.ylabel("Flux (counts)")  # (you are plotting flux/blaze, i.e. blaze-corrected)
plt.tight_layout()
plt.show()


In [ ]:
%matplotlib widget
blaze_correction = True
star.plot_raw_spectra(1,12,blaze_correction=blaze_correction,subtract_sky=True,just_sky = False,just_target=True)
emissions = [(4550,4800),(5700,5900)]
plt.xlim(emissions[1])
if blaze_correction:
    plt.ylim((-1e7,0.5e7))
else:
    plt.ylim((-100,1000))

In [ ]:
obs = obsm()
star_name = specs.star_names[0]
star = obs.load_star_instance(star_name)
norm = star.load_property('normalized_flux',1,'COMBINED')
print(norm)
points = star.load_property('norm_anchor_wavelengths',1,'COMBINED')
print(points)

# Plotting RV binary analasys data

## analysing the correlations of the results of each emission line

### old

In [ ]:
%matplotlib widget
# === ΔRV Analysis: Full Dashboard (User-Defined Weighting Logic) ===

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import matplotlib.ticker as ticker
from matplotlib.markers import MarkerStyle
import textwrap
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import pearsonr

# ---------------- 1. SETUP & CONFIG ----------------
SETTINGS_PATH = "ccf_settings_with_global_lines.json"
try:
    with open(SETTINGS_PATH, "r") as f:
        SETTINGS = json.load(f)
except FileNotFoundError:
    print(f"WARNING: Could not find {SETTINGS_PATH}. Using empty settings.")
    SETTINGS = {}

STAR_CFG = {s["star_name"]: s for s in SETTINGS.get("stars", [])}
LINES_DEFAULT = SETTINGS.get("emission_lines_default", {}) 

# --- PRESENTATION MODE ---
for_presentation = True

# Config
CIV_KEY = "C IV 5808-5812"
THRESH_KMS = 10.0
MAX_RV_THRESH = 400 
EXTRA_ABOVE = 3  
BAR_FIGSIZE = (12, 6)
FIGSIZE_FRAC = (12, 10) # Increased height for 3 subplots
FIGSIZE_THRESH_PLOT = (10, 5)
FIGSIZE_AGREEMENT = (12, 7)
FIGSIZE_FBIN_BAR = (12, 6)
FAIL_FRAC_THRESH = 0.40
MIN_STARS_FOR_CORR = 8   
MAX_POSSIBLE_STARS = 25  
KNOWN_ERR_KEYS = ("full_RV_err", "full_err", "sigma", "err", "RV_err", "RV_sigma")
MEAN_COL = "Mean ΔRV"
STD_COL  = "Std ΔRV"

# Globals for storage
epoch_ew_data = {}        
ew_fail_stats = {}    
rv_epoch_cache = {} 
drverr_map = {} 
equiv_thresholds_map = {} 

# ---------------- 2. DATA PROCESSING HELPERS ----------------

def _is_clean_star(star):
    try:
        for ep in star.get_all_epoch_numbers():
            flux = star.load_property('cleaned_normalized_flux', ep, 'COMBINED')
            if flux is not None: 
                return False 
        return True 
    except Exception: 
        return True 

def _ew_record_for(star, epoch_num, line_key):
    try:
        EWs = star.load_property('EWs', epoch_num, 'COMBINED')
    except Exception: return None
    if EWs is None: return None
    try: rec = EWs.get(line_key)
    except Exception: return None
    if rec is None: return None
    try: rec_dict = rec.item()
    except Exception: return None
    if not isinstance(rec_dict, dict): return None

    def _to_float(x):
        try: return float(x)
        except Exception: return np.nan

    return {
        "EW": _to_float(rec_dict.get("EW")),
        "sigma_EW": _to_float(rec_dict.get("sigma_EW")),
        "SNR": _to_float(rec_dict.get("SNR")),
        "detected": rec_dict.get("detected")
    }

def _extract_full_rv(cell):
    try:
        if isinstance(cell, dict): return cell.get('full_RV', None)
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict): return v.get('full_RV', None)
        if hasattr(cell, "get"): return cell.get('full_RV', None)
    except Exception: pass
    return None

def _extract_full_rv_err(cell):
    try:
        if isinstance(cell, dict):
            for k in KNOWN_ERR_KEYS:
                if k in cell and cell[k] is not None: return float(cell[k])
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict):
                for k in KNOWN_ERR_KEYS:
                    if k in v and v[k] is not None: return float(v[k])
        if hasattr(cell, "get"):
            for k in KNOWN_ERR_KEYS:
                val = cell.get(k, None)
                if val is not None: return float(val)
    except Exception: pass
    return np.nan

def _is_epoch_skipped_for_line(star_name, line_key, ep):
    cfg = STAR_CFG.get(star_name, {})
    if ep in set(cfg.get("skip_epochs", [])): return True
    sl = cfg.get("skip_emission_lines", {})
    if line_key in sl:
        skip_eps = sl[line_key]
        if isinstance(skip_eps, (int, np.integer)): skip_eps = [skip_eps]
        if 0 in skip_eps or ep in skip_eps: return True
        return False

def _line_delta_rv_for_star(star, line_key):
    rv_vals, cache = [], []
    sname = star.star_name

    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep):
            cache.append((ep, np.nan, np.nan, False))
            continue

        ewrec = _ew_record_for(star, ep, line_key)
        if ewrec is not None:
            epoch_ew_data[(sname, line_key, ep)] = ewrec

        RVs = star.load_property('RVs', ep, 'COMBINED')
        if not RVs or line_key not in RVs:
            cache.append((ep, np.nan, np.nan, False))
            continue

        cell = RVs[line_key]
        rv = _extract_full_rv(cell)

        if rv is None:
            cache.append((ep, np.nan, np.nan, False))
            continue

        try: rv = float(rv)
        except Exception:
            cache.append((ep, np.nan, np.nan, False))
            continue

        err = _extract_full_rv_err(cell)
        rv_vals.append((ep, rv, err))
        cache.append((ep, rv, err, True))

    rv_epoch_cache[(sname, line_key)] = cache

    if len(rv_vals) == 0:
        drverr_map[(sname, line_key)] = np.nan
        return np.nan

    ep_min, rv_min, err_min = min(rv_vals, key=lambda t: t[1])
    ep_max, rv_max, err_max = max(rv_vals, key=lambda t: t[1])
    dRV = abs(rv_max - rv_min)

    if np.isfinite(err_min) and np.isfinite(err_max):
        sigma_A = float(np.sqrt(err_min**2 + err_max**2))
    else:
        sigma_A = np.nan

    drverr_map[(sname, line_key)] = sigma_A
    return dRV

def _is_significant_binary(star_name: str, line_key: str, drv_val: float, threshold_val: float) -> bool:
    if not (pd.notna(drv_val) and np.isfinite(drv_val)): return False
    sigma_A = drverr_map.get((star_name, line_key), np.nan)
    if not np.isfinite(sigma_A): return False
    return (float(drv_val) >= threshold_val) and (float(drv_val) >= 4.0 * float(sigma_A))

def _compute_ew_fail_stats(star, line_key):
    sname = star.star_name
    considered = []
    failed = 0
    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep): continue
        considered.append(ep)
        rec = epoch_ew_data.get((sname, line_key, ep))
        if isinstance(rec, dict):
            det = rec.get("detected")
            if det is False: failed += 1
    n_considered = len(considered)
    frac = (failed / n_considered) if n_considered > 0 else 0.0
    return failed, n_considered, frac

# ---------------- 3. LOAD DATA & BUILD DATAFRAME ----------------
rows = []
ordered_lines = list(LINES_DEFAULT.keys())

for star_name in specs.star_names:
    star = obs.load_star_instance(star_name, to_print=False)
    is_clean_bool = _is_clean_star(star)
    row = {"Star": star_name, "Clean": "✓" if is_clean_bool else "X", "is_clean_bool": is_clean_bool}
    
    for lk in ordered_lines: 
        dval = _line_delta_rv_for_star(star, lk)
        ew_fail_stats[(star_name, lk)] = _compute_ew_fail_stats(star, lk)
        row[f"dRV | {lk}"] = dval
    drvs = [v for k, v in row.items() if isinstance(k, str) and k.startswith("dRV | ")]
    row[MEAN_COL] = float(np.nanmean(drvs)) if np.isfinite(np.nanmean(drvs)) else np.nan
    row[STD_COL]  = float(np.nanstd(drvs))  if np.isfinite(np.nanstd(drvs))  else np.nan
    rows.append(row)

df = pd.DataFrame(rows)

# ---------------- 4. CALCULATE THRESHOLDS ----------------
def calculate_equivalent_thresholds(df_in):
    if CIV_KEY not in ordered_lines: return
    ref_col = f"dRV | {CIV_KEY}"
    civ_vals = []
    
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        _, _, frac = ew_fail_stats.get((sname, CIV_KEY), (0,0,0))
        if frac > FAIL_FRAC_THRESH: continue
        val = df_in.at[i, ref_col]
        if pd.notna(val) and np.isfinite(float(val)):
            civ_vals.append((sname, float(val)))
            
    n_above = 0
    for s, v in civ_vals:
        if _is_significant_binary(s, CIV_KEY, v, 20.0):
            n_above += 1
    
    num = n_above + EXTRA_ABOVE
    den = len(civ_vals) + EXTRA_ABOVE
    f_ref = num / den if den > 0 else 0
    
    equiv_thresholds_map.clear()
    search_space = np.arange(1, MAX_RV_THRESH + 1, 1)
    
    for lk in ordered_lines:
        col = f"dRV | {lk}"
        if col not in df_in.columns: continue
        line_vals = []
        for i in range(len(df_in)):
            sname = df_in.at[i, "Star"]
            _, _, frac = ew_fail_stats.get((sname, lk), (0,0,0))
            if frac > FAIL_FRAC_THRESH: continue
            val = df_in.at[i, col]
            if pd.notna(val) and np.isfinite(float(val)):
                line_vals.append((sname, float(val)))
        
        total_line = len(line_vals)
        if total_line == 0:
            equiv_thresholds_map[lk] = np.nan
            continue
            
        best_T = 20.0
        min_diff = 999.0
        
        for T in search_space:
            cnt = 0
            for s, v in line_vals:
                if _is_significant_binary(s, lk, v, T): cnt += 1
            f_curr = (cnt + EXTRA_ABOVE) / (total_line + EXTRA_ABOVE)
            diff = abs(f_curr - f_ref)
            if diff < min_diff:
                min_diff = diff
                best_T = float(T)
        equiv_thresholds_map[lk] = best_T

calculate_equivalent_thresholds(df)

ordered_cols = ["Star", "Clean", MEAN_COL, STD_COL] + [f"dRV | {lk}" for lk in ordered_lines if f"dRV | {lk}" in df.columns]
if f"dRV | {CIV_KEY}" in df.columns:
    df = df.sort_values(f"dRV | {CIV_KEY}", ascending=False, na_position="last").reset_index(drop=True)

# ---------------- 5. HELPER FOR CLASSIFICATION & FITTING ----------------
def get_star_classification_data(df_in, use_equiv):
    results = {}
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin": continue
        is_clean = bool(df_in.at[i, "is_clean_bool"])
        votes_bin = 0
        votes_single = 0
        civ_status = None
        valid_lines = 0
        for lk in ordered_lines:
            _, _, frac = ew_fail_stats.get((sname, lk), (0,0,0))
            if frac > FAIL_FRAC_THRESH: continue
            col_name = f"dRV | {lk}"
            if col_name not in df_in.columns: continue
            val = df_in.at[i, col_name]
            if pd.isna(val) or not np.isfinite(float(val)): continue
            t_use = THRESH_KMS
            if use_equiv:
                t_eq = equiv_thresholds_map.get(lk, np.nan)
                if np.isfinite(t_eq): t_use = t_eq
            is_bin = _is_significant_binary(sname, lk, float(val), t_use)
            if is_bin: votes_bin += 1
            else: votes_single += 1
            if lk == CIV_KEY: civ_status = is_bin
            valid_lines += 1
        if valid_lines == 0: continue
        agreement = max(votes_bin, votes_single) / valid_lines
        if agreement > 0.90: grade = 'Golden'
        elif agreement > 0.70: grade = 'Silver'
        else: grade = 'Bronze'
        if votes_bin > votes_single: final_bin = True
        elif votes_single > votes_bin: final_bin = False
        else: final_bin = civ_status if civ_status is not None else False
        results[sname] = {'grade': grade, 'is_binary': final_bin, 'is_clean': is_clean}
    return results

# === Chi-Squared 2-Segment Linear Fit ===
def _fit_two_segment_linear_weighted(x, y, y_err):
    if len(x) < 5: return None 
    w = 1.0 / (y_err**2)
    w[~np.isfinite(w)] = 0.0001
    best_chi2 = np.inf
    best_res = None
    for i in range(2, len(x) - 2):
        x1, y1, w1 = x[:i], y[:i], w[:i]
        x2, y2, w2 = x[i:], y[i:], w[i:]
        try:
            p1 = np.polyfit(x1, y1, 1, w=np.sqrt(w1)) 
            fit1 = np.polyval(p1, x1)
            chi2_1 = np.sum(w1 * (y1 - fit1)**2)
        except:
            chi2_1 = np.inf
            p1 = [0,0]
        try:
            p2 = np.polyfit(x2, y2, 1, w=np.sqrt(w2))
            fit2 = np.polyval(p2, x2)
            chi2_2 = np.sum(w2 * (y2 - fit2)**2)
        except:
            chi2_2 = np.inf
            p2 = [0,0]
        total_chi2 = chi2_1 + chi2_2
        if total_chi2 < best_chi2:
            best_chi2 = total_chi2
            m1, c1 = p1
            m2, c2 = p2
            if abs(m1 - m2) > 1e-9:
                elbow_x = (c2 - c1) / (m1 - m2)
                elbow_y = m1 * elbow_x + c1
            else:
                elbow_x = x[i]
                elbow_y = y[i]
            best_res = {
                'elbow_x': elbow_x, 'elbow_y': elbow_y,
                'p1': p1, 'p2': p2, 'break_idx': i,
                'range1': (x[0], elbow_x),
                'range2': (elbow_x, x[-1]),
                'chi2': best_chi2
            }
    return best_res

# ---------------- 6. PLOTTING LOGIC ----------------

def wrap_header(colname: str) -> str:
    if colname in ("Star", "Clean", MEAN_COL, STD_COL): return colname
    return "\n".join(textwrap.wrap(colname.replace("dRV | ", "dRV\n"), width=12, break_long_words=True))

def wrap_str(s, width):
    if pd.isna(s): return s
    return "\n".join(textwrap.wrap(str(s), width=width, break_long_words=True)) or str(s)

def wilson_score_interval(k, n, z=1.0):
    if n == 0: return 0.0, 0.0
    p = k / n
    denominator = 1 + z**2/n
    center_adjusted_probability = (p + z**2 / (2*n)) / denominator
    error_margin = (z * np.sqrt((p*(1-p)/n) + (z**2/(4*n**2)))) / denominator
    low = center_adjusted_probability - error_margin
    high = center_adjusted_probability + error_margin
    return max(0.0, low), min(1.0, high)

# === UPDATED: Returns (frac, err_low, err_high) for internal logic ===
def _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh):
    colname = f"dRV | {line_key}"
    if colname not in base_with.columns: return 0.0, 0.0, 0.0
    thresh_to_use = THRESH_KMS
    if use_equiv_thresh:
        t = equiv_thresholds_map.get(line_key, THRESH_KMS)
        if np.isfinite(t): thresh_to_use = t

    vals = base_with[colname]
    idx_with_drv = []
    for i in range(0, len(base_with)):
        star_name = base_with.at[i, "Star"]
        if not isinstance(star_name, str) or star_name == "f_bin": continue
        _, _, frac = ew_fail_stats.get((star_name, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH: continue
        try:
            v = vals.iat[i]
            if pd.notna(v) and np.isfinite(float(v)): idx_with_drv.append(i)
        except Exception: pass

    n_total = len(idx_with_drv)
    if n_total == 0: return 0.0, 0.0, 0.0
    n_above = 0
    for i in idx_with_drv:
        star_name = base_with.at[i, "Star"]
        dval = float(vals.iat[i])
        if _is_significant_binary(star_name, line_key, dval, thresh_to_use): n_above += 1
        
    n_total_adj = n_total + EXTRA_ABOVE
    n_above_adj = n_above + EXTRA_ABOVE
    frac_adj = n_above_adj / n_total_adj
    
    low, high = wilson_score_interval(n_above_adj, n_total_adj, z=1.0)
    err_low = frac_adj - low
    err_high = high - frac_adj
    return frac_adj, err_low, err_high, n_above_adj, n_total_adj

# === UPDATED: Returns formatted string WITH errors ===
def _get_fbin_simple(base_with, line_key, use_equiv_thresh):
    frac, el, eh, num, den = _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh)
    if den == 0: return "0/0 (—)"
    # Format: 5/10 (50% +10/-5)
    return f"{num}/{den} ({frac:.0%} +{eh:.0%}/-{el:.0%})"

def _get_fbin_float(base_with, line_key, use_equiv_thresh):
    f, _, _, _, _ = _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh)
    return f * 100.0

def _calc_ew_stats_for_line(line_key):
    failed = 0
    considered = 0
    all_ews = []
    
    for sname in STAR_CFG.keys():
        n_fail, n_cons, _ = ew_fail_stats.get((sname, line_key), (0, 0, 0))
        failed += n_fail
        considered += n_cons
    
    for (sname, lk, ep), rec in epoch_ew_data.items():
        if lk == line_key and isinstance(rec, dict):
            val = rec.get("EW")
            if val is not None and np.isfinite(val):
                all_ews.append(val)

    if considered > 0:
        success_rate = 1.0 - (failed / considered)
    else:
        success_rate = 0.0
        
    if len(all_ews) > 0:
        mean_ew = np.nanmean(all_ews)
        sem_ew = np.nanstd(all_ews) / np.sqrt(len(all_ews))
    else:
        mean_ew = np.nan
        sem_ew = np.nan
        
    return success_rate, mean_ew, sem_ew

def _get_wavelength(name):
    match = re.search(r"(\d+)", name)
    if match:
        return float(match.group(1))
    return 999999.0

def render_plots(sort_col, ascending):
    # --- STYLE SETTINGS ---
    if for_presentation:
        plt.rcParams.update({
            'font.size': 14,
            'axes.titlesize': 16,
            'axes.labelsize': 14,
            'xtick.labelsize': 12,
            'ytick.labelsize': 12,
            'legend.fontsize': 12
        })
        FS_NOTE = 12
        FS_TINY = 11
        FS_AXIS_LBL = 14
        FS_BAR_VAL = 12
        FS_BOX_STAT = 8
        FS_PIE_LABEL = 14
        FS_LIST_TEXT = 11
    else:
        plt.rcParams.update({
            'font.size': 10,
            'axes.titlesize': 12,
            'axes.labelsize': 10,
            'xtick.labelsize': 10,
            'ytick.labelsize': 10,
            'legend.fontsize': 10
        })
        FS_NOTE = 9
        FS_TINY = 8
        FS_AXIS_LBL = 10
        FS_BAR_VAL = 9
        FS_BOX_STAT = 7
        FS_PIE_LABEL = 11
        FS_LIST_TEXT = 9

    # Masking logic
    base = df.copy()[ordered_cols]
    masked = base.copy()
    for col in [c for c in masked.columns if c.startswith("dRV | ")]:
        line_key = col.replace("dRV | ", "")
        for i in range(len(masked)):
            sname = masked.at[i, "Star"]
            if not isinstance(sname, str) or sname == "f_bin": continue
            n_fail, n_cons, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
            if (n_cons > 0) and (frac > FAIL_FRAC_THRESH):
                if not state.get("force_show_low_detection", False):
                    masked.at[i, col] = np.nan

    sort_col_use = sort_col if sort_col in masked.columns else masked.columns[2]
    masked = masked.sort_values(sort_col_use, ascending=ascending, na_position="last").reset_index(drop=True)
    use_equiv = state.get("use_equiv_thresh", False)

    # --- PLOTS 1-6 ---
    vals = masked[sort_col_use].to_numpy()
    x = np.arange(len(masked))
    line_key_bar = sort_col_use.replace("dRV | ", "") if sort_col_use.startswith("dRV | ") else None
    current_thresh = THRESH_KMS
    if use_equiv and line_key_bar:
        t = equiv_thresholds_map.get(line_key_bar, THRESH_KMS)
        if np.isfinite(t): current_thresh = t
    colors = []
    yerrs = []
    for i in range(len(masked)):
        star_name = masked.at[i, "Star"]
        val = vals[i]
        if pd.isna(val): 
            colors.append("gray"); yerrs.append(0)
        elif line_key_bar: 
            is_sig = _is_significant_binary(star_name, line_key_bar, float(val), current_thresh)
            colors.append("green" if is_sig else "red")
            err = drverr_map.get((star_name, line_key_bar), 0)
            yerrs.append(err if np.isfinite(err) else 0)
        else: 
            colors.append("green" if val >= current_thresh else "red"); yerrs.append(0)
    plt.figure(figsize=BAR_FIGSIZE)
    plt.plot(x, vals, color='black', alpha=0.2, zorder=1)
    for i in range(len(x)):
        plt.errorbar(x[i], vals[i], yerr=yerrs[i], fmt='o', color=colors[i], ecolor=colors[i], capsize=3, zorder=2)
    plt.axhline(current_thresh, linestyle="--", linewidth=1, color='gray', label=f"Threshold {current_thresh:.1f} km/s")
    plt.xticks(x, [wrap_str(s, 10) for s in masked["Star"]], rotation=45, ha="right")
    plt.ylabel("|ΔRV| (km/s)")
    
    use_log = state.get("corner_log_scale", True)
    if use_log:
        plt.yscale('log')
        valid_pos = [v for v in vals if pd.notna(v) and v > 0]
        if valid_pos:
            plt.ylim(min(valid_pos)*0.5, max(valid_pos)*2.0)

    plt.title(f"Peak-to-peak |ΔRV| for {sort_col_use.replace('dRV | ', '')} ({'asc' if ascending else 'desc'})")
    plt.tight_layout()
    plt.show()

    # =========================================================
    #    MODIFIED SECTION: Binary Fraction vs Threshold + Residuals + 2nd Deriv
    # =========================================================
    if line_key_bar:
        valid_indices = []
        for i in range(len(masked)):
            sname = masked.at[i, "Star"]
            _, _, frac = ew_fail_stats.get((sname, line_key_bar), (0, 0, 0.0))
            if frac > FAIL_FRAC_THRESH: continue
            v = masked.at[i, sort_col_use]
            if pd.notna(v) and np.isfinite(float(v)): valid_indices.append(i)
        
        # 1. Calculate Full Resolution first
        t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1)
        fractions_full = []
        err_lows_full = []
        err_highs_full = []
        
        for t in t_vals_full:
            n_pass = 0
            for idx in valid_indices:
                sname = masked.at[idx, "Star"]
                dval = float(masked.at[idx, sort_col_use])
                if _is_significant_binary(sname, line_key_bar, dval, t): n_pass += 1
            num = n_pass + EXTRA_ABOVE
            den = len(valid_indices) + EXTRA_ABOVE
            f = num / den if den > 0 else 0
            
            l, h = wilson_score_interval(num, den, z=1.0)
            fractions_full.append(f)
            err_lows_full.append(f - l)
            err_highs_full.append(h - f)

        # 2. FILTERING LOGIC CONTROLLED BY TICK
        if state.get("filter_threshold_plot", True):
            final_t, final_f, final_el, final_eh = [], [], [], []
            prev_f = -1.0
            
            for k in range(len(fractions_full)):
                curr_f = fractions_full[k]
                # Logic: Keep if it's the first point, OR if the value changed from previous
                if k == 0 or abs(curr_f - prev_f) > 1e-9:
                    final_t.append(t_vals_full[k])
                    final_f.append(curr_f)
                    final_el.append(err_lows_full[k])
                    final_eh.append(err_highs_full[k])
                    prev_f = curr_f
            
            t_vals_plot = np.array(final_t)
            frac_arr_plot = np.array(final_f)
            err_lows_plot = np.array(final_el)
            err_highs_plot = np.array(final_eh)
            plot_label = "Data (Changes Only)"
        else:
            # Use full data
            t_vals_plot = t_vals_full
            frac_arr_plot = np.array(fractions_full)
            err_lows_plot = np.array(err_lows_full)
            err_highs_plot = np.array(err_highs_full)
            plot_label = "Data (Full)"

        low_bound = frac_arr_plot - err_lows_plot
        high_bound = frac_arr_plot + err_highs_plot
        
        # Approx sigma for weighting
        sigma_approx = (err_lows_plot + err_highs_plot) / 2.0
        sigma_approx[sigma_approx == 0] = 0.01
        
        # --- Create Subplots (NOW 3 ROWS) ---
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=FIGSIZE_FRAC, sharex=True, 
                                            gridspec_kw={'height_ratios': [3, 1, 1], 'hspace': 0.1})
        
        # 1. Main Plot
        ax1.fill_between(t_vals_plot, low_bound, high_bound, color='blue', alpha=0.15, label=r"68% Conf. Int.")
        ax1.plot(t_vals_plot, frac_arr_plot, marker='.', color='blue', markersize=8, linestyle='none', label=plot_label)

        # A. Existing 2-Segment Fit
        fit_res = None
        if state.get("fit_elbow", True) and len(frac_arr_plot) > 5:
            fit_res = _fit_two_segment_linear_weighted(t_vals_plot, frac_arr_plot, sigma_approx)
            if fit_res:
                x1_range = np.linspace(fit_res['range1'][0], fit_res['elbow_x'], 50)
                y1_range = np.polyval(fit_res['p1'], x1_range)
                ax1.plot(x1_range, y1_range, color='orange', linewidth=2, linestyle='--', label=f"2-Seg Fit")
                x2_range = np.linspace(fit_res['elbow_x'], fit_res['range2'][1], 50)
                y2_range = np.polyval(fit_res['p2'], x2_range)
                ax1.plot(x2_range, y2_range, color='orange', linewidth=2, linestyle='--')
                ax1.axvline(fit_res['elbow_x'], color='orange', linestyle=':', label=f"Elbow ({fit_res['elbow_x']:.1f})")

        # B. Residuals & Derivatives Calculation
        # We prioritize the 2-Segment fit for residuals as requested.
        
        # Default residuals (if fit fails)
        residuals = np.zeros_like(frac_arr_plot)
        
        if fit_res:
            # Calculate Model Y based on the 2-segment fit
            y_model = []
            for t in t_vals_plot:
                if t <= fit_res['elbow_x']:
                    y_model.append(np.polyval(fit_res['p1'], t))
                else:
                    y_model.append(np.polyval(fit_res['p2'], t))
            y_model = np.array(y_model)
            residuals = frac_arr_plot - y_model
        else:
            # Fallback to Poly if 2-seg disabled (optional, or just zeros)
            y_model = np.zeros_like(frac_arr_plot)

        # --- Calculate Max 2nd Derivative ---
        grad1 = np.gradient(frac_arr_plot, t_vals_plot)
        grad2 = np.gradient(grad1, t_vals_plot)
        
        # Identify Max of 2nd Deriv
        idx_max_grad2 = np.argmax(np.abs(grad2))
        t_max_grad2 = t_vals_plot[idx_max_grad2]

        # --- 2. RESIDUALS PLOT (New Middle Plot) ---
        ax2.axhline(0, color='black', linewidth=1, linestyle='-')
        # ERROR BARS added to residuals: Use the data uncertainties
        ax2.errorbar(t_vals_plot, residuals, yerr=[err_lows_plot, err_highs_plot], 
                     fmt='o', color='purple', ecolor='purple', alpha=0.6, 
                     markersize=4, capsize=0, label="Residuals (vs 2-Seg Fit)")
        ax2.set_ylabel("Residuals")
        ax2.grid(True, alpha=0.3)

        # --- 3. 2ND DERIVATIVE PLOT (Bottom Plot) ---
        ax3.plot(t_vals_plot, grad2, color='blue', alpha=0.6, marker='.', linestyle='-', markersize=6, label="Discrete 2nd Deriv")
        ax3.axhline(0, color='black', linewidth=1, linestyle='-')
        
        # --- Add Threshold Lines to ALL 3 plots ---
        axes_list = [ax1, ax2, ax3]
        for ax in axes_list:
            # Hard
            ax.axvline(THRESH_KMS, color='red', linestyle='--', alpha=0.5, label=f'Hard ({THRESH_KMS})' if ax==ax1 else "")
            # Equiv
            if use_equiv and np.isfinite(current_thresh) and current_thresh != THRESH_KMS:
                 ax.axvline(current_thresh, color='green', linestyle=':', alpha=0.7, label=f'Equiv ({current_thresh:.1f})' if ax==ax1 else "")
            # Elbow (Fit)
            if fit_res:
                 ax.axvline(fit_res['elbow_x'], color='orange', linestyle=':', alpha=0.7, label="Elbow" if ax==ax1 else "")
            # Max 2nd Deriv
            ax.axvline(t_max_grad2, color='magenta', linestyle='-.', linewidth=2, alpha=0.8, label=f"Max D2 ({t_max_grad2:.1f})" if ax==ax1 else "")

        # Finalize Plots
        step_size = max(10, int(MAX_RV_THRESH / 10))
        ax1.set_xticks(np.arange(0, MAX_RV_THRESH + 1, step_size))
        ax1.set_yticks(np.arange(0, 1.1, 0.1))
        ax1.set_ylabel(f"$f_{{\mathrm{{bin}}}}$")
        ax1.set_title(f"Binary Fraction vs Threshold: {line_key_bar}")
        ax1.grid(True, alpha=0.3, which='both')
        ax1.legend(loc='upper right', fontsize=FS_TINY)
        ax1.set_ylim(0, 1.05)
        
        ax3.set_ylabel(r"$d^2y/dx^2$")
        ax3.set_xlabel("Threshold [km/s]")
        ax3.grid(True, alpha=0.3)
        ax3.legend(loc='lower right', fontsize=FS_TINY)
        
        plt.tight_layout()
        plt.show()

    lines_plot = []
    thresholds_plot = []
    for lk in ordered_lines:
        t = equiv_thresholds_map.get(lk, np.nan)
        lines_plot.append(lk)
        thresholds_plot.append(t)
    if len(lines_plot) > 0:
        plt.figure(figsize=FIGSIZE_THRESH_PLOT)
        plt.plot(lines_plot, thresholds_plot, marker='o', linestyle='-', color='purple')
        plt.xticks(rotation=45, ha='right')
        plt.ylabel("Equivalent Threshold [km/s]")
        plt.title(f"Thresholds yielding $f_{{\mathrm{{bin}}}}$ ≈ $f_{{\mathrm{{bin}}}}$(CIV, 20km/s)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    show_names = state.get("show_corner_names", False)
    use_log = state.get("corner_log_scale", True)
    n_lines = len(ordered_lines)
    line_ranges = {}
    for lk in ordered_lines:
        col = f"dRV | {lk}"
        vals = []
        if use_equiv: t_val = equiv_thresholds_map.get(lk, THRESH_KMS)
        else: t_val = THRESH_KMS
        if np.isfinite(t_val): vals.append(t_val)
        if col in masked.columns:
            for i in range(len(masked)):
                sname = masked.at[i, "Star"]
                _, _, frac = ew_fail_stats.get((sname, lk), (0,0,0))
                if frac > FAIL_FRAC_THRESH: continue
                v = masked.at[i, col]
                if pd.notna(v) and np.isfinite(float(v)): vals.append(float(v))
        if not vals: 
            line_ranges[lk] = (0.1, MAX_RV_THRESH)
            continue
        vmin, vmax = min(vals), max(vals)
        if use_log:
            pos_vals = [x for x in vals if x > 0]
            if pos_vals:
                vmin = min(pos_vals)
                log_min, log_max = np.log10(vmin), np.log10(vmax)
                span = log_max - log_min if log_max > log_min else 0.5
                vmin = 10**(log_min - 0.05*span)
                vmax = 10**(log_max + 0.05*span)
            else: vmin, vmax = 0.1, MAX_RV_THRESH
        else:
            span = vmax - vmin if vmax > vmin else 5.0
            vmin = vmin - 0.05*span
            vmax = vmax + 0.05*span
            if vmin < 0: vmin = 0 
        line_ranges[lk] = (vmin, vmax)
    if n_lines > 1:
        dim = max(16, 3 * n_lines)
        fig, axes = plt.subplots(n_lines, n_lines, figsize=(dim, dim), sharex='col')
        plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, wspace=0, hspace=0)
        if n_lines == 1: axes = np.array([[axes]]) 
        trans_diagonal = mtransforms.Affine2D().rotate_deg(-45)
        for i in range(n_lines): 
            line_y_key = ordered_lines[i]
            y_min, y_max = line_ranges.get(line_y_key, (0.1, MAX_RV_THRESH))
            thresh_y = equiv_thresholds_map.get(line_y_key, THRESH_KMS) if use_equiv else THRESH_KMS
            for j in range(n_lines):
                ax = axes[i, j]
                line_x_key = ordered_lines[j]
                x_min, x_max = line_ranges.get(line_x_key, (0.1, MAX_RV_THRESH))
                thresh_x = equiv_thresholds_map.get(line_x_key, THRESH_KMS) if use_equiv else THRESH_KMS
                
                # --- MODIFIED: Single-line labels where possible ---
                if i == n_lines - 1:
                    ax.set_xlabel(textwrap.fill(line_x_key, width=20), fontsize=FS_AXIS_LBL)
                if j == 0 and i != j:
                    ax.set_ylabel(textwrap.fill(line_y_key, width=20), fontsize=FS_AXIS_LBL)
                # --------------------------------
                
                if use_log:
                    ax.set_xscale('log')
                    if i != j: ax.set_yscale('log')
                ax.set_xlim(x_min, x_max)
                if i != j: ax.set_ylim(y_min, y_max)
                if i < n_lines - 1: ax.tick_params(axis='x', labelbottom=False, length=0)
                if j > 0: ax.tick_params(axis='y', labelleft=False, length=0) 
                if j > i: ax.axis('off'); continue
                if i == j:
                    col_name = f"dRV | {line_y_key}"
                    if col_name in masked.columns:
                        vals_hist = []
                        for row_idx in range(len(masked)):
                            sname = masked.at[row_idx, "Star"]
                            _, _, frac = ew_fail_stats.get((sname, line_y_key), (0,0,0))
                            if frac > FAIL_FRAC_THRESH: continue
                            v = masked.at[row_idx, col_name]
                            if pd.notna(v) and np.isfinite(float(v)): vals_hist.append(float(v))
                        if len(vals_hist) > 0:
                            if use_log:
                                hv_min = max(x_min, min([v for v in vals_hist if v > 0]) if any(v>0 for v in vals_hist) else x_min)
                                hv_max = max(x_max, max(vals_hist))
                                bins = np.geomspace(hv_min, hv_max, 15)
                            else: bins = 15
                            ax.hist(vals_hist, bins=bins, color='gray', alpha=0.7, histtype='stepfilled', edgecolor='black')
                    ax.yaxis.set_visible(False); continue
                x_vals, y_vals, x_errs, y_errs = [], [], [], []
                c_topleft, c_botright = [], []
                names_to_plot = []
                col_x = f"dRV | {line_x_key}"
                col_y = f"dRV | {line_y_key}"
                if (col_x in masked.columns) and (col_y in masked.columns):
                    for row_idx in range(len(masked)):
                        sname = masked.at[row_idx, "Star"]
                        _, _, frac_x = ew_fail_stats.get((sname, line_x_key), (0,0,0))
                        _, _, frac_y = ew_fail_stats.get((sname, line_y_key), (0,0,0))
                        if (frac_x > FAIL_FRAC_THRESH) or (frac_y > FAIL_FRAC_THRESH): continue
                        vx = masked.at[row_idx, col_x]; vy = masked.at[row_idx, col_y]
                        if pd.notna(vx) and np.isfinite(float(vx)) and pd.notna(vy) and np.isfinite(float(vy)):
                            vx = float(vx); vy = float(vy)
                            if use_log and (vx <= 0 or vy <= 0): continue
                            x_vals.append(vx); y_vals.append(vy)
                            err_x = drverr_map.get((sname, line_x_key), 0.0)
                            err_y = drverr_map.get((sname, line_y_key), 0.0)
                            x_errs.append(err_x if np.isfinite(err_x) else 0.0)
                            y_errs.append(err_y if np.isfinite(err_y) else 0.0)
                            pass_y = _is_significant_binary(sname, line_y_key, vy, thresh_y)
                            c_topleft.append('green' if pass_y else 'red')
                            pass_x = _is_significant_binary(sname, line_x_key, vx, thresh_x)
                            c_botright.append('green' if pass_x else 'red')
                            names_to_plot.append(sname)
                if len(x_vals) > 0:
                    x_arr = np.array(x_vals); y_arr = np.array(y_vals)
                    x_err_arr = np.array(x_errs); y_err_arr = np.array(y_errs)
                    ax.axvline(thresh_x, color='black', linestyle=':', alpha=0.6, linewidth=1)
                    ax.axhline(thresh_y, color='black', linestyle=':', alpha=0.6, linewidth=1)
                    diag_min = max(x_min, y_min); diag_max = min(x_max, y_max)
                    ax.plot([diag_min, diag_max], [diag_min, diag_max], ls='--', color='gray', alpha=0.5, linewidth=1, zorder=0)
                    fbin_y_str = _get_fbin_simple(masked, line_y_key, use_equiv)
                    ax.text(0.05, 0.95, r"$f_{\mathrm{bin}}$(Y): " + fbin_y_str, transform=ax.transAxes, fontsize=FS_NOTE, va='top', ha='left', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))
                    fbin_x_str = _get_fbin_simple(masked, line_x_key, use_equiv)
                    ax.text(0.95, 0.05, r"$f_{\mathrm{bin}}$(X): " + fbin_x_str, transform=ax.transAxes, fontsize=FS_NOTE, va='bottom', ha='right', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))
                    ax.hlines(y_arr, x_arr - x_err_arr, x_arr + x_err_arr, colors=c_botright, lw=0.8, zorder=1)
                    ax.vlines(x_arr, y_arr - y_err_arr, y_arr + y_err_arr, colors=c_topleft, lw=0.8, zorder=1)
                    ax.scatter(x_vals, y_vals, c=c_botright, marker=MarkerStyle('o', fillstyle='right', transform=trans_diagonal), s=80, edgecolors='black', linewidth=0.5, zorder=2)
                    ax.scatter(x_vals, y_vals, c=c_topleft, marker=MarkerStyle('o', fillstyle='left', transform=trans_diagonal), s=80, edgecolors='black', linewidth=0.5, zorder=3)
                    if len(x_vals) > 1:
                        r_val, p_val = pearsonr(x_vals, y_vals)
                        ax.text(0.05, 0.85, f"r={r_val:.2f}\np={p_val:.2g}\nn={len(x_vals)}", transform=ax.transAxes, fontsize=FS_NOTE, va='top', ha='left', bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8))
                    if show_names:
                        for k, nm in enumerate(names_to_plot):
                            ax.text(x_vals[k], y_vals[k], nm, fontsize=FS_TINY, alpha=0.8, clip_on=True, rotation=25)
                ax.grid(True, alpha=0.2, which='both')
        plt.show()

    drv_cols = [f"dRV | {lk}" for lk in ordered_lines if f"dRV | {lk}" in masked.columns]
    if len(drv_cols) > 1:
        cols_names = [c for c in drv_cols]
        adj_corr_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
        adj_pval_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
        adj_n_df    = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
        for i, c1 in enumerate(cols_names):
            data1 = masked[c1]
            for j, c2 in enumerate(cols_names):
                if i == j: continue 
                data2 = masked[c2]
                valid = data1.notna() & data2.notna()
                n_dots = valid.sum()
                if n_dots < MIN_STARS_FOR_CORR: continue
                r, p = pearsonr(data1[valid], data2[valid])
                adj_corr_df.at[c1, c2] = r 
                adj_pval_df.at[c1, c2] = p
                adj_n_df.at[c1, c2] = n_dots
        w_df = adj_n_df / MAX_POSSIBLE_STARS
        r_weighted_terms = adj_corr_df * w_df
        avg_scores_user = r_weighted_terms.sum(axis=1) / r_weighted_terms.count(axis=1)
        p_times_n = adj_pval_df * adj_n_df
        sum_n = adj_n_df.sum(axis=1)
        avg_pvals_std_weighted = p_times_n.sum(axis=1) / sum_n.replace(0, np.nan)
        best_friends, best_scores = [], []
        second_best_friends, second_best_scores = [], []
        stats_texts = []
        for idx_row, col in enumerate(cols_names):
            line_nm = col.replace("dRV | ", "")
            row_series = adj_corr_df.loc[col].sort_values(ascending=False)
            valid_friends = row_series.dropna()
            def _fmt_friend(friend_col, r_val):
                v = (masked[col].notna() & masked[friend_col].notna())
                n_count = v.sum()
                if n_count < 2: return "Err", np.nan
                _, p_val = pearsonr(masked[col][v], masked[friend_col][v])
                nm = friend_col.replace("dRV | ", "")
                return f"{nm}\nr={r_val:.2f} (p={p_val:.2g}, n={n_count})", r_val
            if len(valid_friends) > 0: b_lbl, b_val = _fmt_friend(valid_friends.index[0], valid_friends.iloc[0])
            else: b_val, b_lbl = np.nan, "None"
            if len(valid_friends) > 1: s_lbl, s_val = _fmt_friend(valid_friends.index[1], valid_friends.iloc[1])
            else: s_val, s_lbl = np.nan, ""
            best_scores.append(b_val); best_friends.append(b_lbl)
            second_best_scores.append(s_val); second_best_friends.append(s_lbl)
            fbin_txt = _get_fbin_simple(masked, line_nm, use_equiv)
            ew_succ, ew_mn, ew_sem = _calc_ew_stats_for_line(line_nm)
            stats_texts.append(r"$f_{\mathrm{bin}}$: " + f"{fbin_txt}\nEW%: {ew_succ:.0%}\nEW: {ew_mn:.2f}±{ew_sem:.2f}")
        plot_data = pd.DataFrame({
            "Line": [c.replace("dRV | ", "") for c in cols_names],
            "Score": avg_scores_user.values,
            "Avg_P_Weighted": avg_pvals_std_weighted.values,
            "Best_Score": best_scores,
            "Best_Friend": best_friends,
            "Second_Score": second_best_scores,
            "Second_Friend": second_best_friends,
            "Stats": stats_texts
        }).sort_values("Score", ascending=False, na_position='last').reset_index(drop=True)
        plt.figure(figsize=FIGSIZE_AGREEMENT)
        valid_plot = plot_data.dropna(subset=["Score"])
        if not valid_plot.empty:
            plt.plot(valid_plot.index, valid_plot["Score"], marker='o', linestyle='-', color='teal', linewidth=2, markersize=8, label=f"Avg Score (Weight = N/{MAX_POSSIBLE_STARS})")
        
        # === MODIFICATION: Control extra data visibility ===
        show_extra = state.get("show_extra_data", False)
        
        if show_extra:
            plt.scatter(plot_data.index, plot_data["Best_Score"], color='orange', s=80, marker='D', zorder=3, label="Max Correlation")
            plt.scatter(plot_data.index, plot_data["Second_Score"], color='lightgreen', s=70, marker='s', edgecolors='green', zorder=2, label="2nd Best Correlation")
        
        plt.xticks(plot_data.index, plot_data["Line"], rotation=45, ha="right")
        plt.ylabel("Agreement Score")
        plt.title(f"Agreement Ranking (Logic: r * n/25, N >= {MIN_STARS_FOR_CORR})")
        plt.grid(True, alpha=0.3, linestyle='--')
        for i in plot_data.index:
            score = plot_data.at[i, "Score"]
            avg_p_w = plot_data.at[i, "Avg_P_Weighted"]
            best = plot_data.at[i, "Best_Score"]
            friend = plot_data.at[i, "Best_Friend"]
            second = plot_data.at[i, "Second_Score"]
            sec_friend = plot_data.at[i, "Second_Friend"]
            stat_txt = plot_data.at[i, "Stats"]
            if pd.notna(score):
                # Only show p_w if show_extra is True
                p_text = ""
                if show_extra and pd.notna(avg_p_w):
                      p_text = f"\n(p_w={avg_p_w:.2g})"
                
                plt.text(i, score - 0.02, f"{score:.3f}{p_text}", ha='center', va='top', fontsize=FS_NOTE, color='teal', fontweight='bold')
            
            # Only show friends and bottom stats if show_extra is True
            if show_extra:
                if pd.notna(best): plt.text(i, best + 0.01, friend, ha='center', va='bottom', fontsize=FS_TINY, color='darkorange', rotation=25)
                if pd.notna(second): plt.text(i, second - 0.01, sec_friend, ha='center', va='top', fontsize=FS_TINY, color='green', rotation=25)
                
                ylim_min, ylim_max = plt.ylim()
                y_txt_pos = ylim_min + (ylim_max - ylim_min) * 0.05
                plt.text(i, y_txt_pos, stat_txt, ha='center', va='bottom', fontsize=FS_BOX_STAT, bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="gray", alpha=0.7))
        
        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.show()

        sort_by_agreement = state.get("sort_fbin_by_agreement", False)
        if sort_by_agreement and 'plot_data' in locals() and not plot_data.empty: sorted_lines_g6 = plot_data["Line"].tolist()
        else: sorted_lines_g6 = sorted(ordered_lines, key=_get_wavelength)
        
        # === UPDATED PLOT 6: Dot Plot with Error Bars ===
        plt.figure(figsize=FIGSIZE_FBIN_BAR)
        
        # Check if we want both or just one
        show_both = state.get("show_both_thresholds", False)
        
        # Prepare Data Containers
        lines_lbl = []
        
        # Data for Set A (Default/Selected)
        vals_A, err_l_A, err_h_A = [], [], []
        
        # Data for Set B (Secondary - only if show_both)
        vals_B, err_l_B, err_h_B = [], [], []

        for lk in sorted_lines_g6:
             lines_lbl.append(lk)
             
             if show_both:
                 # Set A: Hard
                 f, el, eh, _, _ = _calc_fbin_stats_internal(masked, lk, use_equiv_thresh=False) # Force Hard
                 vals_A.append(f * 100.0); err_l_A.append(el * 100.0); err_h_A.append(eh * 100.0)
                 
                 # Set B: Equiv
                 f2, el2, eh2, _, _ = _calc_fbin_stats_internal(masked, lk, use_equiv_thresh=True) # Force Equiv
                 vals_B.append(f2 * 100.0); err_l_B.append(el2 * 100.0); err_h_B.append(eh2 * 100.0)
             else:
                 # Standard Behavior
                 f, el, eh, _, _ = _calc_fbin_stats_internal(masked, lk, use_equiv)
                 vals_A.append(f * 100.0); err_l_A.append(el * 100.0); err_h_A.append(eh * 100.0)

        x_pos = np.arange(len(lines_lbl))
        
        if show_both:
            width = 0.15
            # Plot Hard (Blue) - Gray bars, Blue dots
            plt.errorbar(x_pos - width, vals_A, yerr=[err_l_A, err_h_A], fmt='o', color='blue', ecolor='gray', 
                         capsize=3, elinewidth=1, markersize=6, label=f"Hard ({THRESH_KMS} km/s)")
            
            # Plot Equiv (Magenta) - Gray bars, Magenta dots
            plt.errorbar(x_pos + width, vals_B, yerr=[err_l_B, err_h_B], fmt='o', color='magenta', ecolor='gray', 
                         capsize=3, elinewidth=1, markersize=6, label="Fitting (Equiv)")
            
            plt.legend(loc='upper right')
            
            # Add text labels (optional, might get crowded)
            for i, v in enumerate(vals_A):
                plt.text(i - width, v + 2, f"{v:.0f}", ha='center', va='bottom', fontsize=8, color='blue')
            for i, v in enumerate(vals_B):
                plt.text(i + width, v + 2, f"{v:.0f}", ha='center', va='bottom', fontsize=8, color='magenta')
                
        else:
            # Original Single Mode (Restored Look)
            color_use = 'purple' if use_equiv else 'cornflowerblue'
            lbl_use = "Fitting (Equiv)" if use_equiv else f"Hard ({THRESH_KMS} km/s)"
            
            plt.errorbar(x_pos, vals_A, yerr=[err_l_A, err_h_A], fmt='o', color=color_use, ecolor='gray', 
                         capsize=5, elinewidth=2, markersize=8, label=lbl_use)
            
            for i, v in enumerate(vals_A):
                plt.text(i, v + 2, f"{v:.0f}%", ha='center', va='bottom', fontsize=FS_BAR_VAL, fontweight='bold', color='black')
            plt.legend(loc='upper right')

        plt.xticks(x_pos, lines_lbl, rotation=45, ha='right')
        plt.ylabel("Binary Fraction (%)")
        plt.title(f"Binary Fraction per Emission Line ({'Sorted by Agreement' if sort_by_agreement else 'Sorted by Wavelength'})")
        plt.ylim(0, 105) 
        plt.grid(axis='y', alpha=0.3, linestyle='--')
        
        plt.tight_layout()
        plt.show()
        # ================================================

    # --- PLOT 7: Confidence Grading (Golden/Silver/Bronze) ---
    star_results = get_star_classification_data(df, use_equiv)
    highlight_contam = state.get("highlight_contam", False)
    
    grades_data = {
        'Golden': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []},
        'Silver': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []},
        'Bronze': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []}
    }
    
    for sname, info in star_results.items():
        grade = info['grade']
        is_bin = info['is_binary']
        is_clean = info['is_clean']
        if is_bin:
            if is_clean: grades_data[grade]['bin_clean'].append(sname)
            else: grades_data[grade]['bin_contam'].append(sname)
        else:
            if is_clean: grades_data[grade]['sin_clean'].append(sname)
            else: grades_data[grade]['sin_contam'].append(sname)
            
    fig, axes = plt.subplots(1, 3, figsize=(14, 8)) 
    theme_map = {
        'Golden': (['#FFD700', '#FFFACD'], '#DAA520'), 
        'Silver': (['#A9A9A9', '#DCDCDC'], 'gray'),    
        'Bronze': (['#CD853F', '#FFE4C4'], '#8B4513') 
    }
    
    grade_keys = ['Golden', 'Silver', 'Bronze']
    titles = [f"Golden (>90% Agree)", f"Silver (>70% Agree)", f"Bronze (Rest)"]
    
    for idx, g in enumerate(grade_keys):
        ax = axes[idx]
        data = grades_data[g]
        n_bin_clean = len(data['bin_clean'])
        n_bin_contam = len(data['bin_contam'])
        n_sin_clean = len(data['sin_clean'])
        n_sin_contam = len(data['sin_contam'])
        total = n_bin_clean + n_bin_contam + n_sin_clean + n_sin_contam
        
        colors_theme, title_col = theme_map[g]
        c_bin_main, c_sin_main = colors_theme
        
        if total > 0:
            if highlight_contam:
                # Nested Pie (Donut)
                # Outer Ring: Binary vs Single
                n_bin = n_bin_clean + n_bin_contam
                n_sin = n_sin_clean + n_sin_contam
                outer_sizes = [n_bin, n_sin]
                outer_labels = ["Binary", "Single"]
                outer_colors = [c_bin_main, c_sin_main]
                
                f_out_sizes, f_out_labels, f_out_colors = [], [], []
                for s, l, c in zip(outer_sizes, outer_labels, outer_colors):
                    if s > 0: f_out_sizes.append(s); f_out_labels.append(l); f_out_colors.append(c)
                
                # Outer Pie
                ax.pie(f_out_sizes, labels=f_out_labels, colors=f_out_colors, 
                       radius=1.0, wedgeprops=dict(width=0.3, edgecolor='w'), 
                       startangle=90, textprops={'fontsize': FS_PIE_LABEL, 'weight': 'bold'},
                       autopct='%1.1f%%', pctdistance=0.85)
                
                # Inner Ring: Breakdown (Bin-Clean, Bin-Contam, Sin-Clean, Sin-Contam)
                # Order must match Outer Ring (Binary then Single)
                inner_sizes = [n_bin_clean, n_bin_contam, n_sin_clean, n_sin_contam]
                # Colors: Main, Red, Main2, Red
                inner_colors = [c_bin_main, '#FF4444', c_sin_main, '#FF4444'] 
                
                f_in_sizes, f_in_colors = [], []
                for s, c in zip(inner_sizes, inner_colors):
                    if s > 0: f_in_sizes.append(s); f_in_colors.append(c)
                
                if f_in_sizes:
                    ax.pie(f_in_sizes, colors=f_in_colors, radius=0.7, 
                           wedgeprops=dict(width=0.7, edgecolor='w'), startangle=90,
                           autopct='%1.1f%%', pctdistance=0.6, textprops={'fontsize': FS_TINY})
            else:
                n_bin = n_bin_clean + n_bin_contam
                n_sin = n_sin_clean + n_sin_contam
                ax.pie([n_bin, n_sin], labels=[f"Binary\n{n_bin}", f"Single\n{n_sin}"], colors=colors_theme, autopct='%1.1f%%', startangle=90, textprops={'fontsize': FS_PIE_LABEL})
        else:
            ax.text(0.5, 0.5, "No Stars", ha='center', va='center', fontsize=FS_PIE_LABEL)
            ax.axis('off')
            
        ax.set_title(titles[idx], color=title_col, fontsize=FS_PIE_LABEL+2, fontweight='bold')
        
        def _get_wrapped(lst):
            if not lst: return ""
            return "\n".join(textwrap.wrap(", ".join(lst), width=20))
            
        bin_clean_txt = _get_wrapped(data['bin_clean'])
        bin_contam_txt = _get_wrapped(data['bin_contam'])
        sin_clean_txt = _get_wrapped(data['sin_clean'])
        sin_contam_txt = _get_wrapped(data['sin_contam'])
        
        # Binary Column
        ax.text(0.05, -0.05, f"$\\bf{{Binary:}}$", transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT)
        current_y = -0.12
        if bin_clean_txt:
            ax.text(0.05, current_y, bin_clean_txt, transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')
            current_y -= (bin_clean_txt.count('\n') + 2) * 0.05
        if highlight_contam and bin_contam_txt:
            ax.text(0.05, current_y, bin_contam_txt, transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT, color='red', fontweight='bold')
        elif not highlight_contam and bin_contam_txt:
            ax.text(0.05, current_y, bin_contam_txt, transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')

        # Single Column
        ax.text(0.55, -0.05, f"$\\bf{{Single:}}$", transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT)
        current_y = -0.12
        if sin_clean_txt:
            ax.text(0.55, current_y, sin_clean_txt, transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')
            current_y -= (sin_clean_txt.count('\n') + 2) * 0.05
        if highlight_contam and sin_contam_txt:
            ax.text(0.55, current_y, sin_contam_txt, transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT, color='red', fontweight='bold')
        elif not highlight_contam and sin_contam_txt:
            ax.text(0.55, current_y, sin_contam_txt, transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')

    plt.tight_layout()
    plt.show()
    
    # --- PLOT 8: All / Clean / Contaminated Comparison ---
    clean_counts = {'Binary': 0, 'Single': 0}
    contam_counts = {'Binary': 0, 'Single': 0}
    all_counts = {'Binary': 0, 'Single': 0}
    
    clean_lists = {'Binary': [], 'Single': []}
    contam_lists = {'Binary': [], 'Single': []}
    all_lists = {'Binary': [], 'Single': []}
    
    for sname, info in star_results.items():
        k = 'Binary' if info['is_binary'] else 'Single'
        # Add to All
        all_counts[k] += 1
        all_lists[k].append(sname)
        
        # Add to Specifics
        if info['is_clean']:
            clean_counts[k] += 1
            clean_lists[k].append(sname)
        else:
            contam_counts[k] += 1
            contam_lists[k].append(sname)
            
    fig, axes = plt.subplots(1, 3, figsize=(16, 8)) # 3 Columns now
    
    def plot_pie_with_lists(ax, counts, lists, title):
        n_bin = counts['Binary']
        n_sin = counts['Single']
        if (n_bin + n_sin) > 0:
            ax.pie([n_bin, n_sin], labels=[f"Binary\n{n_bin}", f"Single\n{n_sin}"],
                   colors=['cornflowerblue', 'lightgray'], autopct='%1.1f%%', startangle=90,
                   textprops={'fontsize': FS_PIE_LABEL})
        else:
            ax.text(0.5, 0.5, "No Data", ha='center', va='center', fontsize=FS_PIE_LABEL)
        ax.set_title(title, fontsize=FS_PIE_LABEL+2, fontweight='bold')
        
        # Add Lists
        def _wrp(l): return "\n".join(textwrap.wrap(", ".join(l), width=20))
        bin_txt = _wrp(lists['Binary'])
        sin_txt = _wrp(lists['Single'])
        
        # Binary Column (Side by side using x=0.05 and x=0.55)
        ax.text(0.05, -0.05, f"$\\bf{{Binary:}}$", transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT)
        current_y = -0.12
        if bin_txt:
            ax.text(0.05, current_y, bin_txt, transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT)
        
        # Single Column
        ax.text(0.55, -0.05, f"$\\bf{{Single:}}$", transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT)
        current_y = -0.12
        if sin_txt:
            ax.text(0.55, current_y, sin_txt, transform=ax.transAxes, ha='left', va='top', fontsize=FS_LIST_TEXT)
        
    plot_pie_with_lists(axes[0], all_counts, all_lists, "All Stars Together")
    plot_pie_with_lists(axes[1], clean_counts, clean_lists, "Clean Stars")
    plot_pie_with_lists(axes[2], contam_counts, contam_lists, "Contaminated Stars")
    
    plt.tight_layout()
    plt.show()

# ---------------- 7. WIDGETS & DISPLAY ----------------
default_sort = f"dRV | {CIV_KEY}" if f"dRV | {CIV_KEY}" in df.columns else (MEAN_COL if MEAN_COL in df.columns else "Star")

# UPDATE: Added "show_both_thresholds" to state
state = {"sort_col": default_sort, "ascending": False, "force_show_low_detection": False, 
         "use_equiv_thresh": False, "show_corner_names": False, "corner_log_scale": True,
         "sort_fbin_by_agreement": False, "show_extra_data": False, "highlight_contam": False,
         "fit_elbow": True, "poly_deg": 3, "filter_threshold_plot": True,
         "show_both_thresholds": False} 

btn_for_col = {}

def update_buttons():
    for col, btn in btn_for_col.items():
        active = (col == state["sort_col"])
        arrow = "↑" if (active and state["ascending"]) else ("↓" if active else "")
        btn.description = f"{wrap_header(col)} {arrow}".rstrip()
        btn.button_style = "info" if active else ""

def on_click(col):
    def _handler(_):
        if col == state["sort_col"]: state["ascending"] = not state["ascending"]
        else: state["sort_col"] = col; state["ascending"] = False
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
        update_buttons()
    return _handler

show_low_detect_chk = widgets.Checkbox(value=False, description="Show >40% EW failures", indent=False)
def on_toggle_low_detect(change):
    if change["name"] == "value":
        state["force_show_low_detection"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
show_low_detect_chk.observe(on_toggle_low_detect)

use_equiv_thresh_chk = widgets.Checkbox(value=False, description="Use Equivalent Thresholds", indent=False)
def on_toggle_equiv(change):
    if change["name"] == "value":
        state["use_equiv_thresh"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
use_equiv_thresh_chk.observe(on_toggle_equiv)

corner_names_chk = widgets.Checkbox(value=False, description="Corner Plot: Show Names", indent=False)
def on_toggle_corner_names(change):
    if change["name"] == "value":
        state["show_corner_names"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
corner_names_chk.observe(on_toggle_corner_names)

corner_log_chk = widgets.Checkbox(value=True, description="Log Scale (All Plots)", indent=False)
def on_toggle_corner_log(change):
    if change["name"] == "value":
        state["corner_log_scale"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
corner_log_chk.observe(on_toggle_corner_log)

sort_fbin_chk = widgets.Checkbox(value=False, description="Graph 6: Sort by Agreement Score", indent=False)
def on_toggle_fbin_sort(change):
    if change["name"] == "value":
        state["sort_fbin_by_agreement"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
sort_fbin_chk.observe(on_toggle_fbin_sort)

show_extra_data_chk = widgets.Checkbox(value=False, description="Graph 5: Show Extra Data", indent=False)
def on_toggle_extra(change):
    if change["name"] == "value":
        state["show_extra_data"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
show_extra_data_chk.observe(on_toggle_extra)

highlight_contam_chk = widgets.Checkbox(value=False, description="Graph 7: Highlight Contaminated Stars", indent=False)
def on_toggle_contam(change):
    if change["name"] == "value":
        state["highlight_contam"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
highlight_contam_chk.observe(on_toggle_contam)

fit_elbow_chk = widgets.Checkbox(value=True, description="Fit Elbow (2-Line)", indent=False)
def on_toggle_fit_elbow(change):
    if change["name"] == "value":
        state["fit_elbow"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
fit_elbow_chk.observe(on_toggle_fit_elbow)

# === NEW: Filter Threshold Plot ===
filter_thresh_chk = widgets.Checkbox(value=True, description="Filter Thresh Plot (Changes Only)", indent=False)
def on_toggle_filter(change):
    if change["name"] == "value":
        state["filter_threshold_plot"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
filter_thresh_chk.observe(on_toggle_filter)

# === NEW WIDGET FOR GRAPH 6 ===
show_both_thresh_chk = widgets.Checkbox(value=False, description="Graph 6: Show Both (Hard & Fit)", indent=False)
def on_toggle_both_thresh(change):
    if change["name"] == "value":
        state["show_both_thresholds"] = bool(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
show_both_thresh_chk.observe(on_toggle_both_thresh)

# === NEW WIDGET: Poly Degree ===
poly_deg_drop = widgets.Dropdown(options=[2, 3, 4, 5, 6, 7], value=3, description="Poly Deg:")
def on_poly_change(change):
    if change["name"] == "value":
        state["poly_deg"] = int(change["new"])
        with out: clear_output(wait=True); render_plots(state["sort_col"], state["ascending"])
poly_deg_drop.observe(on_poly_change)

buttons = []
for col in ordered_cols:
    b = widgets.Button(description=wrap_header(col), tooltip=f"Sort by {col}", layout=widgets.Layout(width="auto"))
    b.on_click(on_click(col))
    btn_for_col[col] = b
    buttons.append(b)

controls = widgets.HBox([
    show_low_detect_chk, use_equiv_thresh_chk, corner_names_chk, corner_log_chk, 
    sort_fbin_chk, show_extra_data_chk, highlight_contam_chk, fit_elbow_chk, 
    filter_thresh_chk, poly_deg_drop, show_both_thresh_chk 
], layout=widgets.Layout(gap="12px", flex_flow="row wrap"))

headers = widgets.HBox(buttons, layout=widgets.Layout(flex_flow="row wrap", gap="6px"))
out = widgets.Output()

display(controls, headers, out)
with out:
    clear_output(wait=True)
    render_plots(state["sort_col"], state["ascending"])
update_buttons()

### new

In [ ]:
%matplotlib widget
# === ΔRV Analysis: Updated with Fit-Based Thresholds & Simplified Graphs ===

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import matplotlib.ticker as ticker
from matplotlib.markers import MarkerStyle
import textwrap
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import pearsonr, chi2

# ---------------- 1. SETUP & CONFIG ----------------
SETTINGS_PATH = "ccf_settings_with_global_lines.json"
try:
    with open(SETTINGS_PATH, "r") as f:
        SETTINGS = json.load(f)
except FileNotFoundError:
    print(f"WARNING: Could not find {SETTINGS_PATH}. Using empty settings.")
    SETTINGS = {}

STAR_CFG = {s["star_name"]: s for s in SETTINGS.get("stars", [])}
LINES_DEFAULT = SETTINGS.get("emission_lines_default", {})

# --- PRESENTATION MODE ---
for_presentation = True

# Config
CIV_KEY = "C IV 5808-5812"
THRESH_KMS = 10.0  # Fallback if fit fails
MAX_RV_THRESH = 400
EXTRA_ABOVE = 3
BAR_FIGSIZE = (12, 6)
FIGSIZE_FRAC = (12, 8)  # Reduced height (2 subplots instead of 3)
FIGSIZE_THRESH_PLOT = (10, 5)
FIGSIZE_AGREEMENT = (12, 7)
FIGSIZE_FBIN_BAR = (12, 6)
FAIL_FRAC_THRESH = 0.40
MIN_STARS_FOR_CORR = 8
MAX_POSSIBLE_STARS = 25
KNOWN_ERR_KEYS = ("full_RV_err", "full_err", "sigma", "err", "RV_err", "RV_sigma")
MEAN_COL = "Mean ΔRV"
STD_COL = "Std ΔRV"

# Globals for storage
epoch_ew_data = {}
ew_fail_stats = {}
rv_epoch_cache = {}
drverr_map = {}
equiv_thresholds_map = {}
fit_elbow_thresholds_cache = {} 
d2_thresholds_cache = {} 

# ---------------- 2. DATA PROCESSING HELPERS ----------------

def _is_clean_star(star):
    try:
        for ep in star.get_all_epoch_numbers():
            flux = star.load_property('cleaned_normalized_flux', ep, 'COMBINED')
            if flux is not None:
                return False
        return True
    except Exception:
        return True

def _ew_record_for(star, epoch_num, line_key):
    try:
        EWs = star.load_property('EWs', epoch_num, 'COMBINED')
    except Exception:
        return None
    if EWs is None:
        return None
    try:
        rec = EWs.get(line_key)
    except Exception:
        return None
    if rec is None:
        return None
    try:
        rec_dict = rec.item()
    except Exception:
        return None
    if not isinstance(rec_dict, dict):
        return None

    def _to_float(x):
        try:
            return float(x)
        except Exception:
            return np.nan

    return {
        "EW": _to_float(rec_dict.get("EW")),
        "sigma_EW": _to_float(rec_dict.get("sigma_EW")),
        "SNR": _to_float(rec_dict.get("SNR")),
        "detected": rec_dict.get("detected")
    }

def _extract_full_rv(cell):
    try:
        if isinstance(cell, dict):
            return cell.get('full_RV', None)
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict):
                return v.get('full_RV', None)
        if hasattr(cell, "get"):
            return cell.get('full_RV', None)
    except Exception:
        pass
    return None

def _extract_full_rv_err(cell):
    try:
        if isinstance(cell, dict):
            for k in KNOWN_ERR_KEYS:
                if k in cell and cell[k] is not None:
                    return float(cell[k])
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict):
                for k in KNOWN_ERR_KEYS:
                    if k in v and v[k] is not None:
                        return float(v[k])
        if hasattr(cell, "get"):
            for k in KNOWN_ERR_KEYS:
                val = cell.get(k, None)
                if val is not None:
                    return float(val)
    except Exception:
        pass
    return np.nan

def _is_epoch_skipped_for_line(star_name, line_key, ep):
    cfg = STAR_CFG.get(star_name, {})
    if ep in set(cfg.get("skip_epochs", [])):
        return True
    sl = cfg.get("skip_emission_lines", {})
    if line_key in sl:
        skip_eps = sl[line_key]
        if isinstance(skip_eps, (int, np.integer)):
            skip_eps = [skip_eps]
        if 0 in skip_eps or ep in skip_eps:
            return True
        return False
    return False

def _line_delta_rv_for_star(star, line_key):
    rv_vals, cache = [], []
    sname = star.star_name

    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep):
            cache.append((ep, np.nan, np.nan, False))
            continue

        ewrec = _ew_record_for(star, ep, line_key)
        if ewrec is not None:
            epoch_ew_data[(sname, line_key, ep)] = ewrec

        RVs = star.load_property('RVs', ep, 'COMBINED')
        if not RVs or line_key not in RVs:
            cache.append((ep, np.nan, np.nan, False))
            continue

        cell = RVs[line_key]
        rv = _extract_full_rv(cell)

        if rv is None:
            cache.append((ep, np.nan, np.nan, False))
            continue

        try:
            rv = float(rv)
        except Exception:
            cache.append((ep, np.nan, np.nan, False))
            continue

        err = _extract_full_rv_err(cell)
        rv_vals.append((ep, rv, err))
        cache.append((ep, rv, err, True))

    rv_epoch_cache[(sname, line_key)] = cache

    if len(rv_vals) == 0:
        drverr_map[(sname, line_key)] = np.nan
        return np.nan

    ep_min, rv_min, err_min = min(rv_vals, key=lambda t: t[1])
    ep_max, rv_max, err_max = max(rv_vals, key=lambda t: t[1])
    dRV = abs(rv_max - rv_min)

    if np.isfinite(err_min) and np.isfinite(err_max):
        sigma_A = float(np.sqrt(err_min**2 + err_max**2))
    else:
        sigma_A = np.nan

    drverr_map[(sname, line_key)] = sigma_A
    return dRV

def _is_significant_binary(star_name: str, line_key: str, drv_val: float, threshold_val: float) -> bool:
    if not (pd.notna(drv_val) and np.isfinite(drv_val)):
        return False
    sigma_A = drverr_map.get((star_name, line_key), np.nan)
    if not np.isfinite(sigma_A):
        return False
    return (float(drv_val) >= threshold_val) and (float(drv_val) >= 4.0 * float(sigma_A))

def _compute_ew_fail_stats(star, line_key):
    sname = star.star_name
    considered = []
    failed = 0
    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep):
            continue
        considered.append(ep)
        rec = epoch_ew_data.get((sname, line_key, ep))
        if isinstance(rec, dict):
            det = rec.get("detected")
            if det is False:
                failed += 1
    n_considered = len(considered)
    frac = (failed / n_considered) if n_considered > 0 else 0.0
    return failed, n_considered, frac

# ---------------- 3. LOAD DATA & BUILD DATAFRAME ----------------
rows = []
ordered_lines = list(LINES_DEFAULT.keys())

for star_name in specs.star_names:
    star = obs.load_star_instance(star_name, to_print=False)
    is_clean_bool = _is_clean_star(star)
    row = {"Star": star_name, "Clean": "✓" if is_clean_bool else "X", "is_clean_bool": is_clean_bool}

    for lk in ordered_lines:
        dval = _line_delta_rv_for_star(star, lk)
        ew_fail_stats[(star_name, lk)] = _compute_ew_fail_stats(star, lk)
        row[f"dRV | {lk}"] = dval

    drvs = [v for k, v in row.items() if isinstance(k, str) and k.startswith("dRV | ")]
    row[MEAN_COL] = float(np.nanmean(drvs)) if np.isfinite(np.nanmean(drvs)) else np.nan
    row[STD_COL] = float(np.nanstd(drvs)) if np.isfinite(np.nanstd(drvs)) else np.nan
    rows.append(row)

df = pd.DataFrame(rows)

# ---------------- 4. CALCULATE THRESHOLDS ----------------
def calculate_equivalent_thresholds(df_in):
    if CIV_KEY not in ordered_lines:
        return
    ref_col = f"dRV | {CIV_KEY}"
    civ_vals = []

    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        _, _, frac = ew_fail_stats.get((sname, CIV_KEY), (0, 0, 0))
        if frac > FAIL_FRAC_THRESH:
            continue
        val = df_in.at[i, ref_col]
        if pd.notna(val) and np.isfinite(float(val)):
            civ_vals.append((sname, float(val)))

    n_above = 0
    for s, v in civ_vals:
        if _is_significant_binary(s, CIV_KEY, v, 20.0):
            n_above += 1

    num = n_above + EXTRA_ABOVE
    den = len(civ_vals) + EXTRA_ABOVE
    f_ref = num / den if den > 0 else 0

    equiv_thresholds_map.clear()
    search_space = np.arange(1, MAX_RV_THRESH + 1, 1)

    for lk in ordered_lines:
        col = f"dRV | {lk}"
        if col not in df_in.columns:
            continue
        line_vals = []
        for i in range(len(df_in)):
            sname = df_in.at[i, "Star"]
            _, _, frac = ew_fail_stats.get((sname, lk), (0, 0, 0))
            if frac > FAIL_FRAC_THRESH:
                continue
            val = df_in.at[i, col]
            if pd.notna(val) and np.isfinite(float(val)):
                line_vals.append((sname, float(val)))

        total_line = len(line_vals)
        if total_line == 0:
            equiv_thresholds_map[lk] = np.nan
            continue

        best_T = 20.0
        min_diff = 999.0

        for T in search_space:
            cnt = 0
            for s, v in line_vals:
                if _is_significant_binary(s, lk, v, T):
                    cnt += 1
            f_curr = (cnt + EXTRA_ABOVE) / (total_line + EXTRA_ABOVE)
            diff = abs(f_curr - f_ref)
            if diff < min_diff:
                min_diff = diff
                best_T = float(T)
        equiv_thresholds_map[lk] = best_T

calculate_equivalent_thresholds(df)

ordered_cols = ["Star", "Clean", MEAN_COL, STD_COL] + [f"dRV | {lk}" for lk in ordered_lines if f"dRV | {lk}" in df.columns]
if f"dRV | {CIV_KEY}" in df.columns:
    df = df.sort_values(f"dRV | {CIV_KEY}", ascending=False, na_position="last").reset_index(drop=True)

# ---------------- 5. HELPER FOR CLASSIFICATION & FITTING ----------------
def get_star_classification_data(df_in, use_equiv):
    results = {}
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin":
            continue
        is_clean = bool(df_in.at[i, "is_clean_bool"])
        votes_bin = 0
        votes_single = 0
        civ_status = None
        valid_lines = 0
        for lk in ordered_lines:
            _, _, frac = ew_fail_stats.get((sname, lk), (0, 0, 0))
            if frac > FAIL_FRAC_THRESH:
                continue
            col_name = f"dRV | {lk}"
            if col_name not in df_in.columns:
                continue
            val = df_in.at[i, col_name]
            if pd.isna(val) or not np.isfinite(float(val)):
                continue
            t_use = THRESH_KMS
            if use_equiv:
                t_eq = equiv_thresholds_map.get(lk, np.nan)
                if np.isfinite(t_eq):
                    t_use = t_eq
            is_bin = _is_significant_binary(sname, lk, float(val), t_use)
            if is_bin:
                votes_bin += 1
            else:
                votes_single += 1
            if lk == CIV_KEY:
                civ_status = is_bin
            valid_lines += 1
        if valid_lines == 0:
            continue
        agreement = max(votes_bin, votes_single) / valid_lines
        if agreement > 0.90:
            grade = 'Golden'
        elif agreement > 0.70:
            grade = 'Silver'
        else:
            grade = 'Bronze'
        if votes_bin > votes_single:
            final_bin = True
        elif votes_single > votes_bin:
            final_bin = False
        else:
            final_bin = civ_status if civ_status is not None else False
        results[sname] = {'grade': grade, 'is_binary': final_bin, 'is_clean': is_clean}
    return results

# === Chi-Squared 2-Segment Linear Fit ===
def _fit_two_segment_linear_weighted(x, y, y_err):
    if len(x) < 5:
        return None
    w = 1.0 / (y_err**2)
    w[~np.isfinite(w)] = 0.0001
    best_chi2 = np.inf
    best_res = None
    for i in range(2, len(x) - 2):
        x1, y1, w1 = x[:i], y[:i], w[:i]
        x2, y2, w2 = x[i:], y[i:], w[i:]
        try:
            p1 = np.polyfit(x1, y1, 1, w=np.sqrt(w1))
            fit1 = np.polyval(p1, x1)
            chi2_1 = np.sum(w1 * (y1 - fit1)**2)
        except Exception:
            chi2_1 = np.inf
            p1 = [0, 0]
        try:
            p2 = np.polyfit(x2, y2, 1, w=np.sqrt(w2))
            fit2 = np.polyval(p2, x2)
            chi2_2 = np.sum(w2 * (y2 - fit2)**2)
        except Exception:
            chi2_2 = np.inf
            p2 = [0, 0]
        total_chi2 = chi2_1 + chi2_2
        if total_chi2 < best_chi2:
            best_chi2 = total_chi2
            m1, c1 = p1
            m2, c2 = p2
            if abs(m1 - m2) > 1e-9:
                elbow_x = (c2 - c1) / (m1 - m2)
                elbow_y = m1 * elbow_x + c1
            else:
                elbow_x = x[i]
                elbow_y = y[i]
            best_res = {
                'elbow_x': elbow_x, 'elbow_y': elbow_y,
                'p1': p1, 'p2': p2, 'break_idx': i,
                'range1': (x[0], elbow_x),
                'range2': (elbow_x, x[-1]),
                'chi2': best_chi2
            }
    return best_res

# ---------------- 6. PLOTTING LOGIC ----------------

def wrap_header(colname: str) -> str:
    if colname in ("Star", "Clean", MEAN_COL, STD_COL):
        return colname
    return "\n".join(textwrap.wrap(colname.replace("dRV | ", "dRV\n"), width=12, break_long_words=True))

def wrap_str(s, width):
    if pd.isna(s):
        return s
    return "\n".join(textwrap.wrap(str(s), width=width, break_long_words=True)) or str(s)

def format_line_label(line_key, width=18, max_lines=2):
    s = str(line_key)
    if len(s) <= width:
        return s
    parts = textwrap.wrap(s, width=width, break_long_words=False, break_on_hyphens=False)
    if len(parts) <= max_lines:
        return "\n".join(parts)
    parts = parts[:max_lines]
    if not parts[-1].endswith("..."):
        if len(parts[-1]) > max(0, width - 3):
            parts[-1] = parts[-1][:max(0, width - 3)] + "..."
        else:
            parts[-1] = parts[-1] + "..."
    return "\n".join(parts)

def wilson_score_interval(k, n, z=1.0):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denominator = 1 + z**2/n
    center_adjusted_probability = (p + z**2 / (2*n)) / denominator
    error_margin = (z * np.sqrt((p*(1-p)/n) + (z**2/(4*n**2)))) / denominator
    low = center_adjusted_probability - error_margin
    high = center_adjusted_probability + error_margin
    return max(0.0, low), min(1.0, high)

def get_fit_elbow_threshold_for_line(df_in, line_key, filter_changes=True):
    cache_key = (line_key, bool(filter_changes))
    if cache_key in fit_elbow_thresholds_cache:
        return fit_elbow_thresholds_cache[cache_key]

    col = f"dRV | {line_key}"
    if col not in df_in.columns:
        fit_elbow_thresholds_cache[cache_key] = np.nan
        return np.nan

    valid_indices = []
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin":
            continue
        _, _, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH:
            continue
        v = df_in.at[i, col]
        if pd.notna(v) and np.isfinite(float(v)):
            valid_indices.append(i)

    if len(valid_indices) < 5:
        fit_elbow_thresholds_cache[cache_key] = np.nan
        return np.nan

    t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
    f_full = np.zeros_like(t_vals_full, dtype=float)
    el_full = np.zeros_like(t_vals_full, dtype=float)
    eh_full = np.zeros_like(t_vals_full, dtype=float)

    for k, t in enumerate(t_vals_full):
        n_pass = 0
        for idx in valid_indices:
            sname = df_in.at[idx, "Star"]
            dval = float(df_in.at[idx, col])
            if _is_significant_binary(sname, line_key, dval, t):
                n_pass += 1
        num = n_pass + EXTRA_ABOVE
        den = len(valid_indices) + EXTRA_ABOVE
        f = (num / den) if den > 0 else 0.0
        l, h = wilson_score_interval(num, den, z=1.0)
        f_full[k] = f
        el_full[k] = f - l
        eh_full[k] = h - f

    if filter_changes:
        keep = [0]
        prev = f_full[0]
        for k in range(1, len(f_full)):
            if abs(f_full[k] - prev) > 1e-9:
                keep.append(k)
                prev = f_full[k]
        t_vals = t_vals_full[keep]
        f_arr = f_full[keep]
        el = el_full[keep]
        eh = eh_full[keep]
    else:
        t_vals, f_arr, el, eh = t_vals_full, f_full, el_full, eh_full

    sigma = (el + eh) / 2.0
    sigma[~np.isfinite(sigma)] = 0.01
    sigma[sigma <= 0] = 0.01

    if len(t_vals) < 5:
        fit_elbow_thresholds_cache[cache_key] = np.nan
        return np.nan

    fit_res = _fit_two_segment_linear_weighted(t_vals, f_arr, sigma)
    if fit_res and np.isfinite(fit_res.get("elbow_x", np.nan)):
        elbow = float(np.clip(fit_res["elbow_x"], 1.0, float(MAX_RV_THRESH)))
    else:
        elbow = np.nan

    fit_elbow_thresholds_cache[cache_key] = elbow
    return elbow

def get_discrete_d2_threshold_for_line(df_in, line_key, filter_changes=True):
    cache_key = (line_key, bool(filter_changes))
    if cache_key in d2_thresholds_cache:
        return d2_thresholds_cache[cache_key]

    col = f"dRV | {line_key}"
    if col not in df_in.columns:
        d2_thresholds_cache[cache_key] = np.nan
        return np.nan

    valid_indices = []
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin":
            continue
        _, _, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH:
            continue
        v = df_in.at[i, col]
        if pd.notna(v) and np.isfinite(float(v)):
            valid_indices.append(i)

    if len(valid_indices) < 3:
        d2_thresholds_cache[cache_key] = np.nan
        return np.nan

    t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
    f_full = np.zeros_like(t_vals_full, dtype=float)

    for k, t in enumerate(t_vals_full):
        n_pass = 0
        for idx in valid_indices:
            sname = df_in.at[idx, "Star"]
            dval = float(df_in.at[idx, col])
            if _is_significant_binary(sname, line_key, dval, t):
                n_pass += 1
        num = n_pass + EXTRA_ABOVE
        den = len(valid_indices) + EXTRA_ABOVE
        f_full[k] = (num / den) if den > 0 else 0.0

    if filter_changes:
        keep = [0]
        prev = f_full[0]
        for k in range(1, len(f_full)):
            if abs(f_full[k] - prev) > 1e-9:
                keep.append(k)
                prev = f_full[k]
        t_vals = t_vals_full[keep]
        f_arr = f_full[keep]
    else:
        t_vals, f_arr = t_vals_full, f_full

    if len(t_vals) < 3:
        d2_thresholds_cache[cache_key] = np.nan
        return np.nan

    grad1 = np.gradient(f_arr, t_vals)
    grad2 = np.gradient(grad1, t_vals)

    if len(grad2) > 2:
        inner = np.arange(1, len(grad2) - 1)
        g2_inner = grad2[inner]
        t_inner = t_vals[inner]
        pos = g2_inner > 0
        if np.any(pos):
            t_d2 = float(t_inner[pos][np.argmax(g2_inner[pos])])
        else:
            t_d2 = np.nan
    else:
        t_d2 = np.nan

    d2_thresholds_cache[cache_key] = t_d2
    return t_d2


def _calc_fbin_stats_at_threshold(base_with, line_key, thresh_to_use):
    colname = f"dRV | {line_key}"
    if colname not in base_with.columns:
        return 0.0, 0.0, 0.0, 0, 0
    if thresh_to_use is None or not np.isfinite(float(thresh_to_use)):
        return 0.0, 0.0, 0.0, 0, 0

    thresh_to_use = float(thresh_to_use)
    vals = base_with[colname]

    idx_with_drv = []
    for i in range(len(base_with)):
        star_name = base_with.at[i, "Star"]
        if not isinstance(star_name, str) or star_name == "f_bin":
            continue
        _, _, frac = ew_fail_stats.get((star_name, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH:
            continue
        try:
            v = vals.iat[i]
            if pd.notna(v) and np.isfinite(float(v)):
                idx_with_drv.append(i)
        except Exception:
            pass

    n_total = len(idx_with_drv)
    if n_total == 0:
        return 0.0, 0.0, 0.0, 0, 0

    n_above = 0
    for i in idx_with_drv:
        star_name = base_with.at[i, "Star"]
        dval = float(vals.iat[i])
        if _is_significant_binary(star_name, line_key, dval, thresh_to_use):
            n_above += 1

    n_total_adj = n_total + EXTRA_ABOVE
    n_above_adj = n_above + EXTRA_ABOVE
    frac_adj = n_above_adj / n_total_adj if n_total_adj > 0 else 0.0

    low, high = wilson_score_interval(n_above_adj, n_total_adj, z=1.0)
    err_low = frac_adj - low
    err_high = high - frac_adj

    return frac_adj, err_low, err_high, n_above_adj, n_total_adj

def _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh):
    t = THRESH_KMS
    if use_equiv_thresh:
        t_eq = equiv_thresholds_map.get(line_key, np.nan)
        if np.isfinite(t_eq):
            t = float(t_eq)
    return _calc_fbin_stats_at_threshold(base_with, line_key, t)

def _get_fbin_simple(base_with, line_key, use_equiv_thresh):
    frac, el, eh, num, den = _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh)
    if den == 0:
        return "0/0 (—)"
    return f"{num}/{den} ({frac:.0%} +{eh:.0%}/-{el:.0%})"

def _get_fbin_float(base_with, line_key, use_equiv_thresh):
    f, _, _, _, _ = _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh)
    return f * 100.0

def _calc_ew_stats_for_line(line_key):
    failed = 0
    considered = 0
    all_ews = []

    for sname in STAR_CFG.keys():
        n_fail, n_cons, _ = ew_fail_stats.get((sname, line_key), (0, 0, 0))
        failed += n_fail
        considered += n_cons

    for (sname, lk, ep), rec in epoch_ew_data.items():
        if lk == line_key and isinstance(rec, dict):
            val = rec.get("EW")
            if val is not None and np.isfinite(val):
                all_ews.append(val)

    if considered > 0:
        success_rate = 1.0 - (failed / considered)
    else:
        success_rate = 0.0

    if len(all_ews) > 0:
        mean_ew = np.nanmean(all_ews)
        sem_ew = np.nanstd(all_ews) / np.sqrt(len(all_ews))
    else:
        mean_ew = np.nan
        sem_ew = np.nan

    return success_rate, mean_ew, sem_ew

def _get_wavelength(name):
    match = re.search(r"(\d+)", name)
    if match:
        return float(match.group(1))
    return 999999.0

def render_plots(sort_col, ascending):
    # --- STYLE SETTINGS ---
    if for_presentation:
        plt.rcParams.update({
            'font.size': 14,
            'axes.titlesize': 16,
            'axes.labelsize': 14,
            'xtick.labelsize': 12,
            'ytick.labelsize': 12,
            'legend.fontsize': 12
        })
        FS_NOTE = 12
        FS_TINY = 11
        FS_AXIS_LBL = 14
        FS_BAR_VAL = 12
        FS_BOX_STAT = 8
        FS_PIE_LABEL = 14
        FS_LIST_TEXT = 11
    else:
        plt.rcParams.update({
            'font.size': 10,
            'axes.titlesize': 12,
            'axes.labelsize': 10,
            'xtick.labelsize': 10,
            'ytick.labelsize': 10,
            'legend.fontsize': 10
        })
        FS_NOTE = 9
        FS_TINY = 8
        FS_AXIS_LBL = 10
        FS_BAR_VAL = 9
        FS_BOX_STAT = 7
        FS_PIE_LABEL = 11
        FS_LIST_TEXT = 9

    # Masking logic
    base = df.copy()[ordered_cols]
    masked = base.copy()
    for col in [c for c in masked.columns if c.startswith("dRV | ")]:
        line_key = col.replace("dRV | ", "")
        for i in range(len(masked)):
            sname = masked.at[i, "Star"]
            if not isinstance(sname, str) or sname == "f_bin":
                continue
            n_fail, n_cons, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
            if (n_cons > 0) and (frac > FAIL_FRAC_THRESH):
                if not state.get("force_show_low_detection", False):
                    masked.at[i, col] = np.nan

    sort_col_use = sort_col if sort_col in masked.columns else masked.columns[2]
    masked = masked.sort_values(sort_col_use, ascending=ascending, na_position="last").reset_index(drop=True)
    use_equiv = state.get("use_equiv_thresh", False)
    
    # -------------------------------------------------------------
    # NEW LOGIC: Determine Fit-Based Threshold FIRST for global use
    # -------------------------------------------------------------
    line_key_bar = sort_col_use.replace("dRV | ", "") if sort_col_use.startswith("dRV | ") else None
    
    # Calculate the linear fit "elbow" for the current line
    fitted_thresh_val = np.nan
    if line_key_bar:
        fitted_thresh_val = get_fit_elbow_threshold_for_line(masked, line_key_bar, filter_changes=state.get("filter_threshold_plot", True))

    # Apply global threshold logic: If we have a fit, use it. Else fallback to 10 or Equiv.
    current_thresh = THRESH_KMS
    thresh_source_label = f"Hard ({THRESH_KMS} km/s)"
    
    if np.isfinite(fitted_thresh_val):
        current_thresh = fitted_thresh_val
        thresh_source_label = f"Fit-Based ({fitted_thresh_val:.1f} km/s)"
    elif use_equiv and line_key_bar:
        t = equiv_thresholds_map.get(line_key_bar, THRESH_KMS)
        if np.isfinite(t):
            current_thresh = t
            thresh_source_label = f"Equiv ({t:.1f} km/s)"

    # --- PLOT 1: Bar/Point plot (|ΔRV| vs star) ---
    vals = masked[sort_col_use].to_numpy()
    x = np.arange(len(masked))

    colors = []
    yerrs = []
    for i in range(len(masked)):
        star_name = masked.at[i, "Star"]
        val = vals[i]
        if pd.isna(val):
            colors.append("gray")
            yerrs.append(0)
        elif line_key_bar:
            # Use the calculated GLOBAL threshold (current_thresh) for deciding Green vs Red
            is_sig = _is_significant_binary(star_name, line_key_bar, float(val), current_thresh)
            colors.append("green" if is_sig else "red")
            err = drverr_map.get((star_name, line_key_bar), 0)
            yerrs.append(err if np.isfinite(err) else 0)
        else:
            colors.append("green" if val >= current_thresh else "red")
            yerrs.append(0)

    plt.figure(figsize=BAR_FIGSIZE)
    plt.plot(x, vals, color='black', alpha=0.2, zorder=1)
    for i in range(len(x)):
        plt.errorbar(x[i], vals[i], yerr=yerrs[i], fmt='o',
                     color=colors[i], ecolor=colors[i], capsize=3, zorder=2)
    
    plt.axhline(current_thresh, linestyle="--", linewidth=1, color='blue',
                label=f"Active Threshold: {current_thresh:.1f} km/s")
    
    plt.xticks(x, [wrap_str(s, 10) for s in masked["Star"]], rotation=45, ha="right")
    plt.ylabel("|ΔRV| (km/s)")

    use_log = state.get("corner_log_scale", True)
    if use_log:
        plt.yscale('log')
        valid_pos = [v for v in vals if pd.notna(v) and v > 0]
        if valid_pos:
            plt.ylim(min(valid_pos) * 0.5, max(valid_pos) * 2.0)

    plt.title(f"Peak-to-peak |ΔRV| for {sort_col_use.replace('dRV | ', '')} ({'asc' if ascending else 'desc'})")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # =========================================================
    #    UPDATED PLOT 2: Binary Fraction vs Threshold
    #    Changes: Removed 2nd Derivative, Red Data Points
    # =========================================================
    if line_key_bar:
        valid_indices = []
        for i in range(len(masked)):
            sname = masked.at[i, "Star"]
            _, _, frac = ew_fail_stats.get((sname, line_key_bar), (0, 0, 0.0))
            if frac > FAIL_FRAC_THRESH:
                continue
            v = masked.at[i, sort_col_use]
            if pd.notna(v) and np.isfinite(float(v)):
                valid_indices.append(i)

        t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
        fractions_full = []
        err_lows_full = []
        err_highs_full = []

        for t in t_vals_full:
            n_pass = 0
            for idx in valid_indices:
                sname = masked.at[idx, "Star"]
                dval = float(masked.at[idx, sort_col_use])
                if _is_significant_binary(sname, line_key_bar, dval, t):
                    n_pass += 1

            num = n_pass + EXTRA_ABOVE
            den = len(valid_indices) + EXTRA_ABOVE
            f = (num / den) if den > 0 else 0.0

            l, h = wilson_score_interval(num, den, z=1.0)
            fractions_full.append(f)
            err_lows_full.append(f - l)
            err_highs_full.append(h - f)

        fractions_full = np.array(fractions_full)
        err_lows_full = np.array(err_lows_full)
        err_highs_full = np.array(err_highs_full)

        # Filtering (changes only)
        if state.get("filter_threshold_plot", True):
            keep = [0]
            prev_f = fractions_full[0]
            for k in range(1, len(fractions_full)):
                if abs(fractions_full[k] - prev_f) > 1e-9:
                    keep.append(k)
                    prev_f = fractions_full[k]
            t_vals_plot = t_vals_full[keep]
            frac_arr_plot = fractions_full[keep]
            err_lows_plot = err_lows_full[keep]
            err_highs_plot = err_highs_full[keep]
            plot_label = "Data (Changes Only)"
        else:
            t_vals_plot = t_vals_full
            frac_arr_plot = fractions_full
            err_lows_plot = err_lows_full
            err_highs_plot = err_highs_full
            plot_label = "Data (Full)"

        # Approx sigma for weighting / residual bars
        sigma_approx = (err_lows_plot + err_highs_plot) / 2.0
        sigma_approx[~np.isfinite(sigma_approx)] = 0.01
        sigma_approx[sigma_approx <= 0] = 0.01

        # Use 2 subplots instead of 3
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=FIGSIZE_FRAC, sharex=True,
            gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.10}
        )

        # Main plot: data with error bars (NOW RED)
        ax1.errorbar(
            t_vals_plot, frac_arr_plot,
            yerr=[err_lows_plot, err_highs_plot],
            fmt='o', markersize=4, linestyle='none',
            color='red', ecolor='red', capsize=3,  # CHANGED TO RED
            label=plot_label
        )

        fit_res = None
        if state.get("fit_elbow", True) and len(frac_arr_plot) > 5:
            fit_res = _fit_two_segment_linear_weighted(t_vals_plot, frac_arr_plot, sigma_approx)

        # Plot 2-segment fit and residuals + stats
        if fit_res:
            x1_range = np.linspace(fit_res['range1'][0], fit_res['elbow_x'], 50)
            y1_range = np.polyval(fit_res['p1'], x1_range)
            ax1.plot(x1_range, y1_range, color='orange', linewidth=2, label="2-Segment Fit")

            x2_range = np.linspace(fit_res['elbow_x'], fit_res['range2'][1], 50)
            y2_range = np.polyval(fit_res['p2'], x2_range)
            ax1.plot(x2_range, y2_range, color='orange', linewidth=2)

            # Fit threshold (elbow)
            ax1.axvline(fit_res['elbow_x'], color='orange', linestyle='--',
                        linewidth=1.5, label=f"Fit Thresh ({fit_res['elbow_x']:.1f})")
            ax2.axvline(fit_res['elbow_x'], color='orange', linestyle='--', linewidth=1.2, alpha=0.7)

            # Residuals relative to the 2-seg model
            y_fit_at_data = np.where(
                t_vals_plot <= fit_res['elbow_x'],
                np.polyval(fit_res['p1'], t_vals_plot),
                np.polyval(fit_res['p2'], t_vals_plot)
            )
            residuals = frac_arr_plot - y_fit_at_data

            ax2.axhline(0, color='black', linewidth=1)
            ax2.errorbar(
                t_vals_plot, residuals,
                yerr=sigma_approx,
                fmt='o', markersize=3, linestyle='none',
                color='purple', ecolor='purple', capsize=3
            )
            ax2.set_ylabel("Residuals")
            ax2.grid(True, alpha=0.3)

            # Reduced chi^2 and p-value (approx; dof ignores elbow-search model selection)
            chi2_val = float(np.sum((residuals / sigma_approx) ** 2))
            dof = int(max(len(residuals) - 4, 1))
            chi2_red = chi2_val / dof
            p_val = float(chi2.sf(chi2_val, dof))

            ax2.text(
                0.02, 0.95,
                rf"$\chi^2_\nu$={chi2_red:.2f}\n$p$={p_val:.2g}\nDoF={dof}",
                transform=ax2.transAxes,
                ha='left', va='top',
                fontsize=FS_TINY,
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1)
            )
        else:
            ax2.axhline(0, color='black', linewidth=1)
            ax2.text(0.5, 0.5, "2-segment fit not available", transform=ax2.transAxes,
                     ha='center', va='center')
            ax2.set_ylabel("Residuals")
            ax2.grid(True, alpha=0.3)

        # Threshold guide lines (DASHED) on ALL panels
        if np.isfinite(current_thresh):
             ax1.axvline(current_thresh, color='blue', linestyle='--', linewidth=1.5,
                        label=f'Global Used ({current_thresh:.1f})')
             ax2.axvline(current_thresh, color='blue', linestyle='--', linewidth=1.2, alpha=0.7)

        step_size = max(10, int(MAX_RV_THRESH / 10))
        ax2.set_xticks(np.arange(0, MAX_RV_THRESH + 1, step_size))
        ax2.set_xlabel("Threshold [km/s]")

        ax1.set_yticks(np.arange(0, 1.1, 0.1))
        ax1.set_ylabel(r"$f_{\mathrm{bin}}$")
        ax1.set_title(f"Binary Fraction vs Threshold: {line_key_bar}")
        ax1.grid(True, alpha=0.3, which='both')
        ax1.set_ylim(0, 1.05)
        ax1.legend(loc='upper right', fontsize=FS_TINY)

        plt.tight_layout()
        plt.show()

    # --- PLOT 3: Equivalent Thresholds across lines ---
    lines_plot = []
    thresholds_plot = []
    for lk in ordered_lines:
        t = equiv_thresholds_map.get(lk, np.nan)
        lines_plot.append(lk)
        thresholds_plot.append(t)

    if len(lines_plot) > 0:
        plt.figure(figsize=FIGSIZE_THRESH_PLOT)
        plt.plot(lines_plot, thresholds_plot, marker='o', linestyle='-', color='purple')
        plt.xticks(rotation=45, ha='right')
        plt.ylabel("Equivalent Threshold [km/s]")
        plt.title("Thresholds yielding $f_{\mathrm{bin}}$ ≈ $f_{\mathrm{bin}}$(CIV, 20 km/s)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    # --- PLOT 4: Corner plot (unchanged) ---
    show_names = state.get("show_corner_names", False)
    use_log = state.get("corner_log_scale", True)
    n_lines = len(ordered_lines)
    line_ranges = {}
    for lk in ordered_lines:
        col = f"dRV | {lk}"
        vals_rng = []
        
        # USE GLOBAL FIT THRESHOLD HERE
        t_val = current_thresh

        if np.isfinite(t_val):
            vals_rng.append(t_val)
        if col in masked.columns:
            for i in range(len(masked)):
                sname = masked.at[i, "Star"]
                _, _, frac = ew_fail_stats.get((sname, lk), (0, 0, 0))
                if frac > FAIL_FRAC_THRESH:
                    continue
                v = masked.at[i, col]
                if pd.notna(v) and np.isfinite(float(v)):
                    vals_rng.append(float(v))
        if not vals_rng:
            line_ranges[lk] = (0.1, MAX_RV_THRESH)
            continue

        vmin, vmax = min(vals_rng), max(vals_rng)
        if use_log:
            pos_vals = [x for x in vals_rng if x > 0]
            if pos_vals:
                vmin = min(pos_vals)
                log_min, log_max = np.log10(vmin), np.log10(vmax)
                span = log_max - log_min if log_max > log_min else 0.5
                vmin = 10**(log_min - 0.05 * span)
                vmax = 10**(log_max + 0.05 * span)
            else:
                vmin, vmax = 0.1, MAX_RV_THRESH
        else:
            span = vmax - vmin if vmax > vmin else 5.0
            vmin = vmin - 0.05 * span
            vmax = vmax + 0.05 * span
            if vmin < 0:
                vmin = 0
        line_ranges[lk] = (vmin, vmax)

    if n_lines > 1:
        dim = max(16, 3 * n_lines)
        fig, axes = plt.subplots(n_lines, n_lines, figsize=(dim, dim), sharex='col')
        plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, wspace=0, hspace=0)
        if n_lines == 1:
            axes = np.array([[axes]])

        trans_diagonal = mtransforms.Affine2D().rotate_deg(-45)

        for i in range(n_lines):
            line_y_key = ordered_lines[i]
            y_min, y_max = line_ranges.get(line_y_key, (0.1, MAX_RV_THRESH))
            thresh_y = current_thresh

            for j in range(n_lines):
                ax = axes[i, j]
                line_x_key = ordered_lines[j]
                x_min, x_max = line_ranges.get(line_x_key, (0.1, MAX_RV_THRESH))
                thresh_x = current_thresh

                if i == n_lines - 1:
                    ax.set_xlabel(format_line_label(line_x_key), fontsize=FS_AXIS_LBL)
                if j == 0 and i != j:
                    ax.set_ylabel(format_line_label(line_y_key), fontsize=FS_AXIS_LBL)

                if use_log:
                    ax.set_xscale('log')
                    if i != j:
                        ax.set_yscale('log')

                ax.set_xlim(x_min, x_max)
                if i != j:
                    ax.set_ylim(y_min, y_max)

                if i < n_lines - 1:
                    ax.tick_params(axis='x', labelbottom=False, length=0)
                if j > 0:
                    ax.tick_params(axis='y', labelleft=False, length=0)

                if j > i:
                    ax.axis('off')
                    continue

                if i == j:
                    col_name = f"dRV | {line_y_key}"
                    if col_name in masked.columns:
                        vals_hist = []
                        for row_idx in range(len(masked)):
                            sname = masked.at[row_idx, "Star"]
                            _, _, frac = ew_fail_stats.get((sname, line_y_key), (0, 0, 0))
                            if frac > FAIL_FRAC_THRESH:
                                continue
                            v = masked.at[row_idx, col_name]
                            if pd.notna(v) and np.isfinite(float(v)):
                                vals_hist.append(float(v))
                        if len(vals_hist) > 0:
                            if use_log:
                                hv_min = max(x_min, min([v for v in vals_hist if v > 0]) if any(v > 0 for v in vals_hist) else x_min)
                                hv_max = max(x_max, max(vals_hist))
                                bins = np.geomspace(hv_min, hv_max, 15)
                            else:
                                bins = 15
                            ax.hist(vals_hist, bins=bins, color='gray', alpha=0.7,
                                    histtype='stepfilled', edgecolor='black')
                    ax.yaxis.set_visible(False)
                    continue

                x_vals, y_vals, x_errs, y_errs = [], [], [], []
                c_topleft, c_botright = [], []
                names_to_plot = []
                col_x = f"dRV | {line_x_key}"
                col_y = f"dRV | {line_y_key}"
                if (col_x in masked.columns) and (col_y in masked.columns):
                    for row_idx in range(len(masked)):
                        sname = masked.at[row_idx, "Star"]
                        _, _, frac_x = ew_fail_stats.get((sname, line_x_key), (0, 0, 0))
                        _, _, frac_y = ew_fail_stats.get((sname, line_y_key), (0, 0, 0))
                        if (frac_x > FAIL_FRAC_THRESH) or (frac_y > FAIL_FRAC_THRESH):
                            continue
                        vx = masked.at[row_idx, col_x]
                        vy = masked.at[row_idx, col_y]
                        if pd.notna(vx) and np.isfinite(float(vx)) and pd.notna(vy) and np.isfinite(float(vy)):
                            vx = float(vx)
                            vy = float(vy)
                            if use_log and (vx <= 0 or vy <= 0):
                                continue
                            x_vals.append(vx)
                            y_vals.append(vy)
                            err_x = drverr_map.get((sname, line_x_key), 0.0)
                            err_y = drverr_map.get((sname, line_y_key), 0.0)
                            x_errs.append(err_x if np.isfinite(err_x) else 0.0)
                            y_errs.append(err_y if np.isfinite(err_y) else 0.0)
                            pass_y = _is_significant_binary(sname, line_y_key, vy, thresh_y)
                            c_topleft.append('green' if pass_y else 'red')
                            pass_x = _is_significant_binary(sname, line_x_key, vx, thresh_x)
                            c_botright.append('green' if pass_x else 'red')
                            names_to_plot.append(sname)

                if len(x_vals) > 0:
                    x_arr = np.array(x_vals)
                    y_arr = np.array(y_vals)
                    x_err_arr = np.array(x_errs)
                    y_err_arr = np.array(y_errs)

                    ax.axvline(thresh_x, color='black', linestyle=':', alpha=0.6, linewidth=1)
                    ax.axhline(thresh_y, color='black', linestyle=':', alpha=0.6, linewidth=1)
                    diag_min = max(x_min, y_min)
                    diag_max = min(x_max, y_max)
                    ax.plot([diag_min, diag_max], [diag_min, diag_max], ls='--',
                            color='gray', alpha=0.5, linewidth=1, zorder=0)

                    fbin_y_str = _get_fbin_simple(masked, line_y_key, use_equiv)
                    ax.text(0.05, 0.95, r"$f_{\mathrm{bin}}$(Y): " + fbin_y_str,
                            transform=ax.transAxes, fontsize=FS_NOTE, va='top', ha='left',
                            bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

                    fbin_x_str = _get_fbin_simple(masked, line_x_key, use_equiv)
                    ax.text(0.95, 0.05, r"$f_{\mathrm{bin}}$(X): " + fbin_x_str,
                            transform=ax.transAxes, fontsize=FS_NOTE, va='bottom', ha='right',
                            bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

                    ax.hlines(y_arr, x_arr - x_err_arr, x_arr + x_err_arr, colors=c_botright, lw=0.8, zorder=1)
                    ax.vlines(x_arr, y_arr - y_err_arr, y_arr + y_err_arr, colors=c_topleft, lw=0.8, zorder=1)

                    ax.scatter(x_vals, y_vals, c=c_botright,
                               marker=MarkerStyle('o', fillstyle='right', transform=trans_diagonal),
                               s=80, edgecolors='black', linewidth=0.5, zorder=2)
                    ax.scatter(x_vals, y_vals, c=c_topleft,
                               marker=MarkerStyle('o', fillstyle='left', transform=trans_diagonal),
                               s=80, edgecolors='black', linewidth=0.5, zorder=3)

                    if len(x_vals) > 1:
                        r_val, p_val = pearsonr(x_vals, y_vals)
                        ax.text(0.05, 0.85, f"r={r_val:.2f}\np={p_val:.2g}\nn={len(x_vals)}",
                                transform=ax.transAxes, fontsize=FS_NOTE, va='top', ha='left',
                                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8))

                    if show_names:
                        for k, nm in enumerate(names_to_plot):
                            ax.text(x_vals[k], y_vals[k], nm, fontsize=FS_TINY, alpha=0.8,
                                    clip_on=True, rotation=25)

                ax.grid(True, alpha=0.2, which='both')
        plt.show()

    # --- PLOT 5 + 6 (Agreement ranking + fbin vs emission line) ---
    drv_cols = [f"dRV | {lk}" for lk in ordered_lines if f"dRV | {lk}" in masked.columns]
    if len(drv_cols) > 1:
        cols_names = [c for c in drv_cols]
        adj_corr_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
        adj_pval_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
        adj_n_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)

        for i, c1 in enumerate(cols_names):
            data1 = masked[c1]
            for j, c2 in enumerate(cols_names):
                if i == j:
                    continue
                data2 = masked[c2]
                valid = data1.notna() & data2.notna()
                n_dots = valid.sum()
                if n_dots < MIN_STARS_FOR_CORR:
                    continue
                r, p = pearsonr(data1[valid], data2[valid])
                adj_corr_df.at[c1, c2] = r
                adj_pval_df.at[c1, c2] = p
                adj_n_df.at[c1, c2] = n_dots

        w_df = adj_n_df / MAX_POSSIBLE_STARS
        r_weighted_terms = adj_corr_df * w_df
        avg_scores_user = r_weighted_terms.sum(axis=1) / r_weighted_terms.count(axis=1)

        p_times_n = adj_pval_df * adj_n_df
        sum_n = adj_n_df.sum(axis=1)
        avg_pvals_std_weighted = p_times_n.sum(axis=1) / sum_n.replace(0, np.nan)

        best_friends, best_scores = [], []
        second_best_friends, second_best_scores = [], []
        stats_texts = []

        for idx_row, col in enumerate(cols_names):
            line_nm = col.replace("dRV | ", "")
            row_series = adj_corr_df.loc[col].sort_values(ascending=False)
            valid_friends = row_series.dropna()

            def _fmt_friend(friend_col, r_val):
                v = (masked[col].notna() & masked[friend_col].notna())
                n_count = v.sum()
                if n_count < 2:
                    return "Err", np.nan
                _, p_val = pearsonr(masked[col][v], masked[friend_col][v])
                nm = friend_col.replace("dRV | ", "")
                return f"{nm}\nr={r_val:.2f} (p={p_val:.2g}, n={n_count})", r_val

            if len(valid_friends) > 0:
                b_lbl, b_val = _fmt_friend(valid_friends.index[0], valid_friends.iloc[0])
            else:
                b_val, b_lbl = np.nan, "None"

            if len(valid_friends) > 1:
                s_lbl, s_val = _fmt_friend(valid_friends.index[1], valid_friends.iloc[1])
            else:
                s_val, s_lbl = np.nan, ""

            best_scores.append(b_val)
            best_friends.append(b_lbl)
            second_best_scores.append(s_val)
            second_best_friends.append(s_lbl)

            fbin_txt = _get_fbin_simple(masked, line_nm, use_equiv)
            ew_succ, ew_mn, ew_sem = _calc_ew_stats_for_line(line_nm)
            stats_texts.append(r"$f_{\mathrm{bin}}$: " + f"{fbin_txt}\nEW%: {ew_succ:.0%}\nEW: {ew_mn:.2f}±{ew_sem:.2f}")

        plot_data = pd.DataFrame({
            "Line": [c.replace("dRV | ", "") for c in cols_names],
            "Score": avg_scores_user.values,
            "Avg_P_Weighted": avg_pvals_std_weighted.values,
            "Best_Score": best_scores,
            "Best_Friend": best_friends,
            "Second_Score": second_best_scores,
            "Second_Friend": second_best_friends,
            "Stats": stats_texts
        }).sort_values("Score", ascending=False, na_position='last').reset_index(drop=True)

        plt.figure(figsize=FIGSIZE_AGREEMENT)
        valid_plot = plot_data.dropna(subset=["Score"])
        if not valid_plot.empty:
            plt.plot(valid_plot.index, valid_plot["Score"], marker='o', linestyle='-',
                     color='teal', linewidth=2, markersize=8,
                     label=f"Avg Score (Weight = N/{MAX_POSSIBLE_STARS})")

        show_extra = state.get("show_extra_data", False)
        if show_extra:
            plt.scatter(plot_data.index, plot_data["Best_Score"], color='orange', s=80,
                        marker='D', zorder=3, label="Max Correlation")
            plt.scatter(plot_data.index, plot_data["Second_Score"], color='lightgreen', s=70,
                        marker='s', edgecolors='green', zorder=2, label="2nd Best Correlation")

        plt.xticks(plot_data.index, plot_data["Line"], rotation=45, ha="right")
        plt.ylabel("Agreement Score")
        plt.title(f"Agreement Ranking (Logic: r * n/25, N >= {MIN_STARS_FOR_CORR})")
        plt.grid(True, alpha=0.3, linestyle='--')

        for i in plot_data.index:
            score = plot_data.at[i, "Score"]
            avg_p_w = plot_data.at[i, "Avg_P_Weighted"]
            best = plot_data.at[i, "Best_Score"]
            friend = plot_data.at[i, "Best_Friend"]
            second = plot_data.at[i, "Second_Score"]
            sec_friend = plot_data.at[i, "Second_Friend"]
            stat_txt = plot_data.at[i, "Stats"]

            if pd.notna(score):
                p_text = ""
                if show_extra and pd.notna(avg_p_w):
                    p_text = f"\n(p_w={avg_p_w:.2g})"
                plt.text(i, score - 0.02, f"{score:.3f}{p_text}",
                         ha='center', va='top', fontsize=FS_NOTE,
                         color='teal', fontweight='bold')

            if show_extra:
                if pd.notna(best):
                    plt.text(i, best + 0.01, friend, ha='center', va='bottom',
                             fontsize=FS_TINY, color='darkorange', rotation=25)
                if pd.notna(second):
                    plt.text(i, second - 0.01, sec_friend, ha='center', va='top',
                             fontsize=FS_TINY, color='green', rotation=25)

                ylim_min, ylim_max = plt.ylim()
                y_txt_pos = ylim_min + (ylim_max - ylim_min) * 0.05
                plt.text(i, y_txt_pos, stat_txt, ha='center', va='bottom',
                         fontsize=FS_BOX_STAT,
                         bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="gray", alpha=0.7))

        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.show()

        sort_by_agreement = state.get("sort_fbin_by_agreement", False)
        if sort_by_agreement and 'plot_data' in locals() and not plot_data.empty:
            sorted_lines_g6 = plot_data["Line"].tolist()
        else:
            sorted_lines_g6 = sorted(ordered_lines, key=_get_wavelength)

        # === UPDATED PLOT 6: Binary fraction per emission line ===
        show_hard_and_fit = state.get("graph6_show_hard_and_fit", False)
        filter_changes = state.get("filter_threshold_plot", True)

        x_pos = np.arange(len(sorted_lines_g6))
        plt.figure(figsize=FIGSIZE_FBIN_BAR)

        if show_hard_and_fit:
            # Hard series
            hard_vals, hard_el, hard_eh = [], [], []
            # Fit-threshold (2-line elbow) series
            fit_vals, fit_el, fit_eh = [], [], []
            # Discrete 2nd-derivative max-positive threshold series
            d2_vals, d2_el, d2_eh = [], [], []

            for lk in sorted_lines_g6:
                # --- Hard threshold ---
                f_h, el_h, eh_h, _, _ = _calc_fbin_stats_at_threshold(masked, lk, THRESH_KMS)
                hard_vals.append(f_h * 100.0)
                hard_el.append(el_h * 100.0)
                hard_eh.append(eh_h * 100.0)

                # --- Fit (2-seg elbow) threshold ---
                t_fit = get_fit_elbow_threshold_for_line(masked, lk, filter_changes=filter_changes)
                if np.isfinite(t_fit):
                    f_f, el_f, eh_f, _, _ = _calc_fbin_stats_at_threshold(masked, lk, t_fit)
                    fit_vals.append(f_f * 100.0)
                    fit_el.append(el_f * 100.0)
                    fit_eh.append(eh_f * 100.0)
                else:
                    fit_vals.append(np.nan)
                    fit_el.append(0.0)
                    fit_eh.append(0.0)

                # --- Max positive discrete 2nd-derivative threshold ---
                t_d2 = get_discrete_d2_threshold_for_line(masked, lk, filter_changes=filter_changes)
                if np.isfinite(t_d2):
                    f_d, el_d, eh_d, _, _ = _calc_fbin_stats_at_threshold(masked, lk, t_d2)
                    d2_vals.append(f_d * 100.0)
                    d2_el.append(el_d * 100.0)
                    d2_eh.append(eh_d * 100.0)
                else:
                    d2_vals.append(np.nan)
                    d2_el.append(0.0)
                    d2_eh.append(0.0)

            offset = 0.25
            plt.errorbar(x_pos - offset, hard_vals, yerr=[hard_el, hard_eh],
                         fmt='o', color='cornflowerblue', ecolor='gray',
                         capsize=5, elinewidth=2, markersize=7,
                         label=f"Hard ({THRESH_KMS:.0f} km/s)")

            plt.errorbar(x_pos, fit_vals, yerr=[fit_el, fit_eh],
                         fmt='o', color='darkorange', ecolor='gray',
                         capsize=5, elinewidth=2, markersize=7,
                         label="Fit threshold (2-line elbow)")

            plt.errorbar(x_pos + offset, d2_vals, yerr=[d2_el, d2_eh],
                         fmt='o', color='magenta', ecolor='gray',
                         capsize=5, elinewidth=2, markersize=7,
                         label="Max +d^2 threshold (discrete)")

            # Value labels
            for i, (hv, fv, dv) in enumerate(zip(hard_vals, fit_vals, d2_vals)):
                if np.isfinite(hv):
                    plt.text(i - offset, hv + 2, f"{hv:.0f}%", ha='center', va='bottom',
                             fontsize=max(FS_BAR_VAL - 2, 8), fontweight='bold', color='black')
                if np.isfinite(fv):
                    plt.text(i, fv + 2, f"{fv:.0f}%", ha='center', va='bottom',
                             fontsize=max(FS_BAR_VAL - 2, 8), fontweight='bold', color='black')
                if np.isfinite(dv):
                    plt.text(i + offset, dv + 2, f"{dv:.0f}%", ha='center', va='bottom',
                             fontsize=max(FS_BAR_VAL - 2, 8), fontweight='bold', color='black')

            plt.title("Binary Fraction per Emission Line (Hard vs Fit vs Max +d^2)")
            plt.legend(loc='upper right')

        else:
            # Original single series (hard or equiv depending on toggle)
            fbin_vals, err_lows, err_highs = [], [], []
            for lk in sorted_lines_g6:
                # Use current_thresh calculated at start of render_plots
                f, el, eh, _, _ = _calc_fbin_stats_at_threshold(masked, lk, current_thresh)
                fbin_vals.append(f * 100.0)
                err_lows.append(el * 100.0)
                err_highs.append(eh * 100.0)

            plt.errorbar(x_pos, fbin_vals, yerr=[err_lows, err_highs],
                         fmt='o', color='cornflowerblue', ecolor='gray',
                         capsize=5, elinewidth=2, markersize=8)

            for i, v in enumerate(fbin_vals):
                plt.text(i, v + 2, f"{v:.0f}%", ha='center', va='bottom',
                         fontsize=FS_BAR_VAL, fontweight='bold', color='black')

            plt.title(
                f"Binary Fraction per Emission Line ({thresh_source_label})"
            )

        plt.xticks(x_pos, sorted_lines_g6, rotation=45, ha='right')
        plt.ylabel("Binary Fraction (%)")
        plt.ylim(0, 105)
        plt.grid(axis='y', alpha=0.3, linestyle='--')
        plt.tight_layout()
        plt.show()

    # --- PLOT 7: Confidence Grading (Golden/Silver/Bronze) ---
    star_results = get_star_classification_data(df, use_equiv)
    highlight_contam = state.get("highlight_contam", False)

    grades_data = {
        'Golden': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []},
        'Silver': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []},
        'Bronze': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []}
    }

    for sname, info in star_results.items():
        grade = info['grade']
        is_bin = info['is_binary']
        is_clean = info['is_clean']
        if is_bin:
            if is_clean:
                grades_data[grade]['bin_clean'].append(sname)
            else:
                grades_data[grade]['bin_contam'].append(sname)
        else:
            if is_clean:
                grades_data[grade]['sin_clean'].append(sname)
            else:
                grades_data[grade]['sin_contam'].append(sname)

    fig, axes = plt.subplots(1, 3, figsize=(14, 8))
    theme_map = {
        'Golden': (['#FFD700', '#FFFACD'], '#DAA520'),
        'Silver': (['#A9A9A9', '#DCDCDC'], 'gray'),
        'Bronze': (['#CD853F', '#FFE4C4'], '#8B4513')
    }

    grade_keys = ['Golden', 'Silver', 'Bronze']
    titles = ["Golden (>90% Agree)", "Silver (>70% Agree)", "Bronze (Rest)"]

    for idx, g in enumerate(grade_keys):
        ax = axes[idx]
        data = grades_data[g]
        n_bin_clean = len(data['bin_clean'])
        n_bin_contam = len(data['bin_contam'])
        n_sin_clean = len(data['sin_clean'])
        n_sin_contam = len(data['sin_contam'])
        total = n_bin_clean + n_bin_contam + n_sin_clean + n_sin_contam

        colors_theme, title_col = theme_map[g]
        c_bin_main, c_sin_main = colors_theme

        if total > 0:
            if highlight_contam:
                n_bin = n_bin_clean + n_bin_contam
                n_sin = n_sin_clean + n_sin_contam
                outer_sizes = [n_bin, n_sin]
                outer_labels = ["Binary", "Single"]
                outer_colors = [c_bin_main, c_sin_main]

                f_out_sizes, f_out_labels, f_out_colors = [], [], []
                for s, l, c in zip(outer_sizes, outer_labels, outer_colors):
                    if s > 0:
                        f_out_sizes.append(s)
                        f_out_labels.append(l)
                        f_out_colors.append(c)

                ax.pie(
                    f_out_sizes, labels=f_out_labels, colors=f_out_colors,
                    radius=1.0, wedgeprops=dict(width=0.3, edgecolor='w'),
                    startangle=90, textprops={'fontsize': FS_PIE_LABEL, 'weight': 'bold'},
                    autopct='%1.1f%%', pctdistance=0.85
                )

                inner_sizes = [n_bin_clean, n_bin_contam, n_sin_clean, n_sin_contam]
                inner_colors = [c_bin_main, '#FF4444', c_sin_main, '#FF4444']

                f_in_sizes, f_in_colors = [], []
                for s, c in zip(inner_sizes, inner_colors):
                    if s > 0:
                        f_in_sizes.append(s)
                        f_in_colors.append(c)

                if f_in_sizes:
                    ax.pie(
                        f_in_sizes, colors=f_in_colors, radius=0.7,
                        wedgeprops=dict(width=0.7, edgecolor='w'), startangle=90,
                        autopct='%1.1f%%', pctdistance=0.6,
                        textprops={'fontsize': FS_TINY}
                    )
            else:
                n_bin = n_bin_clean + n_bin_contam
                n_sin = n_sin_clean + n_sin_contam
                ax.pie([n_bin, n_sin],
                       labels=[f"Binary\n{n_bin}", f"Single\n{n_sin}"],
                       colors=colors_theme, autopct='%1.1f%%', startangle=90,
                       textprops={'fontsize': FS_PIE_LABEL})
        else:
            ax.text(0.5, 0.5, "No Stars", ha='center', va='center', fontsize=FS_PIE_LABEL)
            ax.axis('off')

        ax.set_title(titles[idx], color=title_col, fontsize=FS_PIE_LABEL + 2, fontweight='bold')

        def _get_wrapped(lst):
            if not lst:
                return ""
            return "\n".join(textwrap.wrap(", ".join(lst), width=20))

        bin_clean_txt = _get_wrapped(data['bin_clean'])
        bin_contam_txt = _get_wrapped(data['bin_contam'])
        sin_clean_txt = _get_wrapped(data['sin_clean'])
        sin_contam_txt = _get_wrapped(data['sin_contam'])

        ax.text(0.05, -0.05, f"$\\bf{{Binary:}}$", transform=ax.transAxes,
                ha='left', va='top', fontsize=FS_LIST_TEXT)
        current_y = -0.12
        if bin_clean_txt:
            ax.text(0.05, current_y, bin_clean_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')
            current_y -= (bin_clean_txt.count('\n') + 2) * 0.05
        if highlight_contam and bin_contam_txt:
            ax.text(0.05, current_y, bin_contam_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT,
                    color='red', fontweight='bold')
        elif (not highlight_contam) and bin_contam_txt:
            ax.text(0.05, current_y, bin_contam_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')

        ax.text(0.55, -0.05, f"$\\bf{{Single:}}$", transform=ax.transAxes,
                ha='left', va='top', fontsize=FS_LIST_TEXT)
        current_y = -0.12
        if sin_clean_txt:
            ax.text(0.55, current_y, sin_clean_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')
            current_y -= (sin_clean_txt.count('\n') + 2) * 0.05
        if highlight_contam and sin_contam_txt:
            ax.text(0.55, current_y, sin_contam_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT,
                    color='red', fontweight='bold')
        elif (not highlight_contam) and sin_contam_txt:
            ax.text(0.55, current_y, sin_contam_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')

    plt.tight_layout()
    plt.show()

    # --- PLOT 8: All / Clean / Contaminated Comparison ---
    clean_counts = {'Binary': 0, 'Single': 0}
    contam_counts = {'Binary': 0, 'Single': 0}
    all_counts = {'Binary': 0, 'Single': 0}

    clean_lists = {'Binary': [], 'Single': []}
    contam_lists = {'Binary': [], 'Single': []}
    all_lists = {'Binary': [], 'Single': []}

    for sname, info in star_results.items():
        k = 'Binary' if info['is_binary'] else 'Single'
        all_counts[k] += 1
        all_lists[k].append(sname)
        if info['is_clean']:
            clean_counts[k] += 1
            clean_lists[k].append(sname)
        else:
            contam_counts[k] += 1
            contam_lists[k].append(sname)

    fig, axes = plt.subplots(1, 3, figsize=(16, 8))

    def plot_pie_with_lists(ax, counts, lists, title):
        n_bin = counts['Binary']
        n_sin = counts['Single']
        if (n_bin + n_sin) > 0:
            ax.pie([n_bin, n_sin],
                   labels=[f"Binary\n{n_bin}", f"Single\n{n_sin}"],
                   colors=['cornflowerblue', 'lightgray'],
                   autopct='%1.1f%%', startangle=90,
                   textprops={'fontsize': FS_PIE_LABEL})
        else:
            ax.text(0.5, 0.5, "No Data", ha='center', va='center', fontsize=FS_PIE_LABEL)
        ax.set_title(title, fontsize=FS_PIE_LABEL + 2, fontweight='bold')

        def _wrp(l):
            return "\n".join(textwrap.wrap(", ".join(l), width=20))

        bin_txt = _wrp(lists['Binary'])
        sin_txt = _wrp(lists['Single'])

        ax.text(0.05, -0.05, f"$\\bf{{Binary:}}$", transform=ax.transAxes,
                ha='left', va='top', fontsize=FS_LIST_TEXT)
        if bin_txt:
            ax.text(0.05, -0.12, bin_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT)

        ax.text(0.55, -0.05, f"$\\bf{{Single:}}$", transform=ax.transAxes,
                ha='left', va='top', fontsize=FS_LIST_TEXT)
        if sin_txt:
            ax.text(0.55, -0.12, sin_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT)

    plot_pie_with_lists(axes[0], all_counts, all_lists, "All Stars Together")
    plot_pie_with_lists(axes[1], clean_counts, clean_lists, "Clean Stars")
    plot_pie_with_lists(axes[2], contam_counts, contam_lists, "Contaminated Stars")

    plt.tight_layout()
    plt.show()

# ---------------- 7. WIDGETS & DISPLAY ----------------
default_sort = f"dRV | {CIV_KEY}" if f"dRV | {CIV_KEY}" in df.columns else (MEAN_COL if MEAN_COL in df.columns else "Star")

state = {
    "sort_col": default_sort,
    "ascending": False,
    "force_show_low_detection": False,
    "use_equiv_thresh": False,
    "show_corner_names": False,
    "corner_log_scale": True,
    "sort_fbin_by_agreement": False,
    "graph6_show_hard_and_fit": False,
    "show_extra_data": False,
    "highlight_contam": False,
    "fit_elbow": True,
    "filter_threshold_plot": True
}

btn_for_col = {}

def update_buttons():
    for col, btn in btn_for_col.items():
        active = (col == state["sort_col"])
        arrow = "↑" if (active and state["ascending"]) else ("↓" if active else "")
        btn.description = f"{wrap_header(col)} {arrow}".rstrip()
        btn.button_style = "info" if active else ""

def on_click(col):
    def _handler(_):
        if col == state["sort_col"]:
            state["ascending"] = not state["ascending"]
        else:
            state["sort_col"] = col
            state["ascending"] = False
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
        update_buttons()
    return _handler

show_low_detect_chk = widgets.Checkbox(value=False, description="Show >40% EW failures", indent=False)
def on_toggle_low_detect(change):
    if change["name"] == "value":
        state["force_show_low_detection"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
show_low_detect_chk.observe(on_toggle_low_detect)

use_equiv_thresh_chk = widgets.Checkbox(value=False, description="Use Equivalent Thresholds", indent=False)
def on_toggle_equiv(change):
    if change["name"] == "value":
        state["use_equiv_thresh"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
use_equiv_thresh_chk.observe(on_toggle_equiv)

corner_names_chk = widgets.Checkbox(value=False, description="Corner Plot: Show Names", indent=False)
def on_toggle_corner_names(change):
    if change["name"] == "value":
        state["show_corner_names"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
corner_names_chk.observe(on_toggle_corner_names)

corner_log_chk = widgets.Checkbox(value=True, description="Log Scale (All Plots)", indent=False)
def on_toggle_corner_log(change):
    if change["name"] == "value":
        state["corner_log_scale"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
corner_log_chk.observe(on_toggle_corner_log)

sort_fbin_chk = widgets.Checkbox(value=False, description="Graph 6: Sort by Agreement Score", indent=False)
def on_toggle_fbin_sort(change):
    if change["name"] == "value":
        state["sort_fbin_by_agreement"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
sort_fbin_chk.observe(on_toggle_fbin_sort)

graph6_hard_fit_chk = widgets.Checkbox(value=False, description="Graph 6: Show Hard + Fit + d^2 Threshold", indent=False)
def on_toggle_graph6_hard_fit(change):
    if change["name"] == "value":
        state["graph6_show_hard_and_fit"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
graph6_hard_fit_chk.observe(on_toggle_graph6_hard_fit)

show_extra_data_chk = widgets.Checkbox(value=False, description="Graph 5: Show Extra Data", indent=False)
def on_toggle_extra(change):
    if change["name"] == "value":
        state["show_extra_data"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
show_extra_data_chk.observe(on_toggle_extra)

highlight_contam_chk = widgets.Checkbox(value=False, description="Graph 7: Highlight Contaminated Stars", indent=False)
def on_toggle_contam(change):
    if change["name"] == "value":
        state["highlight_contam"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
highlight_contam_chk.observe(on_toggle_contam)

fit_elbow_chk = widgets.Checkbox(value=True, description="Fit Elbow (2-Line)", indent=False)
def on_toggle_fit_elbow(change):
    if change["name"] == "value":
        state["fit_elbow"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
fit_elbow_chk.observe(on_toggle_fit_elbow)

filter_thresh_chk = widgets.Checkbox(value=True, description="Filter Thresh Plot (Changes Only)", indent=False)
def on_toggle_filter(change):
    if change["name"] == "value":
        state["filter_threshold_plot"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
filter_thresh_chk.observe(on_toggle_filter)

buttons = []
for col in ordered_cols:
    b = widgets.Button(description=wrap_header(col), tooltip=f"Sort by {col}",
                       layout=widgets.Layout(width="auto"))
    b.on_click(on_click(col))
    btn_for_col[col] = b
    buttons.append(b)

controls = widgets.HBox([
    show_low_detect_chk,
    use_equiv_thresh_chk,
    corner_names_chk,
    corner_log_chk,
    sort_fbin_chk,
    graph6_hard_fit_chk,
    show_extra_data_chk,
    highlight_contam_chk,
    fit_elbow_chk,
    filter_thresh_chk
], layout=widgets.Layout(gap="12px", flex_flow="row wrap"))

headers = widgets.HBox(buttons, layout=widgets.Layout(flex_flow="row wrap", gap="6px"))
out = widgets.Output()

display(controls, headers, out)
with out:
    clear_output(wait=True)
    render_plots(state["sort_col"], state["ascending"])
update_buttons()

### gpt

In [ ]:
%matplotlib widget
# === ΔRV Analysis: Full Dashboard (User-Defined Weighting Logic) ===

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import matplotlib.ticker as ticker
from matplotlib.markers import MarkerStyle
import textwrap
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import pearsonr, chi2

# ---------------- 1. SETUP & CONFIG ----------------
SETTINGS_PATH = "ccf_settings_with_global_lines.json"
try:
    with open(SETTINGS_PATH, "r") as f:
        SETTINGS = json.load(f)
except FileNotFoundError:
    print(f"WARNING: Could not find {SETTINGS_PATH}. Using empty settings.")
    SETTINGS = {}

STAR_CFG = {s["star_name"]: s for s in SETTINGS.get("stars", [])}
LINES_DEFAULT = SETTINGS.get("emission_lines_default", {})

# --- PRESENTATION MODE ---
for_presentation = True

# Config
CIV_KEY = "C IV 5808-5812"
THRESH_KMS = 10.0
MAX_RV_THRESH = 400
EXTRA_ABOVE = 3
BAR_FIGSIZE = (12, 6)
FIGSIZE_FRAC = (12, 10)  # 3 subplots now (main + residuals + 2nd deriv)
FIGSIZE_THRESH_PLOT = (10, 5)
FIGSIZE_AGREEMENT = (12, 7)
FIGSIZE_FBIN_BAR = (12, 6)
FAIL_FRAC_THRESH = 0.40
MIN_STARS_FOR_CORR = 8
MAX_POSSIBLE_STARS = 25
KNOWN_ERR_KEYS = ("full_RV_err", "full_err", "sigma", "err", "RV_err", "RV_sigma")
MEAN_COL = "Mean ΔRV"
STD_COL = "Std ΔRV"

# Globals for storage
epoch_ew_data = {}
ew_fail_stats = {}
rv_epoch_cache = {}
drverr_map = {}
equiv_thresholds_map = {}
fit_elbow_thresholds_cache = {}  # (line_key, filter_changes_bool) -> elbow_x
d2_thresholds_cache = {}  # (line_key, filter_changes_bool) -> t_maxpos_d2

# ---------------- 2. DATA PROCESSING HELPERS ----------------

def _is_clean_star(star):
    try:
        for ep in star.get_all_epoch_numbers():
            flux = star.load_property('cleaned_normalized_flux', ep, 'COMBINED')
            if flux is not None:
                return False
        return True
    except Exception:
        return True

def _ew_record_for(star, epoch_num, line_key):
    try:
        EWs = star.load_property('EWs', epoch_num, 'COMBINED')
    except Exception:
        return None
    if EWs is None:
        return None
    try:
        rec = EWs.get(line_key)
    except Exception:
        return None
    if rec is None:
        return None
    try:
        rec_dict = rec.item()
    except Exception:
        return None
    if not isinstance(rec_dict, dict):
        return None

    def _to_float(x):
        try:
            return float(x)
        except Exception:
            return np.nan

    return {
        "EW": _to_float(rec_dict.get("EW")),
        "sigma_EW": _to_float(rec_dict.get("sigma_EW")),
        "SNR": _to_float(rec_dict.get("SNR")),
        "detected": rec_dict.get("detected")
    }

def _extract_full_rv(cell):
    try:
        if isinstance(cell, dict):
            return cell.get('full_RV', None)
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict):
                return v.get('full_RV', None)
        if hasattr(cell, "get"):
            return cell.get('full_RV', None)
    except Exception:
        pass
    return None

def _extract_full_rv_err(cell):
    try:
        if isinstance(cell, dict):
            for k in KNOWN_ERR_KEYS:
                if k in cell and cell[k] is not None:
                    return float(cell[k])
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict):
                for k in KNOWN_ERR_KEYS:
                    if k in v and v[k] is not None:
                        return float(v[k])
        if hasattr(cell, "get"):
            for k in KNOWN_ERR_KEYS:
                val = cell.get(k, None)
                if val is not None:
                    return float(val)
    except Exception:
        pass
    return np.nan

def _is_epoch_skipped_for_line(star_name, line_key, ep):
    cfg = STAR_CFG.get(star_name, {})
    if ep in set(cfg.get("skip_epochs", [])):
        return True
    sl = cfg.get("skip_emission_lines", {})
    if line_key in sl:
        skip_eps = sl[line_key]
        if isinstance(skip_eps, (int, np.integer)):
            skip_eps = [skip_eps]
        if 0 in skip_eps or ep in skip_eps:
            return True
        return False
    return False

def _line_delta_rv_for_star(star, line_key):
    rv_vals, cache = [], []
    sname = star.star_name

    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep):
            cache.append((ep, np.nan, np.nan, False))
            continue

        ewrec = _ew_record_for(star, ep, line_key)
        if ewrec is not None:
            epoch_ew_data[(sname, line_key, ep)] = ewrec

        RVs = star.load_property('RVs', ep, 'COMBINED')
        if not RVs or line_key not in RVs:
            cache.append((ep, np.nan, np.nan, False))
            continue

        cell = RVs[line_key]
        rv = _extract_full_rv(cell)

        if rv is None:
            cache.append((ep, np.nan, np.nan, False))
            continue

        try:
            rv = float(rv)
        except Exception:
            cache.append((ep, np.nan, np.nan, False))
            continue

        err = _extract_full_rv_err(cell)
        rv_vals.append((ep, rv, err))
        cache.append((ep, rv, err, True))

    rv_epoch_cache[(sname, line_key)] = cache

    if len(rv_vals) == 0:
        drverr_map[(sname, line_key)] = np.nan
        return np.nan

    ep_min, rv_min, err_min = min(rv_vals, key=lambda t: t[1])
    ep_max, rv_max, err_max = max(rv_vals, key=lambda t: t[1])
    dRV = abs(rv_max - rv_min)

    if np.isfinite(err_min) and np.isfinite(err_max):
        sigma_A = float(np.sqrt(err_min**2 + err_max**2))
    else:
        sigma_A = np.nan

    drverr_map[(sname, line_key)] = sigma_A
    return dRV

def _is_significant_binary(star_name: str, line_key: str, drv_val: float, threshold_val: float) -> bool:
    if not (pd.notna(drv_val) and np.isfinite(drv_val)):
        return False
    sigma_A = drverr_map.get((star_name, line_key), np.nan)
    if not np.isfinite(sigma_A):
        return False
    return (float(drv_val) >= threshold_val) and (float(drv_val) >= 4.0 * float(sigma_A))

def _compute_ew_fail_stats(star, line_key):
    sname = star.star_name
    considered = []
    failed = 0
    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep):
            continue
        considered.append(ep)
        rec = epoch_ew_data.get((sname, line_key, ep))
        if isinstance(rec, dict):
            det = rec.get("detected")
            if det is False:
                failed += 1
    n_considered = len(considered)
    frac = (failed / n_considered) if n_considered > 0 else 0.0
    return failed, n_considered, frac

# ---------------- 3. LOAD DATA & BUILD DATAFRAME ----------------
rows = []
ordered_lines = list(LINES_DEFAULT.keys())

for star_name in specs.star_names:
    star = obs.load_star_instance(star_name, to_print=False)
    is_clean_bool = _is_clean_star(star)
    row = {"Star": star_name, "Clean": "✓" if is_clean_bool else "X", "is_clean_bool": is_clean_bool}

    for lk in ordered_lines:
        dval = _line_delta_rv_for_star(star, lk)
        ew_fail_stats[(star_name, lk)] = _compute_ew_fail_stats(star, lk)
        row[f"dRV | {lk}"] = dval

    drvs = [v for k, v in row.items() if isinstance(k, str) and k.startswith("dRV | ")]
    row[MEAN_COL] = float(np.nanmean(drvs)) if np.isfinite(np.nanmean(drvs)) else np.nan
    row[STD_COL] = float(np.nanstd(drvs)) if np.isfinite(np.nanstd(drvs)) else np.nan
    rows.append(row)

df = pd.DataFrame(rows)

# ---------------- 4. CALCULATE THRESHOLDS ----------------
def calculate_equivalent_thresholds(df_in):
    if CIV_KEY not in ordered_lines:
        return
    ref_col = f"dRV | {CIV_KEY}"
    civ_vals = []

    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        _, _, frac = ew_fail_stats.get((sname, CIV_KEY), (0, 0, 0))
        if frac > FAIL_FRAC_THRESH:
            continue
        val = df_in.at[i, ref_col]
        if pd.notna(val) and np.isfinite(float(val)):
            civ_vals.append((sname, float(val)))

    n_above = 0
    for s, v in civ_vals:
        if _is_significant_binary(s, CIV_KEY, v, 20.0):
            n_above += 1

    num = n_above + EXTRA_ABOVE
    den = len(civ_vals) + EXTRA_ABOVE
    f_ref = num / den if den > 0 else 0

    equiv_thresholds_map.clear()
    search_space = np.arange(1, MAX_RV_THRESH + 1, 1)

    for lk in ordered_lines:
        col = f"dRV | {lk}"
        if col not in df_in.columns:
            continue
        line_vals = []
        for i in range(len(df_in)):
            sname = df_in.at[i, "Star"]
            _, _, frac = ew_fail_stats.get((sname, lk), (0, 0, 0))
            if frac > FAIL_FRAC_THRESH:
                continue
            val = df_in.at[i, col]
            if pd.notna(val) and np.isfinite(float(val)):
                line_vals.append((sname, float(val)))

        total_line = len(line_vals)
        if total_line == 0:
            equiv_thresholds_map[lk] = np.nan
            continue

        best_T = 20.0
        min_diff = 999.0

        for T in search_space:
            cnt = 0
            for s, v in line_vals:
                if _is_significant_binary(s, lk, v, T):
                    cnt += 1
            f_curr = (cnt + EXTRA_ABOVE) / (total_line + EXTRA_ABOVE)
            diff = abs(f_curr - f_ref)
            if diff < min_diff:
                min_diff = diff
                best_T = float(T)
        equiv_thresholds_map[lk] = best_T

calculate_equivalent_thresholds(df)

ordered_cols = ["Star", "Clean", MEAN_COL, STD_COL] + [f"dRV | {lk}" for lk in ordered_lines if f"dRV | {lk}" in df.columns]
if f"dRV | {CIV_KEY}" in df.columns:
    df = df.sort_values(f"dRV | {CIV_KEY}", ascending=False, na_position="last").reset_index(drop=True)

# ---------------- 5. HELPER FOR CLASSIFICATION & FITTING ----------------
def get_star_classification_data(df_in, use_equiv):
    results = {}
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin":
            continue
        is_clean = bool(df_in.at[i, "is_clean_bool"])
        votes_bin = 0
        votes_single = 0
        civ_status = None
        valid_lines = 0
        for lk in ordered_lines:
            _, _, frac = ew_fail_stats.get((sname, lk), (0, 0, 0))
            if frac > FAIL_FRAC_THRESH:
                continue
            col_name = f"dRV | {lk}"
            if col_name not in df_in.columns:
                continue
            val = df_in.at[i, col_name]
            if pd.isna(val) or not np.isfinite(float(val)):
                continue
            t_use = THRESH_KMS
            if use_equiv:
                t_eq = equiv_thresholds_map.get(lk, np.nan)
                if np.isfinite(t_eq):
                    t_use = t_eq
            is_bin = _is_significant_binary(sname, lk, float(val), t_use)
            if is_bin:
                votes_bin += 1
            else:
                votes_single += 1
            if lk == CIV_KEY:
                civ_status = is_bin
            valid_lines += 1
        if valid_lines == 0:
            continue
        agreement = max(votes_bin, votes_single) / valid_lines
        if agreement > 0.90:
            grade = 'Golden'
        elif agreement > 0.70:
            grade = 'Silver'
        else:
            grade = 'Bronze'
        if votes_bin > votes_single:
            final_bin = True
        elif votes_single > votes_bin:
            final_bin = False
        else:
            final_bin = civ_status if civ_status is not None else False
        results[sname] = {'grade': grade, 'is_binary': final_bin, 'is_clean': is_clean}
    return results

# === Chi-Squared 2-Segment Linear Fit ===
def _fit_two_segment_linear_weighted(x, y, y_err):
    if len(x) < 5:
        return None
    w = 1.0 / (y_err**2)
    w[~np.isfinite(w)] = 0.0001
    best_chi2 = np.inf
    best_res = None
    for i in range(2, len(x) - 2):
        x1, y1, w1 = x[:i], y[:i], w[:i]
        x2, y2, w2 = x[i:], y[i:], w[i:]
        try:
            p1 = np.polyfit(x1, y1, 1, w=np.sqrt(w1))
            fit1 = np.polyval(p1, x1)
            chi2_1 = np.sum(w1 * (y1 - fit1)**2)
        except Exception:
            chi2_1 = np.inf
            p1 = [0, 0]
        try:
            p2 = np.polyfit(x2, y2, 1, w=np.sqrt(w2))
            fit2 = np.polyval(p2, x2)
            chi2_2 = np.sum(w2 * (y2 - fit2)**2)
        except Exception:
            chi2_2 = np.inf
            p2 = [0, 0]
        total_chi2 = chi2_1 + chi2_2
        if total_chi2 < best_chi2:
            best_chi2 = total_chi2
            m1, c1 = p1
            m2, c2 = p2
            if abs(m1 - m2) > 1e-9:
                elbow_x = (c2 - c1) / (m1 - m2)
                elbow_y = m1 * elbow_x + c1
            else:
                elbow_x = x[i]
                elbow_y = y[i]
            best_res = {
                'elbow_x': elbow_x, 'elbow_y': elbow_y,
                'p1': p1, 'p2': p2, 'break_idx': i,
                'range1': (x[0], elbow_x),
                'range2': (elbow_x, x[-1]),
                'chi2': best_chi2
            }
    return best_res

# ---------------- 6. PLOTTING LOGIC ----------------

def wrap_header(colname: str) -> str:
    if colname in ("Star", "Clean", MEAN_COL, STD_COL):
        return colname
    return "\n".join(textwrap.wrap(colname.replace("dRV | ", "dRV\n"), width=12, break_long_words=True))

def wrap_str(s, width):
    if pd.isna(s):
        return s
    return "\n".join(textwrap.wrap(str(s), width=width, break_long_words=True)) or str(s)

def format_line_label(line_key, width=18, max_lines=2):
    """Corner-plot axis label helper: single line if possible, else wrap to <=2 lines."""
    s = str(line_key)
    if len(s) <= width:
        return s
    parts = textwrap.wrap(s, width=width, break_long_words=False, break_on_hyphens=False)
    if len(parts) <= max_lines:
        return "\n".join(parts)
    parts = parts[:max_lines]
    # add ellipsis to last line if truncated
    if not parts[-1].endswith("..."):
        if len(parts[-1]) > max(0, width - 3):
            parts[-1] = parts[-1][:max(0, width - 3)] + "..."
        else:
            parts[-1] = parts[-1] + "..."
    return "\n".join(parts)

def wilson_score_interval(k, n, z=1.0):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denominator = 1 + z**2/n
    center_adjusted_probability = (p + z**2 / (2*n)) / denominator
    error_margin = (z * np.sqrt((p*(1-p)/n) + (z**2/(4*n**2)))) / denominator
    low = center_adjusted_probability - error_margin
    high = center_adjusted_probability + error_margin
    return max(0.0, low), min(1.0, high)

def get_fit_elbow_threshold_for_line(df_in, line_key, filter_changes=True):
    """Return elbow_x from the 2-seg fit for f_bin(threshold) for a given emission line."""
    cache_key = (line_key, bool(filter_changes))
    if cache_key in fit_elbow_thresholds_cache:
        return fit_elbow_thresholds_cache[cache_key]

    col = f"dRV | {line_key}"
    if col not in df_in.columns:
        fit_elbow_thresholds_cache[cache_key] = np.nan
        return np.nan

    valid_indices = []
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin":
            continue
        _, _, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH:
            continue
        v = df_in.at[i, col]
        if pd.notna(v) and np.isfinite(float(v)):
            valid_indices.append(i)

    if len(valid_indices) < 5:
        fit_elbow_thresholds_cache[cache_key] = np.nan
        return np.nan

    t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
    f_full = np.zeros_like(t_vals_full, dtype=float)
    el_full = np.zeros_like(t_vals_full, dtype=float)
    eh_full = np.zeros_like(t_vals_full, dtype=float)

    for k, t in enumerate(t_vals_full):
        n_pass = 0
        for idx in valid_indices:
            sname = df_in.at[idx, "Star"]
            dval = float(df_in.at[idx, col])
            if _is_significant_binary(sname, line_key, dval, t):
                n_pass += 1
        num = n_pass + EXTRA_ABOVE
        den = len(valid_indices) + EXTRA_ABOVE
        f = (num / den) if den > 0 else 0.0
        l, h = wilson_score_interval(num, den, z=1.0)
        f_full[k] = f
        el_full[k] = f - l
        eh_full[k] = h - f

    if filter_changes:
        keep = [0]
        prev = f_full[0]
        for k in range(1, len(f_full)):
            if abs(f_full[k] - prev) > 1e-9:
                keep.append(k)
                prev = f_full[k]
        t_vals = t_vals_full[keep]
        f_arr = f_full[keep]
        el = el_full[keep]
        eh = eh_full[keep]
    else:
        t_vals, f_arr, el, eh = t_vals_full, f_full, el_full, eh_full

    sigma = (el + eh) / 2.0
    sigma[~np.isfinite(sigma)] = 0.01
    sigma[sigma <= 0] = 0.01

    if len(t_vals) < 5:
        fit_elbow_thresholds_cache[cache_key] = np.nan
        return np.nan

    fit_res = _fit_two_segment_linear_weighted(t_vals, f_arr, sigma)
    if fit_res and np.isfinite(fit_res.get("elbow_x", np.nan)):
        elbow = float(np.clip(fit_res["elbow_x"], 1.0, float(MAX_RV_THRESH)))
    else:
        elbow = np.nan

    fit_elbow_thresholds_cache[cache_key] = elbow
    return elbow

def get_discrete_d2_threshold_for_line(df_in, line_key, filter_changes=True):
    """Return threshold where the discrete 2nd derivative of f_bin(threshold) is maximally positive."""
    cache_key = (line_key, bool(filter_changes))
    if cache_key in d2_thresholds_cache:
        return d2_thresholds_cache[cache_key]

    col = f"dRV | {line_key}"
    if col not in df_in.columns:
        d2_thresholds_cache[cache_key] = np.nan
        return np.nan

    valid_indices = []
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin":
            continue
        _, _, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH:
            continue
        v = df_in.at[i, col]
        if pd.notna(v) and np.isfinite(float(v)):
            valid_indices.append(i)

    if len(valid_indices) < 3:
        d2_thresholds_cache[cache_key] = np.nan
        return np.nan

    t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
    f_full = np.zeros_like(t_vals_full, dtype=float)

    for k, t in enumerate(t_vals_full):
        n_pass = 0
        for idx in valid_indices:
            sname = df_in.at[idx, "Star"]
            dval = float(df_in.at[idx, col])
            if _is_significant_binary(sname, line_key, dval, t):
                n_pass += 1
        num = n_pass + EXTRA_ABOVE
        den = len(valid_indices) + EXTRA_ABOVE
        f_full[k] = (num / den) if den > 0 else 0.0

    # Optional "changes only" filtering
    if filter_changes:
        keep = [0]
        prev = f_full[0]
        for k in range(1, len(f_full)):
            if abs(f_full[k] - prev) > 1e-9:
                keep.append(k)
                prev = f_full[k]
        t_vals = t_vals_full[keep]
        f_arr = f_full[keep]
    else:
        t_vals, f_arr = t_vals_full, f_full

    if len(t_vals) < 3:
        d2_thresholds_cache[cache_key] = np.nan
        return np.nan

    grad1 = np.gradient(f_arr, t_vals)
    grad2 = np.gradient(grad1, t_vals)

    # Avoid edge artifacts by searching interior points only
    if len(grad2) > 2:
        inner = np.arange(1, len(grad2) - 1)
        g2_inner = grad2[inner]
        t_inner = t_vals[inner]
        pos = g2_inner > 0
        if np.any(pos):
            t_d2 = float(t_inner[pos][np.argmax(g2_inner[pos])])
        else:
            t_d2 = np.nan
    else:
        t_d2 = np.nan

    d2_thresholds_cache[cache_key] = t_d2
    return t_d2


def _calc_fbin_stats_at_threshold(base_with, line_key, thresh_to_use):
    colname = f"dRV | {line_key}"
    if colname not in base_with.columns:
        return 0.0, 0.0, 0.0, 0, 0
    if thresh_to_use is None or not np.isfinite(float(thresh_to_use)):
        return 0.0, 0.0, 0.0, 0, 0

    thresh_to_use = float(thresh_to_use)
    vals = base_with[colname]

    idx_with_drv = []
    for i in range(len(base_with)):
        star_name = base_with.at[i, "Star"]
        if not isinstance(star_name, str) or star_name == "f_bin":
            continue
        _, _, frac = ew_fail_stats.get((star_name, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH:
            continue
        try:
            v = vals.iat[i]
            if pd.notna(v) and np.isfinite(float(v)):
                idx_with_drv.append(i)
        except Exception:
            pass

    n_total = len(idx_with_drv)
    if n_total == 0:
        return 0.0, 0.0, 0.0, 0, 0

    n_above = 0
    for i in idx_with_drv:
        star_name = base_with.at[i, "Star"]
        dval = float(vals.iat[i])
        if _is_significant_binary(star_name, line_key, dval, thresh_to_use):
            n_above += 1

    n_total_adj = n_total + EXTRA_ABOVE
    n_above_adj = n_above + EXTRA_ABOVE
    frac_adj = n_above_adj / n_total_adj if n_total_adj > 0 else 0.0

    low, high = wilson_score_interval(n_above_adj, n_total_adj, z=1.0)
    err_low = frac_adj - low
    err_high = high - frac_adj

    return frac_adj, err_low, err_high, n_above_adj, n_total_adj

def _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh):
    thresh_to_use = THRESH_KMS
    if use_equiv_thresh:
        t = equiv_thresholds_map.get(line_key, np.nan)
        if np.isfinite(t):
            thresh_to_use = float(t)
    return _calc_fbin_stats_at_threshold(base_with, line_key, thresh_to_use)

def _get_fbin_simple(base_with, line_key, use_equiv_thresh):
    frac, el, eh, num, den = _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh)
    if den == 0:
        return "0/0 (—)"
    return f"{num}/{den} ({frac:.0%} +{eh:.0%}/-{el:.0%})"

def _get_fbin_float(base_with, line_key, use_equiv_thresh):
    f, _, _, _, _ = _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh)
    return f * 100.0

def _calc_ew_stats_for_line(line_key):
    failed = 0
    considered = 0
    all_ews = []

    for sname in STAR_CFG.keys():
        n_fail, n_cons, _ = ew_fail_stats.get((sname, line_key), (0, 0, 0))
        failed += n_fail
        considered += n_cons

    for (sname, lk, ep), rec in epoch_ew_data.items():
        if lk == line_key and isinstance(rec, dict):
            val = rec.get("EW")
            if val is not None and np.isfinite(val):
                all_ews.append(val)

    if considered > 0:
        success_rate = 1.0 - (failed / considered)
    else:
        success_rate = 0.0

    if len(all_ews) > 0:
        mean_ew = np.nanmean(all_ews)
        sem_ew = np.nanstd(all_ews) / np.sqrt(len(all_ews))
    else:
        mean_ew = np.nan
        sem_ew = np.nan

    return success_rate, mean_ew, sem_ew

def _get_wavelength(name):
    match = re.search(r"(\d+)", name)
    if match:
        return float(match.group(1))
    return 999999.0

def render_plots(sort_col, ascending):
    # --- STYLE SETTINGS ---
    if for_presentation:
        plt.rcParams.update({
            'font.size': 14,
            'axes.titlesize': 16,
            'axes.labelsize': 14,
            'xtick.labelsize': 12,
            'ytick.labelsize': 12,
            'legend.fontsize': 12
        })
        FS_NOTE = 12
        FS_TINY = 11
        FS_AXIS_LBL = 14
        FS_BAR_VAL = 12
        FS_BOX_STAT = 8
        FS_PIE_LABEL = 14
        FS_LIST_TEXT = 11
    else:
        plt.rcParams.update({
            'font.size': 10,
            'axes.titlesize': 12,
            'axes.labelsize': 10,
            'xtick.labelsize': 10,
            'ytick.labelsize': 10,
            'legend.fontsize': 10
        })
        FS_NOTE = 9
        FS_TINY = 8
        FS_AXIS_LBL = 10
        FS_BAR_VAL = 9
        FS_BOX_STAT = 7
        FS_PIE_LABEL = 11
        FS_LIST_TEXT = 9

    # Masking logic
    base = df.copy()[ordered_cols]
    masked = base.copy()
    for col in [c for c in masked.columns if c.startswith("dRV | ")]:
        line_key = col.replace("dRV | ", "")
        for i in range(len(masked)):
            sname = masked.at[i, "Star"]
            if not isinstance(sname, str) or sname == "f_bin":
                continue
            n_fail, n_cons, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
            if (n_cons > 0) and (frac > FAIL_FRAC_THRESH):
                if not state.get("force_show_low_detection", False):
                    masked.at[i, col] = np.nan

    sort_col_use = sort_col if sort_col in masked.columns else masked.columns[2]
    masked = masked.sort_values(sort_col_use, ascending=ascending, na_position="last").reset_index(drop=True)
    use_equiv = state.get("use_equiv_thresh", False)

    # --- PLOT 1: Bar/Point plot (|ΔRV| vs star) ---
    vals = masked[sort_col_use].to_numpy()
    x = np.arange(len(masked))
    line_key_bar = sort_col_use.replace("dRV | ", "") if sort_col_use.startswith("dRV | ") else None
    current_thresh = THRESH_KMS
    if use_equiv and line_key_bar:
        t = equiv_thresholds_map.get(line_key_bar, THRESH_KMS)
        if np.isfinite(t):
            current_thresh = t

    colors = []
    yerrs = []
    for i in range(len(masked)):
        star_name = masked.at[i, "Star"]
        val = vals[i]
        if pd.isna(val):
            colors.append("gray")
            yerrs.append(0)
        elif line_key_bar:
            is_sig = _is_significant_binary(star_name, line_key_bar, float(val), current_thresh)
            colors.append("green" if is_sig else "red")
            err = drverr_map.get((star_name, line_key_bar), 0)
            yerrs.append(err if np.isfinite(err) else 0)
        else:
            colors.append("green" if val >= current_thresh else "red")
            yerrs.append(0)

    plt.figure(figsize=BAR_FIGSIZE)
    plt.plot(x, vals, color='black', alpha=0.2, zorder=1)
    for i in range(len(x)):
        plt.errorbar(x[i], vals[i], yerr=yerrs[i], fmt='o',
                     color=colors[i], ecolor=colors[i], capsize=3, zorder=2)
    plt.axhline(current_thresh, linestyle="--", linewidth=1, color='gray',
                label=f"Threshold {current_thresh:.1f} km/s")
    plt.xticks(x, [wrap_str(s, 10) for s in masked["Star"]], rotation=45, ha="right")
    plt.ylabel("|ΔRV| (km/s)")

    use_log = state.get("corner_log_scale", True)
    if use_log:
        plt.yscale('log')
        valid_pos = [v for v in vals if pd.notna(v) and v > 0]
        if valid_pos:
            plt.ylim(min(valid_pos) * 0.5, max(valid_pos) * 2.0)

    plt.title(f"Peak-to-peak |ΔRV| for {sort_col_use.replace('dRV | ', '')} ({'asc' if ascending else 'desc'})")
    plt.tight_layout()
    plt.show()

    # =========================================================
    #   UPDATED PLOT 2: Binary Fraction vs Threshold (2-Seg Fit ONLY) + Residuals + 2nd Deriv
    # =========================================================
    if line_key_bar:
        valid_indices = []
        for i in range(len(masked)):
            sname = masked.at[i, "Star"]
            _, _, frac = ew_fail_stats.get((sname, line_key_bar), (0, 0, 0.0))
            if frac > FAIL_FRAC_THRESH:
                continue
            v = masked.at[i, sort_col_use]
            if pd.notna(v) and np.isfinite(float(v)):
                valid_indices.append(i)

        t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
        fractions_full = []
        err_lows_full = []
        err_highs_full = []

        for t in t_vals_full:
            n_pass = 0
            for idx in valid_indices:
                sname = masked.at[idx, "Star"]
                dval = float(masked.at[idx, sort_col_use])
                if _is_significant_binary(sname, line_key_bar, dval, t):
                    n_pass += 1

            num = n_pass + EXTRA_ABOVE
            den = len(valid_indices) + EXTRA_ABOVE
            f = (num / den) if den > 0 else 0.0

            l, h = wilson_score_interval(num, den, z=1.0)
            fractions_full.append(f)
            err_lows_full.append(f - l)
            err_highs_full.append(h - f)

        fractions_full = np.array(fractions_full)
        err_lows_full = np.array(err_lows_full)
        err_highs_full = np.array(err_highs_full)

        # Filtering (changes only)
        if state.get("filter_threshold_plot", True):
            keep = [0]
            prev_f = fractions_full[0]
            for k in range(1, len(fractions_full)):
                if abs(fractions_full[k] - prev_f) > 1e-9:
                    keep.append(k)
                    prev_f = fractions_full[k]
            t_vals_plot = t_vals_full[keep]
            frac_arr_plot = fractions_full[keep]
            err_lows_plot = err_lows_full[keep]
            err_highs_plot = err_highs_full[keep]
            plot_label = "Data (Changes Only)"
        else:
            t_vals_plot = t_vals_full
            frac_arr_plot = fractions_full
            err_lows_plot = err_lows_full
            err_highs_plot = err_highs_full
            plot_label = "Data (Full)"

        # Approx sigma for weighting / residual bars
        sigma_approx = (err_lows_plot + err_highs_plot) / 2.0
        sigma_approx[~np.isfinite(sigma_approx)] = 0.01
        sigma_approx[sigma_approx <= 0] = 0.01

        # Discrete 2nd derivative (for automated threshold + third panel)
        grad2 = None
        t_d2 = np.nan
        if len(frac_arr_plot) >= 3:
            grad1 = np.gradient(frac_arr_plot, t_vals_plot)
            grad2 = np.gradient(grad1, t_vals_plot)
            if len(grad2) > 2:
                inner = np.arange(1, len(grad2) - 1)
                g2_inner = grad2[inner]
                t_inner = t_vals_plot[inner]
                pos = g2_inner > 0
                if np.any(pos):
                    t_d2 = float(t_inner[pos][np.argmax(g2_inner[pos])])

        fig, (ax1, ax2, ax3) = plt.subplots(
            3, 1, figsize=FIGSIZE_FRAC, sharex=True,
            gridspec_kw={'height_ratios': [3, 1, 1], 'hspace': 0.10}
        )

        # Main plot: data with error bars
        ax1.errorbar(
            t_vals_plot, frac_arr_plot,
            yerr=[err_lows_plot, err_highs_plot],
            fmt='o', markersize=4, linestyle='none',
            color='blue', ecolor='blue', capsize=3,
            label=plot_label
        )

        fit_res = None
        if state.get("fit_elbow", True) and len(frac_arr_plot) > 5:
            fit_res = _fit_two_segment_linear_weighted(t_vals_plot, frac_arr_plot, sigma_approx)

        # Plot 2-segment fit and residuals + stats
        if fit_res:
            x1_range = np.linspace(fit_res['range1'][0], fit_res['elbow_x'], 50)
            y1_range = np.polyval(fit_res['p1'], x1_range)
            ax1.plot(x1_range, y1_range, color='orange', linewidth=2, label="2-Segment Fit")

            x2_range = np.linspace(fit_res['elbow_x'], fit_res['range2'][1], 50)
            y2_range = np.polyval(fit_res['p2'], x2_range)
            ax1.plot(x2_range, y2_range, color='orange', linewidth=2)

            # Fit threshold (elbow)
            ax1.axvline(fit_res['elbow_x'], color='orange', linestyle='--',
                        linewidth=1.5, label=f"Fit Thresh ({fit_res['elbow_x']:.1f})")
            ax2.axvline(fit_res['elbow_x'], color='orange', linestyle='--', linewidth=1.2, alpha=0.7)
            ax3.axvline(fit_res['elbow_x'], color='orange', linestyle='--', linewidth=1.2, alpha=0.7)

            # Residuals relative to the 2-seg model
            y_fit_at_data = np.where(
                t_vals_plot <= fit_res['elbow_x'],
                np.polyval(fit_res['p1'], t_vals_plot),
                np.polyval(fit_res['p2'], t_vals_plot)
            )
            residuals = frac_arr_plot - y_fit_at_data

            ax2.axhline(0, color='black', linewidth=1)
            ax2.errorbar(
                t_vals_plot, residuals,
                yerr=sigma_approx,
                fmt='o', markersize=3, linestyle='none',
                color='purple', ecolor='purple', capsize=3
            )
            ax2.set_ylabel("Residuals")
            ax2.grid(True, alpha=0.3)

            # Reduced chi^2 and p-value (approx; dof ignores elbow-search model selection)
            chi2_val = float(np.sum((residuals / sigma_approx) ** 2))
            dof = int(max(len(residuals) - 4, 1))
            chi2_red = chi2_val / dof
            p_val = float(chi2.sf(chi2_val, dof))

            ax2.text(
                0.02, 0.95,
                rf"$\chi^2_\nu$={chi2_red:.2f}\n$p$={p_val:.2g}\nDoF={dof}",
                transform=ax2.transAxes,
                ha='left', va='top',
                fontsize=FS_TINY,
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1)
            )
        else:
            ax2.axhline(0, color='black', linewidth=1)
            ax2.text(0.5, 0.5, "2-segment fit not available", transform=ax2.transAxes,
                     ha='center', va='center')
            ax2.set_ylabel("Residuals")
            ax2.grid(True, alpha=0.3)

        # Third panel: discrete 2nd derivative
        ax3.axhline(0, color='black', linewidth=1)
        if grad2 is not None:
            ax3.plot(t_vals_plot, grad2, marker='o', markersize=3,
                     linestyle='-', linewidth=1.5, color='magenta',
                     label="Discrete 2nd Deriv")
        else:
            ax3.text(0.5, 0.5, "2nd derivative not available", transform=ax3.transAxes,
                     ha='center', va='center')
        ax3.set_ylabel(r"$d^2f/dT^2$")
        ax3.grid(True, alpha=0.3)

        # Threshold guide lines (DASHED) on ALL panels
        ax1.axvline(THRESH_KMS, color='red', linestyle='--', linewidth=1.5, label=f'Hard ({THRESH_KMS} km/s)')
        ax2.axvline(THRESH_KMS, color='red', linestyle='--', linewidth=1.2, alpha=0.7)
        ax3.axvline(THRESH_KMS, color='red', linestyle='--', linewidth=1.2, alpha=0.7)

        if use_equiv and np.isfinite(current_thresh) and current_thresh != THRESH_KMS:
            ax1.axvline(current_thresh, color='green', linestyle='--', linewidth=1.5,
                        label=f'Equiv ({current_thresh:.1f})')
            ax2.axvline(current_thresh, color='green', linestyle='--', linewidth=1.2, alpha=0.7)
            ax3.axvline(current_thresh, color='green', linestyle='--', linewidth=1.2, alpha=0.7)

        if np.isfinite(t_d2):
            ax1.axvline(t_d2, color='magenta', linestyle='--', linewidth=1.5,
                        label=f"Max +d^2 ({t_d2:.1f})")
            ax2.axvline(t_d2, color='magenta', linestyle='--', linewidth=1.2, alpha=0.7)
            ax3.axvline(t_d2, color='magenta', linestyle='--', linewidth=1.2, alpha=0.7)

        step_size = max(10, int(MAX_RV_THRESH / 10))
        ax3.set_xticks(np.arange(0, MAX_RV_THRESH + 1, step_size))

        ax1.set_yticks(np.arange(0, 1.1, 0.1))
        ax1.set_ylabel(r"$f_{\mathrm{bin}}$")
        ax1.set_title(f"Binary Fraction vs Threshold: {line_key_bar}")
        ax1.grid(True, alpha=0.3, which='both')
        ax1.set_ylim(0, 1.05)
        ax1.legend(loc='upper right', fontsize=FS_TINY)

        ax3.set_xlabel("Threshold [km/s]")
        ax3.legend(loc='lower right', fontsize=FS_TINY)

        plt.tight_layout()
        plt.show()

    # --- PLOT 3: Equivalent Thresholds across lines ---
    lines_plot = []
    thresholds_plot = []
    for lk in ordered_lines:
        t = equiv_thresholds_map.get(lk, np.nan)
        lines_plot.append(lk)
        thresholds_plot.append(t)

    if len(lines_plot) > 0:
        plt.figure(figsize=FIGSIZE_THRESH_PLOT)
        plt.plot(lines_plot, thresholds_plot, marker='o', linestyle='-', color='purple')
        plt.xticks(rotation=45, ha='right')
        plt.ylabel("Equivalent Threshold [km/s]")
        plt.title("Thresholds yielding $f_{\mathrm{bin}}$ ≈ $f_{\mathrm{bin}}$(CIV, 20 km/s)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    # --- PLOT 4: Corner plot (unchanged) ---
    show_names = state.get("show_corner_names", False)
    use_log = state.get("corner_log_scale", True)
    n_lines = len(ordered_lines)
    line_ranges = {}
    for lk in ordered_lines:
        col = f"dRV | {lk}"
        vals_rng = []
        if use_equiv:
            t_val = equiv_thresholds_map.get(lk, THRESH_KMS)
        else:
            t_val = THRESH_KMS
        if np.isfinite(t_val):
            vals_rng.append(t_val)
        if col in masked.columns:
            for i in range(len(masked)):
                sname = masked.at[i, "Star"]
                _, _, frac = ew_fail_stats.get((sname, lk), (0, 0, 0))
                if frac > FAIL_FRAC_THRESH:
                    continue
                v = masked.at[i, col]
                if pd.notna(v) and np.isfinite(float(v)):
                    vals_rng.append(float(v))
        if not vals_rng:
            line_ranges[lk] = (0.1, MAX_RV_THRESH)
            continue

        vmin, vmax = min(vals_rng), max(vals_rng)
        if use_log:
            pos_vals = [x for x in vals_rng if x > 0]
            if pos_vals:
                vmin = min(pos_vals)
                log_min, log_max = np.log10(vmin), np.log10(vmax)
                span = log_max - log_min if log_max > log_min else 0.5
                vmin = 10**(log_min - 0.05 * span)
                vmax = 10**(log_max + 0.05 * span)
            else:
                vmin, vmax = 0.1, MAX_RV_THRESH
        else:
            span = vmax - vmin if vmax > vmin else 5.0
            vmin = vmin - 0.05 * span
            vmax = vmax + 0.05 * span
            if vmin < 0:
                vmin = 0
        line_ranges[lk] = (vmin, vmax)

    if n_lines > 1:
        dim = max(16, 3 * n_lines)
        fig, axes = plt.subplots(n_lines, n_lines, figsize=(dim, dim), sharex='col')
        plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, wspace=0, hspace=0)
        if n_lines == 1:
            axes = np.array([[axes]])

        trans_diagonal = mtransforms.Affine2D().rotate_deg(-45)

        for i in range(n_lines):
            line_y_key = ordered_lines[i]
            y_min, y_max = line_ranges.get(line_y_key, (0.1, MAX_RV_THRESH))
            thresh_y = equiv_thresholds_map.get(line_y_key, THRESH_KMS) if use_equiv else THRESH_KMS

            for j in range(n_lines):
                ax = axes[i, j]
                line_x_key = ordered_lines[j]
                x_min, x_max = line_ranges.get(line_x_key, (0.1, MAX_RV_THRESH))
                thresh_x = equiv_thresholds_map.get(line_x_key, THRESH_KMS) if use_equiv else THRESH_KMS

                if i == n_lines - 1:
                    ax.set_xlabel(format_line_label(line_x_key), fontsize=FS_AXIS_LBL)
                if j == 0 and i != j:
                    ax.set_ylabel(format_line_label(line_y_key), fontsize=FS_AXIS_LBL)

                if use_log:
                    ax.set_xscale('log')
                    if i != j:
                        ax.set_yscale('log')

                ax.set_xlim(x_min, x_max)
                if i != j:
                    ax.set_ylim(y_min, y_max)

                if i < n_lines - 1:
                    ax.tick_params(axis='x', labelbottom=False, length=0)
                if j > 0:
                    ax.tick_params(axis='y', labelleft=False, length=0)

                if j > i:
                    ax.axis('off')
                    continue

                if i == j:
                    col_name = f"dRV | {line_y_key}"
                    if col_name in masked.columns:
                        vals_hist = []
                        for row_idx in range(len(masked)):
                            sname = masked.at[row_idx, "Star"]
                            _, _, frac = ew_fail_stats.get((sname, line_y_key), (0, 0, 0))
                            if frac > FAIL_FRAC_THRESH:
                                continue
                            v = masked.at[row_idx, col_name]
                            if pd.notna(v) and np.isfinite(float(v)):
                                vals_hist.append(float(v))
                        if len(vals_hist) > 0:
                            if use_log:
                                hv_min = max(x_min, min([v for v in vals_hist if v > 0]) if any(v > 0 for v in vals_hist) else x_min)
                                hv_max = max(x_max, max(vals_hist))
                                bins = np.geomspace(hv_min, hv_max, 15)
                            else:
                                bins = 15
                            ax.hist(vals_hist, bins=bins, color='gray', alpha=0.7,
                                    histtype='stepfilled', edgecolor='black')
                    ax.yaxis.set_visible(False)
                    continue

                x_vals, y_vals, x_errs, y_errs = [], [], [], []
                c_topleft, c_botright = [], []
                names_to_plot = []
                col_x = f"dRV | {line_x_key}"
                col_y = f"dRV | {line_y_key}"
                if (col_x in masked.columns) and (col_y in masked.columns):
                    for row_idx in range(len(masked)):
                        sname = masked.at[row_idx, "Star"]
                        _, _, frac_x = ew_fail_stats.get((sname, line_x_key), (0, 0, 0))
                        _, _, frac_y = ew_fail_stats.get((sname, line_y_key), (0, 0, 0))
                        if (frac_x > FAIL_FRAC_THRESH) or (frac_y > FAIL_FRAC_THRESH):
                            continue
                        vx = masked.at[row_idx, col_x]
                        vy = masked.at[row_idx, col_y]
                        if pd.notna(vx) and np.isfinite(float(vx)) and pd.notna(vy) and np.isfinite(float(vy)):
                            vx = float(vx)
                            vy = float(vy)
                            if use_log and (vx <= 0 or vy <= 0):
                                continue
                            x_vals.append(vx)
                            y_vals.append(vy)
                            err_x = drverr_map.get((sname, line_x_key), 0.0)
                            err_y = drverr_map.get((sname, line_y_key), 0.0)
                            x_errs.append(err_x if np.isfinite(err_x) else 0.0)
                            y_errs.append(err_y if np.isfinite(err_y) else 0.0)
                            pass_y = _is_significant_binary(sname, line_y_key, vy, thresh_y)
                            c_topleft.append('green' if pass_y else 'red')
                            pass_x = _is_significant_binary(sname, line_x_key, vx, thresh_x)
                            c_botright.append('green' if pass_x else 'red')
                            names_to_plot.append(sname)

                if len(x_vals) > 0:
                    x_arr = np.array(x_vals)
                    y_arr = np.array(y_vals)
                    x_err_arr = np.array(x_errs)
                    y_err_arr = np.array(y_errs)

                    ax.axvline(thresh_x, color='black', linestyle=':', alpha=0.6, linewidth=1)
                    ax.axhline(thresh_y, color='black', linestyle=':', alpha=0.6, linewidth=1)
                    diag_min = max(x_min, y_min)
                    diag_max = min(x_max, y_max)
                    ax.plot([diag_min, diag_max], [diag_min, diag_max], ls='--',
                            color='gray', alpha=0.5, linewidth=1, zorder=0)

                    fbin_y_str = _get_fbin_simple(masked, line_y_key, use_equiv)
                    ax.text(0.05, 0.95, r"$f_{\mathrm{bin}}$(Y): " + fbin_y_str,
                            transform=ax.transAxes, fontsize=FS_NOTE, va='top', ha='left',
                            bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

                    fbin_x_str = _get_fbin_simple(masked, line_x_key, use_equiv)
                    ax.text(0.95, 0.05, r"$f_{\mathrm{bin}}$(X): " + fbin_x_str,
                            transform=ax.transAxes, fontsize=FS_NOTE, va='bottom', ha='right',
                            bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

                    ax.hlines(y_arr, x_arr - x_err_arr, x_arr + x_err_arr, colors=c_botright, lw=0.8, zorder=1)
                    ax.vlines(x_arr, y_arr - y_err_arr, y_arr + y_err_arr, colors=c_topleft, lw=0.8, zorder=1)

                    ax.scatter(x_vals, y_vals, c=c_botright,
                               marker=MarkerStyle('o', fillstyle='right', transform=trans_diagonal),
                               s=80, edgecolors='black', linewidth=0.5, zorder=2)
                    ax.scatter(x_vals, y_vals, c=c_topleft,
                               marker=MarkerStyle('o', fillstyle='left', transform=trans_diagonal),
                               s=80, edgecolors='black', linewidth=0.5, zorder=3)

                    if len(x_vals) > 1:
                        r_val, p_val = pearsonr(x_vals, y_vals)
                        ax.text(0.05, 0.85, f"r={r_val:.2f}\np={p_val:.2g}\nn={len(x_vals)}",
                                transform=ax.transAxes, fontsize=FS_NOTE, va='top', ha='left',
                                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8))

                    if show_names:
                        for k, nm in enumerate(names_to_plot):
                            ax.text(x_vals[k], y_vals[k], nm, fontsize=FS_TINY, alpha=0.8,
                                    clip_on=True, rotation=25)

                ax.grid(True, alpha=0.2, which='both')
        plt.show()

    # --- PLOT 5 + 6 (Agreement ranking + fbin vs emission line) ---
    drv_cols = [f"dRV | {lk}" for lk in ordered_lines if f"dRV | {lk}" in masked.columns]
    if len(drv_cols) > 1:
        cols_names = [c for c in drv_cols]
        adj_corr_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
        adj_pval_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
        adj_n_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)

        for i, c1 in enumerate(cols_names):
            data1 = masked[c1]
            for j, c2 in enumerate(cols_names):
                if i == j:
                    continue
                data2 = masked[c2]
                valid = data1.notna() & data2.notna()
                n_dots = valid.sum()
                if n_dots < MIN_STARS_FOR_CORR:
                    continue
                r, p = pearsonr(data1[valid], data2[valid])
                adj_corr_df.at[c1, c2] = r
                adj_pval_df.at[c1, c2] = p
                adj_n_df.at[c1, c2] = n_dots

        w_df = adj_n_df / MAX_POSSIBLE_STARS
        r_weighted_terms = adj_corr_df * w_df
        avg_scores_user = r_weighted_terms.sum(axis=1) / r_weighted_terms.count(axis=1)

        p_times_n = adj_pval_df * adj_n_df
        sum_n = adj_n_df.sum(axis=1)
        avg_pvals_std_weighted = p_times_n.sum(axis=1) / sum_n.replace(0, np.nan)

        best_friends, best_scores = [], []
        second_best_friends, second_best_scores = [], []
        stats_texts = []

        for idx_row, col in enumerate(cols_names):
            line_nm = col.replace("dRV | ", "")
            row_series = adj_corr_df.loc[col].sort_values(ascending=False)
            valid_friends = row_series.dropna()

            def _fmt_friend(friend_col, r_val):
                v = (masked[col].notna() & masked[friend_col].notna())
                n_count = v.sum()
                if n_count < 2:
                    return "Err", np.nan
                _, p_val = pearsonr(masked[col][v], masked[friend_col][v])
                nm = friend_col.replace("dRV | ", "")
                return f"{nm}\nr={r_val:.2f} (p={p_val:.2g}, n={n_count})", r_val

            if len(valid_friends) > 0:
                b_lbl, b_val = _fmt_friend(valid_friends.index[0], valid_friends.iloc[0])
            else:
                b_val, b_lbl = np.nan, "None"

            if len(valid_friends) > 1:
                s_lbl, s_val = _fmt_friend(valid_friends.index[1], valid_friends.iloc[1])
            else:
                s_val, s_lbl = np.nan, ""

            best_scores.append(b_val)
            best_friends.append(b_lbl)
            second_best_scores.append(s_val)
            second_best_friends.append(s_lbl)

            fbin_txt = _get_fbin_simple(masked, line_nm, use_equiv)
            ew_succ, ew_mn, ew_sem = _calc_ew_stats_for_line(line_nm)
            stats_texts.append(r"$f_{\mathrm{bin}}$: " + f"{fbin_txt}\nEW%: {ew_succ:.0%}\nEW: {ew_mn:.2f}±{ew_sem:.2f}")

        plot_data = pd.DataFrame({
            "Line": [c.replace("dRV | ", "") for c in cols_names],
            "Score": avg_scores_user.values,
            "Avg_P_Weighted": avg_pvals_std_weighted.values,
            "Best_Score": best_scores,
            "Best_Friend": best_friends,
            "Second_Score": second_best_scores,
            "Second_Friend": second_best_friends,
            "Stats": stats_texts
        }).sort_values("Score", ascending=False, na_position='last').reset_index(drop=True)

        plt.figure(figsize=FIGSIZE_AGREEMENT)
        valid_plot = plot_data.dropna(subset=["Score"])
        if not valid_plot.empty:
            plt.plot(valid_plot.index, valid_plot["Score"], marker='o', linestyle='-',
                     color='teal', linewidth=2, markersize=8,
                     label=f"Avg Score (Weight = N/{MAX_POSSIBLE_STARS})")

        show_extra = state.get("show_extra_data", False)
        if show_extra:
            plt.scatter(plot_data.index, plot_data["Best_Score"], color='orange', s=80,
                        marker='D', zorder=3, label="Max Correlation")
            plt.scatter(plot_data.index, plot_data["Second_Score"], color='lightgreen', s=70,
                        marker='s', edgecolors='green', zorder=2, label="2nd Best Correlation")

        plt.xticks(plot_data.index, plot_data["Line"], rotation=45, ha="right")
        plt.ylabel("Agreement Score")
        plt.title(f"Agreement Ranking (Logic: r * n/25, N >= {MIN_STARS_FOR_CORR})")
        plt.grid(True, alpha=0.3, linestyle='--')

        for i in plot_data.index:
            score = plot_data.at[i, "Score"]
            avg_p_w = plot_data.at[i, "Avg_P_Weighted"]
            best = plot_data.at[i, "Best_Score"]
            friend = plot_data.at[i, "Best_Friend"]
            second = plot_data.at[i, "Second_Score"]
            sec_friend = plot_data.at[i, "Second_Friend"]
            stat_txt = plot_data.at[i, "Stats"]

            if pd.notna(score):
                p_text = ""
                if show_extra and pd.notna(avg_p_w):
                    p_text = f"\n(p_w={avg_p_w:.2g})"
                plt.text(i, score - 0.02, f"{score:.3f}{p_text}",
                         ha='center', va='top', fontsize=FS_NOTE,
                         color='teal', fontweight='bold')

            if show_extra:
                if pd.notna(best):
                    plt.text(i, best + 0.01, friend, ha='center', va='bottom',
                             fontsize=FS_TINY, color='darkorange', rotation=25)
                if pd.notna(second):
                    plt.text(i, second - 0.01, sec_friend, ha='center', va='top',
                             fontsize=FS_TINY, color='green', rotation=25)

                ylim_min, ylim_max = plt.ylim()
                y_txt_pos = ylim_min + (ylim_max - ylim_min) * 0.05
                plt.text(i, y_txt_pos, stat_txt, ha='center', va='bottom',
                         fontsize=FS_BOX_STAT,
                         bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="gray", alpha=0.7))

        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.show()

        sort_by_agreement = state.get("sort_fbin_by_agreement", False)
        if sort_by_agreement and 'plot_data' in locals() and not plot_data.empty:
            sorted_lines_g6 = plot_data["Line"].tolist()
        else:
            sorted_lines_g6 = sorted(ordered_lines, key=_get_wavelength)

        # === UPDATED PLOT 6: Binary fraction per emission line ===
        show_hard_and_fit = state.get("graph6_show_hard_and_fit", False)
        filter_changes = state.get("filter_threshold_plot", True)

        x_pos = np.arange(len(sorted_lines_g6))
        plt.figure(figsize=FIGSIZE_FBIN_BAR)

        if show_hard_and_fit:
            # Hard series
            hard_vals, hard_el, hard_eh = [], [], []
            # Fit-threshold (2-line elbow) series
            fit_vals, fit_el, fit_eh = [], [], []
            # Discrete 2nd-derivative max-positive threshold series
            d2_vals, d2_el, d2_eh = [], [], []

            for lk in sorted_lines_g6:
                # --- Hard threshold ---
                f_h, el_h, eh_h, _, _ = _calc_fbin_stats_at_threshold(masked, lk, THRESH_KMS)
                hard_vals.append(f_h * 100.0)
                hard_el.append(el_h * 100.0)
                hard_eh.append(eh_h * 100.0)

                # --- Fit (2-seg elbow) threshold ---
                t_fit = get_fit_elbow_threshold_for_line(masked, lk, filter_changes=filter_changes)
                if np.isfinite(t_fit):
                    f_f, el_f, eh_f, _, _ = _calc_fbin_stats_at_threshold(masked, lk, t_fit)
                    fit_vals.append(f_f * 100.0)
                    fit_el.append(el_f * 100.0)
                    fit_eh.append(eh_f * 100.0)
                else:
                    fit_vals.append(np.nan)
                    fit_el.append(0.0)
                    fit_eh.append(0.0)

                # --- Max positive discrete 2nd-derivative threshold ---
                t_d2 = get_discrete_d2_threshold_for_line(masked, lk, filter_changes=filter_changes)
                if np.isfinite(t_d2):
                    f_d, el_d, eh_d, _, _ = _calc_fbin_stats_at_threshold(masked, lk, t_d2)
                    d2_vals.append(f_d * 100.0)
                    d2_el.append(el_d * 100.0)
                    d2_eh.append(eh_d * 100.0)
                else:
                    d2_vals.append(np.nan)
                    d2_el.append(0.0)
                    d2_eh.append(0.0)

            offset = 0.25
            plt.errorbar(x_pos - offset, hard_vals, yerr=[hard_el, hard_eh],
                         fmt='o', color='cornflowerblue', ecolor='gray',
                         capsize=5, elinewidth=2, markersize=7,
                         label=f"Hard ({THRESH_KMS:.0f} km/s)")

            plt.errorbar(x_pos, fit_vals, yerr=[fit_el, fit_eh],
                         fmt='o', color='darkorange', ecolor='gray',
                         capsize=5, elinewidth=2, markersize=7,
                         label="Fit threshold (2-line elbow)")

            plt.errorbar(x_pos + offset, d2_vals, yerr=[d2_el, d2_eh],
                         fmt='o', color='magenta', ecolor='gray',
                         capsize=5, elinewidth=2, markersize=7,
                         label="Max +d^2 threshold (discrete)")

            # Value labels (can get busy; keep compact)
            for i, (hv, fv, dv) in enumerate(zip(hard_vals, fit_vals, d2_vals)):
                if np.isfinite(hv):
                    plt.text(i - offset, hv + 2, f"{hv:.0f}%", ha='center', va='bottom',
                             fontsize=max(FS_BAR_VAL - 2, 8), fontweight='bold', color='black')
                if np.isfinite(fv):
                    plt.text(i, fv + 2, f"{fv:.0f}%", ha='center', va='bottom',
                             fontsize=max(FS_BAR_VAL - 2, 8), fontweight='bold', color='black')
                if np.isfinite(dv):
                    plt.text(i + offset, dv + 2, f"{dv:.0f}%", ha='center', va='bottom',
                             fontsize=max(FS_BAR_VAL - 2, 8), fontweight='bold', color='black')

            plt.title("Binary Fraction per Emission Line (Hard vs Fit vs Max +d^2)")
            plt.legend(loc='upper right')

        else:
            # Original single series (hard or equiv depending on toggle)
            fbin_vals, err_lows, err_highs = [], [], []
            for lk in sorted_lines_g6:
                f, el, eh, _, _ = _calc_fbin_stats_internal(masked, lk, use_equiv)
                fbin_vals.append(f * 100.0)
                err_lows.append(el * 100.0)
                err_highs.append(eh * 100.0)

            plt.errorbar(x_pos, fbin_vals, yerr=[err_lows, err_highs],
                         fmt='o', color='cornflowerblue', ecolor='gray',
                         capsize=5, elinewidth=2, markersize=8)

            for i, v in enumerate(fbin_vals):
                plt.text(i, v + 2, f"{v:.0f}%", ha='center', va='bottom',
                         fontsize=FS_BAR_VAL, fontweight='bold', color='black')

            plt.title(
                f"Binary Fraction per Emission Line ({'Equiv Thresholds' if use_equiv else 'Hard Threshold'})"
            )

        plt.xticks(x_pos, sorted_lines_g6, rotation=45, ha='right')
        plt.ylabel("Binary Fraction (%)")
        plt.ylim(0, 105)
        plt.grid(axis='y', alpha=0.3, linestyle='--')
        plt.tight_layout()
        plt.show()

    # --- PLOT 7: Confidence Grading (Golden/Silver/Bronze) ---
    star_results = get_star_classification_data(df, use_equiv)
    highlight_contam = state.get("highlight_contam", False)

    grades_data = {
        'Golden': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []},
        'Silver': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []},
        'Bronze': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []}
    }

    for sname, info in star_results.items():
        grade = info['grade']
        is_bin = info['is_binary']
        is_clean = info['is_clean']
        if is_bin:
            if is_clean:
                grades_data[grade]['bin_clean'].append(sname)
            else:
                grades_data[grade]['bin_contam'].append(sname)
        else:
            if is_clean:
                grades_data[grade]['sin_clean'].append(sname)
            else:
                grades_data[grade]['sin_contam'].append(sname)

    fig, axes = plt.subplots(1, 3, figsize=(14, 8))
    theme_map = {
        'Golden': (['#FFD700', '#FFFACD'], '#DAA520'),
        'Silver': (['#A9A9A9', '#DCDCDC'], 'gray'),
        'Bronze': (['#CD853F', '#FFE4C4'], '#8B4513')
    }

    grade_keys = ['Golden', 'Silver', 'Bronze']
    titles = ["Golden (>90% Agree)", "Silver (>70% Agree)", "Bronze (Rest)"]

    for idx, g in enumerate(grade_keys):
        ax = axes[idx]
        data = grades_data[g]
        n_bin_clean = len(data['bin_clean'])
        n_bin_contam = len(data['bin_contam'])
        n_sin_clean = len(data['sin_clean'])
        n_sin_contam = len(data['sin_contam'])
        total = n_bin_clean + n_bin_contam + n_sin_clean + n_sin_contam

        colors_theme, title_col = theme_map[g]
        c_bin_main, c_sin_main = colors_theme

        if total > 0:
            if highlight_contam:
                n_bin = n_bin_clean + n_bin_contam
                n_sin = n_sin_clean + n_sin_contam
                outer_sizes = [n_bin, n_sin]
                outer_labels = ["Binary", "Single"]
                outer_colors = [c_bin_main, c_sin_main]

                f_out_sizes, f_out_labels, f_out_colors = [], [], []
                for s, l, c in zip(outer_sizes, outer_labels, outer_colors):
                    if s > 0:
                        f_out_sizes.append(s)
                        f_out_labels.append(l)
                        f_out_colors.append(c)

                ax.pie(
                    f_out_sizes, labels=f_out_labels, colors=f_out_colors,
                    radius=1.0, wedgeprops=dict(width=0.3, edgecolor='w'),
                    startangle=90, textprops={'fontsize': FS_PIE_LABEL, 'weight': 'bold'},
                    autopct='%1.1f%%', pctdistance=0.85
                )

                inner_sizes = [n_bin_clean, n_bin_contam, n_sin_clean, n_sin_contam]
                inner_colors = [c_bin_main, '#FF4444', c_sin_main, '#FF4444']

                f_in_sizes, f_in_colors = [], []
                for s, c in zip(inner_sizes, inner_colors):
                    if s > 0:
                        f_in_sizes.append(s)
                        f_in_colors.append(c)

                if f_in_sizes:
                    ax.pie(
                        f_in_sizes, colors=f_in_colors, radius=0.7,
                        wedgeprops=dict(width=0.7, edgecolor='w'), startangle=90,
                        autopct='%1.1f%%', pctdistance=0.6,
                        textprops={'fontsize': FS_TINY}
                    )
            else:
                n_bin = n_bin_clean + n_bin_contam
                n_sin = n_sin_clean + n_sin_contam
                ax.pie([n_bin, n_sin],
                       labels=[f"Binary\n{n_bin}", f"Single\n{n_sin}"],
                       colors=colors_theme, autopct='%1.1f%%', startangle=90,
                       textprops={'fontsize': FS_PIE_LABEL})
        else:
            ax.text(0.5, 0.5, "No Stars", ha='center', va='center', fontsize=FS_PIE_LABEL)
            ax.axis('off')

        ax.set_title(titles[idx], color=title_col, fontsize=FS_PIE_LABEL + 2, fontweight='bold')

        def _get_wrapped(lst):
            if not lst:
                return ""
            return "\n".join(textwrap.wrap(", ".join(lst), width=20))

        bin_clean_txt = _get_wrapped(data['bin_clean'])
        bin_contam_txt = _get_wrapped(data['bin_contam'])
        sin_clean_txt = _get_wrapped(data['sin_clean'])
        sin_contam_txt = _get_wrapped(data['sin_contam'])

        ax.text(0.05, -0.05, f"$\\bf{{Binary:}}$", transform=ax.transAxes,
                ha='left', va='top', fontsize=FS_LIST_TEXT)
        current_y = -0.12
        if bin_clean_txt:
            ax.text(0.05, current_y, bin_clean_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')
            current_y -= (bin_clean_txt.count('\n') + 2) * 0.05
        if highlight_contam and bin_contam_txt:
            ax.text(0.05, current_y, bin_contam_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT,
                    color='red', fontweight='bold')
        elif (not highlight_contam) and bin_contam_txt:
            ax.text(0.05, current_y, bin_contam_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')

        ax.text(0.55, -0.05, f"$\\bf{{Single:}}$", transform=ax.transAxes,
                ha='left', va='top', fontsize=FS_LIST_TEXT)
        current_y = -0.12
        if sin_clean_txt:
            ax.text(0.55, current_y, sin_clean_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')
            current_y -= (sin_clean_txt.count('\n') + 2) * 0.05
        if highlight_contam and sin_contam_txt:
            ax.text(0.55, current_y, sin_contam_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT,
                    color='red', fontweight='bold')
        elif (not highlight_contam) and sin_contam_txt:
            ax.text(0.55, current_y, sin_contam_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT, color='black')

    plt.tight_layout()
    plt.show()

    # --- PLOT 8: All / Clean / Contaminated Comparison ---
    clean_counts = {'Binary': 0, 'Single': 0}
    contam_counts = {'Binary': 0, 'Single': 0}
    all_counts = {'Binary': 0, 'Single': 0}

    clean_lists = {'Binary': [], 'Single': []}
    contam_lists = {'Binary': [], 'Single': []}
    all_lists = {'Binary': [], 'Single': []}

    for sname, info in star_results.items():
        k = 'Binary' if info['is_binary'] else 'Single'
        all_counts[k] += 1
        all_lists[k].append(sname)
        if info['is_clean']:
            clean_counts[k] += 1
            clean_lists[k].append(sname)
        else:
            contam_counts[k] += 1
            contam_lists[k].append(sname)

    fig, axes = plt.subplots(1, 3, figsize=(16, 8))

    def plot_pie_with_lists(ax, counts, lists, title):
        n_bin = counts['Binary']
        n_sin = counts['Single']
        if (n_bin + n_sin) > 0:
            ax.pie([n_bin, n_sin],
                   labels=[f"Binary\n{n_bin}", f"Single\n{n_sin}"],
                   colors=['cornflowerblue', 'lightgray'],
                   autopct='%1.1f%%', startangle=90,
                   textprops={'fontsize': FS_PIE_LABEL})
        else:
            ax.text(0.5, 0.5, "No Data", ha='center', va='center', fontsize=FS_PIE_LABEL)
        ax.set_title(title, fontsize=FS_PIE_LABEL + 2, fontweight='bold')

        def _wrp(l):
            return "\n".join(textwrap.wrap(", ".join(l), width=20))

        bin_txt = _wrp(lists['Binary'])
        sin_txt = _wrp(lists['Single'])

        ax.text(0.05, -0.05, f"$\\bf{{Binary:}}$", transform=ax.transAxes,
                ha='left', va='top', fontsize=FS_LIST_TEXT)
        if bin_txt:
            ax.text(0.05, -0.12, bin_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT)

        ax.text(0.55, -0.05, f"$\\bf{{Single:}}$", transform=ax.transAxes,
                ha='left', va='top', fontsize=FS_LIST_TEXT)
        if sin_txt:
            ax.text(0.55, -0.12, sin_txt, transform=ax.transAxes,
                    ha='left', va='top', fontsize=FS_LIST_TEXT)

    plot_pie_with_lists(axes[0], all_counts, all_lists, "All Stars Together")
    plot_pie_with_lists(axes[1], clean_counts, clean_lists, "Clean Stars")
    plot_pie_with_lists(axes[2], contam_counts, contam_lists, "Contaminated Stars")

    plt.tight_layout()
    plt.show()

# ---------------- 7. WIDGETS & DISPLAY ----------------
default_sort = f"dRV | {CIV_KEY}" if f"dRV | {CIV_KEY}" in df.columns else (MEAN_COL if MEAN_COL in df.columns else "Star")

state = {
    "sort_col": default_sort,
    "ascending": False,
    "force_show_low_detection": False,
    "use_equiv_thresh": False,
    "show_corner_names": False,
    "corner_log_scale": True,
    "sort_fbin_by_agreement": False,
    "graph6_show_hard_and_fit": False,  # NEW tick
    "show_extra_data": False,
    "highlight_contam": False,
    "fit_elbow": True,
    "filter_threshold_plot": True
}

btn_for_col = {}

def update_buttons():
    for col, btn in btn_for_col.items():
        active = (col == state["sort_col"])
        arrow = "↑" if (active and state["ascending"]) else ("↓" if active else "")
        btn.description = f"{wrap_header(col)} {arrow}".rstrip()
        btn.button_style = "info" if active else ""

def on_click(col):
    def _handler(_):
        if col == state["sort_col"]:
            state["ascending"] = not state["ascending"]
        else:
            state["sort_col"] = col
            state["ascending"] = False
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
        update_buttons()
    return _handler

show_low_detect_chk = widgets.Checkbox(value=False, description="Show >40% EW failures", indent=False)
def on_toggle_low_detect(change):
    if change["name"] == "value":
        state["force_show_low_detection"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
show_low_detect_chk.observe(on_toggle_low_detect)

use_equiv_thresh_chk = widgets.Checkbox(value=False, description="Use Equivalent Thresholds", indent=False)
def on_toggle_equiv(change):
    if change["name"] == "value":
        state["use_equiv_thresh"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
use_equiv_thresh_chk.observe(on_toggle_equiv)

corner_names_chk = widgets.Checkbox(value=False, description="Corner Plot: Show Names", indent=False)
def on_toggle_corner_names(change):
    if change["name"] == "value":
        state["show_corner_names"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
corner_names_chk.observe(on_toggle_corner_names)

corner_log_chk = widgets.Checkbox(value=True, description="Log Scale (All Plots)", indent=False)
def on_toggle_corner_log(change):
    if change["name"] == "value":
        state["corner_log_scale"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
corner_log_chk.observe(on_toggle_corner_log)

sort_fbin_chk = widgets.Checkbox(value=False, description="Graph 6: Sort by Agreement Score", indent=False)
def on_toggle_fbin_sort(change):
    if change["name"] == "value":
        state["sort_fbin_by_agreement"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
sort_fbin_chk.observe(on_toggle_fbin_sort)

graph6_hard_fit_chk = widgets.Checkbox(value=False, description="Graph 6: Show Hard + Fit + d^2 Threshold", indent=False)
def on_toggle_graph6_hard_fit(change):
    if change["name"] == "value":
        state["graph6_show_hard_and_fit"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
graph6_hard_fit_chk.observe(on_toggle_graph6_hard_fit)

show_extra_data_chk = widgets.Checkbox(value=False, description="Graph 5: Show Extra Data", indent=False)
def on_toggle_extra(change):
    if change["name"] == "value":
        state["show_extra_data"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
show_extra_data_chk.observe(on_toggle_extra)

highlight_contam_chk = widgets.Checkbox(value=False, description="Graph 7: Highlight Contaminated Stars", indent=False)
def on_toggle_contam(change):
    if change["name"] == "value":
        state["highlight_contam"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
highlight_contam_chk.observe(on_toggle_contam)

fit_elbow_chk = widgets.Checkbox(value=True, description="Fit Elbow (2-Line)", indent=False)
def on_toggle_fit_elbow(change):
    if change["name"] == "value":
        state["fit_elbow"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
fit_elbow_chk.observe(on_toggle_fit_elbow)

filter_thresh_chk = widgets.Checkbox(value=True, description="Filter Thresh Plot (Changes Only)", indent=False)
def on_toggle_filter(change):
    if change["name"] == "value":
        state["filter_threshold_plot"] = bool(change["new"])
        with out:
            clear_output(wait=True)
            render_plots(state["sort_col"], state["ascending"])
filter_thresh_chk.observe(on_toggle_filter)

buttons = []
for col in ordered_cols:
    b = widgets.Button(description=wrap_header(col), tooltip=f"Sort by {col}",
                       layout=widgets.Layout(width="auto"))
    b.on_click(on_click(col))
    btn_for_col[col] = b
    buttons.append(b)

controls = widgets.HBox([
    show_low_detect_chk,
    use_equiv_thresh_chk,
    corner_names_chk,
    corner_log_chk,
    sort_fbin_chk,
    graph6_hard_fit_chk,
    show_extra_data_chk,
    highlight_contam_chk,
    fit_elbow_chk,
    filter_thresh_chk
], layout=widgets.Layout(gap="12px", flex_flow="row wrap"))

headers = widgets.HBox(buttons, layout=widgets.Layout(flex_flow="row wrap", gap="6px"))
out = widgets.Output()

display(controls, headers, out)
with out:
    clear_output(wait=True)
    render_plots(state["sort_col"], state["ascending"])
update_buttons()


## test of cell divions

In [ ]:
%matplotlib widget

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.markers import MarkerStyle
import textwrap
from IPython.display import display
from scipy.stats import pearsonr, chi2

# ============================================================
# 1) SETUP & CONFIG
# ============================================================
SETTINGS_PATH = "ccf_settings_with_global_lines.json"
try:
    with open(SETTINGS_PATH, "r") as f:
        SETTINGS = json.load(f)
except FileNotFoundError:
    print(f"WARNING: Could not find {SETTINGS_PATH}. Using empty settings.")
    SETTINGS = {}

STAR_CFG = {s["star_name"]: s for s in SETTINGS.get("stars", [])}
LINES_DEFAULT = SETTINGS.get("emission_lines_default", {})

# --- PRESENTATION MODE ---
for_presentation = True

# Config
CIV_KEY = "C IV 5808-5812"
THRESH_KMS = 10.0
MAX_RV_THRESH = 400
EXTRA_ABOVE = 3

BAR_FIGSIZE = (12, 6)
FIGSIZE_FRAC = (12, 10)  # 3 subplots now (main + residuals + 2nd deriv)
FIGSIZE_THRESH_PLOT = (10, 5)
FIGSIZE_AGREEMENT = (12, 7)
FIGSIZE_FBIN_BAR = (12, 6)

FAIL_FRAC_THRESH = 0.40
MIN_STARS_FOR_CORR = 8
MAX_POSSIBLE_STARS = 25
KNOWN_ERR_KEYS = ("full_RV_err", "full_err", "sigma", "err", "RV_err", "RV_sigma")

MEAN_COL = "Mean ΔRV"
STD_COL = "Std ΔRV"

# Globals for storage
epoch_ew_data = {}
ew_fail_stats = {}
rv_epoch_cache = {}
drverr_map = {}
equiv_thresholds_map = {}
fit_elbow_thresholds_cache = {}  # (line_key, filter_changes_bool) -> elbow_x
d2_thresholds_cache = {}         # (line_key, filter_changes_bool) -> t_maxpos_d2


# ============================================================
# 2) STYLE HELPER (used by plot cells)
# ============================================================
def set_plot_style(for_presentation: bool = True):
    """Sets matplotlib rcParams and returns a dict of font sizes used by plots."""
    if for_presentation:
        plt.rcParams.update({
            'font.size': 14,
            'axes.titlesize': 16,
            'axes.labelsize': 14,
            'xtick.labelsize': 12,
            'ytick.labelsize': 12,
            'legend.fontsize': 12
        })
        return dict(
            FS_NOTE=12,
            FS_TINY=11,
            FS_AXIS_LBL=14,
            FS_BAR_VAL=12,
            FS_BOX_STAT=8,
            FS_PIE_LABEL=14,
            FS_LIST_TEXT=11,
        )
    else:
        plt.rcParams.update({
            'font.size': 10,
            'axes.titlesize': 12,
            'axes.labelsize': 10,
            'xtick.labelsize': 10,
            'ytick.labelsize': 10,
            'legend.fontsize': 10
        })
        return dict(
            FS_NOTE=9,
            FS_TINY=8,
            FS_AXIS_LBL=10,
            FS_BAR_VAL=9,
            FS_BOX_STAT=7,
            FS_PIE_LABEL=11,
            FS_LIST_TEXT=9,
        )


# ============================================================
# 3) DATA PROCESSING HELPERS
# ============================================================
def _is_clean_star(star):
    try:
        for ep in star.get_all_epoch_numbers():
            flux = star.load_property('cleaned_normalized_flux', ep, 'COMBINED')
            if flux is not None:
                return False
        return True
    except Exception:
        return True


def _ew_record_for(star, epoch_num, line_key):
    try:
        EWs = star.load_property('EWs', epoch_num, 'COMBINED')
    except Exception:
        return None
    if EWs is None:
        return None
    try:
        rec = EWs.get(line_key)
    except Exception:
        return None
    if rec is None:
        return None
    try:
        rec_dict = rec.item()
    except Exception:
        return None
    if not isinstance(rec_dict, dict):
        return None

    def _to_float(x):
        try:
            return float(x)
        except Exception:
            return np.nan

    return {
        "EW": _to_float(rec_dict.get("EW")),
        "sigma_EW": _to_float(rec_dict.get("sigma_EW")),
        "SNR": _to_float(rec_dict.get("SNR")),
        "detected": rec_dict.get("detected")
    }


def _extract_full_rv(cell):
    try:
        if isinstance(cell, dict):
            return cell.get('full_RV', None)
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict):
                return v.get('full_RV', None)
        if hasattr(cell, "get"):
            return cell.get('full_RV', None)
    except Exception:
        pass
    return None


def _extract_full_rv_err(cell):
    try:
        if isinstance(cell, dict):
            for k in KNOWN_ERR_KEYS:
                if k in cell and cell[k] is not None:
                    return float(cell[k])
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict):
                for k in KNOWN_ERR_KEYS:
                    if k in v and v[k] is not None:
                        return float(v[k])
        if hasattr(cell, "get"):
            for k in KNOWN_ERR_KEYS:
                val = cell.get(k, None)
                if val is not None:
                    return float(val)
    except Exception:
        pass
    return np.nan


def _is_epoch_skipped_for_line(star_name, line_key, ep):
    cfg = STAR_CFG.get(star_name, {})
    if ep in set(cfg.get("skip_epochs", [])):
        return True
    sl = cfg.get("skip_emission_lines", {})
    if line_key in sl:
        skip_eps = sl[line_key]
        if isinstance(skip_eps, (int, np.integer)):
            skip_eps = [skip_eps]
        if 0 in skip_eps or ep in skip_eps:
            return True
        return False
    return False


def _line_delta_rv_for_star(star, line_key):
    rv_vals, cache = [], []
    sname = star.star_name

    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep):
            cache.append((ep, np.nan, np.nan, False))
            continue

        ewrec = _ew_record_for(star, ep, line_key)
        if ewrec is not None:
            epoch_ew_data[(sname, line_key, ep)] = ewrec

        RVs = star.load_property('RVs', ep, 'COMBINED')
        if not RVs or line_key not in RVs:
            cache.append((ep, np.nan, np.nan, False))
            continue

        cell = RVs[line_key]
        rv = _extract_full_rv(cell)

        if rv is None:
            cache.append((ep, np.nan, np.nan, False))
            continue

        try:
            rv = float(rv)
        except Exception:
            cache.append((ep, np.nan, np.nan, False))
            continue

        err = _extract_full_rv_err(cell)
        rv_vals.append((ep, rv, err))
        cache.append((ep, rv, err, True))

    rv_epoch_cache[(sname, line_key)] = cache

    if len(rv_vals) == 0:
        drverr_map[(sname, line_key)] = np.nan
        return np.nan

    ep_min, rv_min, err_min = min(rv_vals, key=lambda t: t[1])
    ep_max, rv_max, err_max = max(rv_vals, key=lambda t: t[1])
    dRV = abs(rv_max - rv_min)

    if np.isfinite(err_min) and np.isfinite(err_max):
        sigma_A = float(np.sqrt(err_min**2 + err_max**2))
    else:
        sigma_A = np.nan

    drverr_map[(sname, line_key)] = sigma_A
    return dRV


def _is_significant_binary(star_name: str, line_key: str, drv_val: float, threshold_val: float) -> bool:
    if not (pd.notna(drv_val) and np.isfinite(drv_val)):
        return False
    sigma_A = drverr_map.get((star_name, line_key), np.nan)
    if not np.isfinite(sigma_A):
        return False
    return (float(drv_val) >= threshold_val) and (float(drv_val) >= 4.0 * float(sigma_A))


def _compute_ew_fail_stats(star, line_key):
    sname = star.star_name
    considered = []
    failed = 0
    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep):
            continue
        considered.append(ep)
        rec = epoch_ew_data.get((sname, line_key, ep))
        if isinstance(rec, dict):
            det = rec.get("detected")
            if det is False:
                failed += 1
    n_considered = len(considered)
    frac = (failed / n_considered) if n_considered > 0 else 0.0
    return failed, n_considered, frac


# ============================================================
# 4) LOAD DATA & BUILD DATAFRAME
# ============================================================
rows = []
ordered_lines = list(LINES_DEFAULT.keys())

for star_name in specs.star_names:
    star = obs.load_star_instance(star_name, to_print=False)
    is_clean_bool = _is_clean_star(star)
    row = {"Star": star_name, "Clean": "✓" if is_clean_bool else "X", "is_clean_bool": is_clean_bool}

    for lk in ordered_lines:
        dval = _line_delta_rv_for_star(star, lk)
        ew_fail_stats[(star_name, lk)] = _compute_ew_fail_stats(star, lk)
        row[f"dRV | {lk}"] = dval

    drvs = [v for k, v in row.items() if isinstance(k, str) and k.startswith("dRV | ")]
    row[MEAN_COL] = float(np.nanmean(drvs)) if np.isfinite(np.nanmean(drvs)) else np.nan
    row[STD_COL] = float(np.nanstd(drvs)) if np.isfinite(np.nanstd(drvs)) else np.nan
    rows.append(row)

df = pd.DataFrame(rows)


# ============================================================
# 5) CALCULATE EQUIVALENT THRESHOLDS
# ============================================================
def calculate_equivalent_thresholds(df_in):
    if CIV_KEY not in ordered_lines:
        return
    ref_col = f"dRV | {CIV_KEY}"
    civ_vals = []

    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        _, _, frac = ew_fail_stats.get((sname, CIV_KEY), (0, 0, 0))
        if frac > FAIL_FRAC_THRESH:
            continue
        val = df_in.at[i, ref_col]
        if pd.notna(val) and np.isfinite(float(val)):
            civ_vals.append((sname, float(val)))

    n_above = 0
    for s, v in civ_vals:
        if _is_significant_binary(s, CIV_KEY, v, 20.0):
            n_above += 1

    num = n_above + EXTRA_ABOVE
    den = len(civ_vals) + EXTRA_ABOVE
    f_ref = num / den if den > 0 else 0

    equiv_thresholds_map.clear()
    search_space = np.arange(1, MAX_RV_THRESH + 1, 1)

    for lk in ordered_lines:
        col = f"dRV | {lk}"
        if col not in df_in.columns:
            continue
        line_vals = []
        for i in range(len(df_in)):
            sname = df_in.at[i, "Star"]
            _, _, frac = ew_fail_stats.get((sname, lk), (0, 0, 0))
            if frac > FAIL_FRAC_THRESH:
                continue
            val = df_in.at[i, col]
            if pd.notna(val) and np.isfinite(float(val)):
                line_vals.append((sname, float(val)))

        total_line = len(line_vals)
        if total_line == 0:
            equiv_thresholds_map[lk] = np.nan
            continue

        best_T = 20.0
        min_diff = 999.0

        for T in search_space:
            cnt = 0
            for s, v in line_vals:
                if _is_significant_binary(s, lk, v, T):
                    cnt += 1
            f_curr = (cnt + EXTRA_ABOVE) / (total_line + EXTRA_ABOVE)
            diff = abs(f_curr - f_ref)
            if diff < min_diff:
                min_diff = diff
                best_T = float(T)
        equiv_thresholds_map[lk] = best_T

calculate_equivalent_thresholds(df)

ordered_cols = ["Star", "Clean", MEAN_COL, STD_COL] + [
    f"dRV | {lk}" for lk in ordered_lines if f"dRV | {lk}" in df.columns
]

if f"dRV | {CIV_KEY}" in df.columns:
    df = df.sort_values(f"dRV | {CIV_KEY}", ascending=False, na_position="last").reset_index(drop=True)


# ============================================================
# 6) COMMON HELPERS USED BY PLOTS
# ============================================================
def wrap_str(s, width):
    if pd.isna(s):
        return s
    return "\n".join(textwrap.wrap(str(s), width=width, break_long_words=True)) or str(s)

def format_line_label(line_key, width=18, max_lines=2):
    """Corner-plot axis label helper: single line if possible, else wrap to <=2 lines."""
    s = str(line_key)
    if len(s) <= width:
        return s
    parts = textwrap.wrap(s, width=width, break_long_words=False, break_on_hyphens=False)
    if len(parts) <= max_lines:
        return "\n".join(parts)
    parts = parts[:max_lines]
    if not parts[-1].endswith("..."):
        if len(parts[-1]) > max(0, width - 3):
            parts[-1] = parts[-1][:max(0, width - 3)] + "..."
        else:
            parts[-1] = parts[-1] + "..."
    return "\n".join(parts)

def wilson_score_interval(k, n, z=1.0):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denominator = 1 + z**2/n
    center_adjusted_probability = (p + z**2 / (2*n)) / denominator
    error_margin = (z * np.sqrt((p*(1-p)/n) + (z**2/(4*n**2)))) / denominator
    low = center_adjusted_probability - error_margin
    high = center_adjusted_probability + error_margin
    return max(0.0, low), min(1.0, high)

# === Chi-Squared 2-Segment Linear Fit ===
def _fit_two_segment_linear_weighted(x, y, y_err):
    if len(x) < 5:
        return None
    w = 1.0 / (y_err**2)
    w[~np.isfinite(w)] = 0.0001
    best_chi2 = np.inf
    best_res = None
    for i in range(2, len(x) - 2):
        x1, y1, w1 = x[:i], y[:i], w[:i]
        x2, y2, w2 = x[i:], y[i:], w[i:]
        try:
            p1 = np.polyfit(x1, y1, 1, w=np.sqrt(w1))
            fit1 = np.polyval(p1, x1)
            chi2_1 = np.sum(w1 * (y1 - fit1)**2)
        except Exception:
            chi2_1 = np.inf
            p1 = [0, 0]
        try:
            p2 = np.polyfit(x2, y2, 1, w=np.sqrt(w2))
            fit2 = np.polyval(p2, x2)
            chi2_2 = np.sum(w2 * (y2 - fit2)**2)
        except Exception:
            chi2_2 = np.inf
            p2 = [0, 0]
        total_chi2 = chi2_1 + chi2_2
        if total_chi2 < best_chi2:
            best_chi2 = total_chi2
            m1, c1 = p1
            m2, c2 = p2
            if abs(m1 - m2) > 1e-9:
                elbow_x = (c2 - c1) / (m1 - m2)
                elbow_y = m1 * elbow_x + c1
            else:
                elbow_x = x[i]
                elbow_y = y[i]
            best_res = {
                'elbow_x': elbow_x, 'elbow_y': elbow_y,
                'p1': p1, 'p2': p2, 'break_idx': i,
                'range1': (x[0], elbow_x),
                'range2': (elbow_x, x[-1]),
                'chi2': best_chi2
            }
    return best_res

def get_fit_elbow_threshold_for_line(df_in, line_key, filter_changes=True):
    cache_key = (line_key, bool(filter_changes))
    if cache_key in fit_elbow_thresholds_cache:
        return fit_elbow_thresholds_cache[cache_key]

    col = f"dRV | {line_key}"
    if col not in df_in.columns:
        fit_elbow_thresholds_cache[cache_key] = np.nan
        return np.nan

    valid_indices = []
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin":
            continue
        _, _, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH:
            continue
        v = df_in.at[i, col]
        if pd.notna(v) and np.isfinite(float(v)):
            valid_indices.append(i)

    if len(valid_indices) < 5:
        fit_elbow_thresholds_cache[cache_key] = np.nan
        return np.nan

    t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
    f_full = np.zeros_like(t_vals_full, dtype=float)
    el_full = np.zeros_like(t_vals_full, dtype=float)
    eh_full = np.zeros_like(t_vals_full, dtype=float)

    for k, t in enumerate(t_vals_full):
        n_pass = 0
        for idx in valid_indices:
            sname = df_in.at[idx, "Star"]
            dval = float(df_in.at[idx, col])
            if _is_significant_binary(sname, line_key, dval, t):
                n_pass += 1
        num = n_pass + EXTRA_ABOVE
        den = len(valid_indices) + EXTRA_ABOVE
        f = (num / den) if den > 0 else 0.0
        l, h = wilson_score_interval(num, den, z=1.0)
        f_full[k] = f
        el_full[k] = f - l
        eh_full[k] = h - f

    if filter_changes:
        keep = [0]
        prev = f_full[0]
        for k in range(1, len(f_full)):
            if abs(f_full[k] - prev) > 1e-9:
                keep.append(k)
                prev = f_full[k]
        t_vals = t_vals_full[keep]
        f_arr = f_full[keep]
        el = el_full[keep]
        eh = eh_full[keep]
    else:
        t_vals, f_arr, el, eh = t_vals_full, f_full, el_full, eh_full

    sigma = (el + eh) / 2.0
    sigma[~np.isfinite(sigma)] = 0.01
    sigma[sigma <= 0] = 0.01

    if len(t_vals) < 5:
        fit_elbow_thresholds_cache[cache_key] = np.nan
        return np.nan

    fit_res = _fit_two_segment_linear_weighted(t_vals, f_arr, sigma)
    elbow = float(np.clip(fit_res["elbow_x"], 1.0, float(MAX_RV_THRESH))) if (fit_res and np.isfinite(fit_res.get("elbow_x", np.nan))) else np.nan
    fit_elbow_thresholds_cache[cache_key] = elbow
    return elbow

def get_discrete_d2_threshold_for_line(df_in, line_key, filter_changes=True):
    cache_key = (line_key, bool(filter_changes))
    if cache_key in d2_thresholds_cache:
        return d2_thresholds_cache[cache_key]

    col = f"dRV | {line_key}"
    if col not in df_in.columns:
        d2_thresholds_cache[cache_key] = np.nan
        return np.nan

    valid_indices = []
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin":
            continue
        _, _, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH:
            continue
        v = df_in.at[i, col]
        if pd.notna(v) and np.isfinite(float(v)):
            valid_indices.append(i)

    if len(valid_indices) < 3:
        d2_thresholds_cache[cache_key] = np.nan
        return np.nan

    t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
    f_full = np.zeros_like(t_vals_full, dtype=float)

    for k, t in enumerate(t_vals_full):
        n_pass = 0
        for idx in valid_indices:
            sname = df_in.at[idx, "Star"]
            dval = float(df_in.at[idx, col])
            if _is_significant_binary(sname, line_key, dval, t):
                n_pass += 1
        num = n_pass + EXTRA_ABOVE
        den = len(valid_indices) + EXTRA_ABOVE
        f_full[k] = (num / den) if den > 0 else 0.0

    if filter_changes:
        keep = [0]
        prev = f_full[0]
        for k in range(1, len(f_full)):
            if abs(f_full[k] - prev) > 1e-9:
                keep.append(k)
                prev = f_full[k]
        t_vals = t_vals_full[keep]
        f_arr = f_full[keep]
    else:
        t_vals, f_arr = t_vals_full, f_full

    if len(t_vals) < 3:
        d2_thresholds_cache[cache_key] = np.nan
        return np.nan

    grad1 = np.gradient(f_arr, t_vals)
    grad2 = np.gradient(grad1, t_vals)

    if len(grad2) > 2:
        inner = np.arange(1, len(grad2) - 1)
        g2_inner = grad2[inner]
        t_inner = t_vals[inner]
        pos = g2_inner > 0
        t_d2 = float(t_inner[pos][np.argmax(g2_inner[pos])]) if np.any(pos) else np.nan
    else:
        t_d2 = np.nan

    d2_thresholds_cache[cache_key] = t_d2
    return t_d2

def _calc_fbin_stats_at_threshold(base_with, line_key, thresh_to_use):
    colname = f"dRV | {line_key}"
    if colname not in base_with.columns:
        return 0.0, 0.0, 0.0, 0, 0
    if thresh_to_use is None or not np.isfinite(float(thresh_to_use)):
        return 0.0, 0.0, 0.0, 0, 0

    thresh_to_use = float(thresh_to_use)
    vals = base_with[colname]

    idx_with_drv = []
    for i in range(len(base_with)):
        star_name = base_with.at[i, "Star"]
        if not isinstance(star_name, str) or star_name == "f_bin":
            continue
        _, _, frac = ew_fail_stats.get((star_name, line_key), (0, 0, 0.0))
        if frac > FAIL_FRAC_THRESH:
            continue
        v = vals.iat[i]
        if pd.notna(v) and np.isfinite(float(v)):
            idx_with_drv.append(i)

    n_total = len(idx_with_drv)
    if n_total == 0:
        return 0.0, 0.0, 0.0, 0, 0

    n_above = 0
    for i in idx_with_drv:
        star_name = base_with.at[i, "Star"]
        dval = float(vals.iat[i])
        if _is_significant_binary(star_name, line_key, dval, thresh_to_use):
            n_above += 1

    n_total_adj = n_total + EXTRA_ABOVE
    n_above_adj = n_above + EXTRA_ABOVE
    frac_adj = n_above_adj / n_total_adj if n_total_adj > 0 else 0.0

    low, high = wilson_score_interval(n_above_adj, n_total_adj, z=1.0)
    err_low = frac_adj - low
    err_high = high - frac_adj

    return frac_adj, err_low, err_high, n_above_adj, n_total_adj

def _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh):
    thresh_to_use = THRESH_KMS
    if use_equiv_thresh:
        t = equiv_thresholds_map.get(line_key, np.nan)
        if np.isfinite(t):
            thresh_to_use = float(t)
    return _calc_fbin_stats_at_threshold(base_with, line_key, thresh_to_use)

def _get_fbin_simple(base_with, line_key, use_equiv_thresh):
    frac, el, eh, num, den = _calc_fbin_stats_internal(base_with, line_key, use_equiv_thresh)
    if den == 0:
        return "0/0 (—)"
    return f"{num}/{den} ({frac:.0%} +{eh:.0%}/-{el:.0%})"

def _calc_ew_stats_for_line(line_key):
    failed = 0
    considered = 0
    all_ews = []

    for sname in STAR_CFG.keys():
        n_fail, n_cons, _ = ew_fail_stats.get((sname, line_key), (0, 0, 0))
        failed += n_fail
        considered += n_cons

    for (sname, lk, ep), rec in epoch_ew_data.items():
        if lk == line_key and isinstance(rec, dict):
            val = rec.get("EW")
            if val is not None and np.isfinite(val):
                all_ews.append(val)

    success_rate = (1.0 - failed / considered) if considered > 0 else 0.0
    mean_ew = np.nanmean(all_ews) if len(all_ews) > 0 else np.nan
    sem_ew = (np.nanstd(all_ews) / np.sqrt(len(all_ews))) if len(all_ews) > 0 else np.nan

    return success_rate, mean_ew, sem_ew

def _get_wavelength(name):
    match = re.search(r"(\d+)", name)
    return float(match.group(1)) if match else 999999.0

def build_masked_df(df_in, sort_col=None, ascending=False, force_show_low_detection=False):
    """Apply EW-failure masking and sort. Returns (masked_df, sort_col_used)."""
    base = df_in.copy()[ordered_cols]
    masked = base.copy()

    for col in [c for c in masked.columns if c.startswith("dRV | ")]:
        line_key = col.replace("dRV | ", "")
        for i in range(len(masked)):
            sname = masked.at[i, "Star"]
            if not isinstance(sname, str) or sname == "f_bin":
                continue
            n_fail, n_cons, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
            if (n_cons > 0) and (frac > FAIL_FRAC_THRESH):
                if not force_show_low_detection:
                    masked.at[i, col] = np.nan

    if sort_col is None:
        sort_col_use = masked.columns[2] if len(masked.columns) > 2 else masked.columns[0]
    else:
        sort_col_use = sort_col if sort_col in masked.columns else (masked.columns[2] if len(masked.columns) > 2 else masked.columns[0])

    masked = masked.sort_values(sort_col_use, ascending=ascending, na_position="last").reset_index(drop=True)
    return masked, sort_col_use

def get_star_classification_data(df_in, use_equiv):
    results = {}
    for i in range(len(df_in)):
        sname = df_in.at[i, "Star"]
        if not isinstance(sname, str) or sname == "f_bin":
            continue
        is_clean = bool(df_in.at[i, "is_clean_bool"])
        votes_bin = 0
        votes_single = 0
        civ_status = None
        valid_lines = 0
        for lk in ordered_lines:
            _, _, frac = ew_fail_stats.get((sname, lk), (0, 0, 0.0))
            if frac > FAIL_FRAC_THRESH:
                continue
            col_name = f"dRV | {lk}"
            if col_name not in df_in.columns:
                continue
            val = df_in.at[i, col_name]
            if pd.isna(val) or not np.isfinite(float(val)):
                continue
            t_use = THRESH_KMS
            if use_equiv:
                t_eq = equiv_thresholds_map.get(lk, np.nan)
                if np.isfinite(t_eq):
                    t_use = t_eq
            is_bin = _is_significant_binary(sname, lk, float(val), t_use)
            votes_bin += int(is_bin)
            votes_single += int(not is_bin)
            if lk == CIV_KEY:
                civ_status = is_bin
            valid_lines += 1
        if valid_lines == 0:
            continue
        agreement = max(votes_bin, votes_single) / valid_lines
        grade = 'Golden' if agreement > 0.90 else ('Silver' if agreement > 0.70 else 'Bronze')
        if votes_bin > votes_single:
            final_bin = True
        elif votes_single > votes_bin:
            final_bin = False
        else:
            final_bin = civ_status if civ_status is not None else False
        results[sname] = {'grade': grade, 'is_binary': final_bin, 'is_clean': is_clean}
    return results

print(f"Built df: {len(df)} stars × {len(ordered_lines)} emission lines")


### Graph 1 - Peak-to-Peak ΔRV

In [ ]:
# Cell 2 — Plot 1: Bar/Point plot (|ΔRV| vs star)

# --- USER SETTINGS ---
sort_col = f"dRV | {CIV_KEY}"     # e.g. MEAN_COL, STD_COL, or f"dRV | {some_line}"
ascending = False
force_show_low_detection = False  # True keeps >40% EW-fail stars instead of masking them
use_equiv_thresh = False          # True uses equiv_thresholds_map[line] for threshold line
use_log_scale = True
for_presentation = True

S = set_plot_style(for_presentation)

masked, sort_col_use = build_masked_df(
    df,
    sort_col=sort_col,
    ascending=ascending,
    force_show_low_detection=force_show_low_detection
)

vals = masked[sort_col_use].to_numpy()
x = np.arange(len(masked))

line_key_bar = sort_col_use.replace("dRV | ", "") if sort_col_use.startswith("dRV | ") else None

current_thresh = THRESH_KMS
if use_equiv_thresh and line_key_bar:
    t = equiv_thresholds_map.get(line_key_bar, THRESH_KMS)
    if np.isfinite(t):
        current_thresh = float(t)

colors = []
yerrs = []
for i in range(len(masked)):
    star_name = masked.at[i, "Star"]
    val = vals[i]
    if pd.isna(val):
        colors.append("gray")
        yerrs.append(0.0)
    elif line_key_bar:
        is_sig = _is_significant_binary(star_name, line_key_bar, float(val), current_thresh)
        colors.append("green" if is_sig else "red")
        err = drverr_map.get((star_name, line_key_bar), 0.0)
        yerrs.append(err if np.isfinite(err) else 0.0)
    else:
        colors.append("green" if float(val) >= current_thresh else "red")
        yerrs.append(0.0)

plt.figure(figsize=BAR_FIGSIZE)
plt.plot(x, vals, color='black', alpha=0.2, zorder=1)

for i in range(len(x)):
    plt.errorbar(x[i], vals[i], yerr=yerrs[i], fmt='o',
                 color=colors[i], ecolor=colors[i], capsize=3, zorder=2)

plt.axhline(current_thresh, linestyle="--", linewidth=1, color='gray',
            label=f"Threshold {current_thresh:.1f} km/s")

plt.xticks(x, [wrap_str(s, 10) for s in masked["Star"]], rotation=45, ha="right")
plt.ylabel("|ΔRV| (km/s)")

if use_log_scale:
    plt.yscale('log')
    valid_pos = [v for v in vals if pd.notna(v) and v > 0]
    if valid_pos:
        plt.ylim(min(valid_pos) * 0.5, max(valid_pos) * 2.0)

plt.title(f"Peak-to-peak |ΔRV| for {sort_col_use.replace('dRV | ', '')} ({'asc' if ascending else 'desc'})")
plt.tight_layout()
plt.show()


### Graph 2 - Binary Fraction vs Threshold

In [ ]:
# Cell 3 — Plot 2: Binary Fraction vs Threshold (Interactive)
# [Updated: Added 'Show All Blue' Checkbox (Default=True)]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- USER SETTINGS ---
line_key = CIV_KEY
force_show_low_detection = False
filter_changes_only = True
fit_elbow = True
use_equiv_thresh = False

# --- PRESENTATION MODE ---
presentation = True 

# --- STYLE HELPERS ---
def get_style_dict(is_presentation):
    if is_presentation:
        plt.rcParams.update({
            'font.size': 14,
            'axes.titlesize': 18,
            'axes.labelsize': 16,
            'xtick.labelsize': 14,
            'ytick.labelsize': 14,
            'legend.fontsize': 12,
            'lines.markersize': 8,
            'lines.linewidth': 2.5
        })
        return dict(
            FS_NOTE=14,
            FS_TINY=12,
            FS_AXIS_LBL=16,
            FIG_SIZE_FIT=(12, 8),
            FIG_SIZE_NOFIT=(12, 6)
        )
    else:
        plt.rcParams.update({
            'font.size': 10,
            'axes.titlesize': 12,
            'axes.labelsize': 10,
            'xtick.labelsize': 10,
            'ytick.labelsize': 10,
            'legend.fontsize': 10,
            'lines.markersize': 6,
            'lines.linewidth': 1.5
        })
        return dict(
            FS_NOTE=10,
            FS_TINY=10,
            FS_AXIS_LBL=10,
            FIG_SIZE_FIT=(10, 7),
            FIG_SIZE_NOFIT=(10, 5)
        )

S = get_style_dict(presentation)

# ============================================================
# 1) STATIC DATA LOADING (Run once)
# ============================================================
sort_col = f"dRV | {line_key}"
masked, sort_col_use = build_masked_df(
    df, sort_col=sort_col, ascending=False,
    force_show_low_detection=force_show_low_detection
)

if not sort_col_use.startswith("dRV | "):
    raise ValueError("Plot 2 expects sort_col_use to be a dRV column like 'dRV | <line>'")

# Identify all valid indices initially
all_valid_indices = []
for i in range(len(masked)):
    sname = masked.at[i, "Star"]
    _, _, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
    if frac > FAIL_FRAC_THRESH:
        continue
    v = masked.at[i, sort_col_use]
    if pd.notna(v) and np.isfinite(float(v)):
        all_valid_indices.append(i)

# Map indices to cleanliness
clean_map = {}
for idx in all_valid_indices:
    sname = masked.at[idx, "Star"]
    try:
        star_obj = obs.load_star_instance(sname, to_print=False)
        is_clean = _is_clean_star(star_obj)
    except Exception:
        is_clean = True
    clean_map[idx] = is_clean

equiv_thresh = np.nan
if use_equiv_thresh:
    t_eq = equiv_thresholds_map.get(line_key, np.nan)
    if np.isfinite(t_eq):
        equiv_thresh = float(t_eq)

# ============================================================
# 2) DYNAMIC CALCULATION FUNCTION
# ============================================================
def recalc_data(exclude_dirty):
    """Re-calculates the binary fractions based on the filter."""
    
    # 1. Filter indices
    if exclude_dirty:
        current_indices = [i for i in all_valid_indices if clean_map[i]]
    else:
        current_indices = all_valid_indices

    if not current_indices:
        return None # No data

    # 2. Arrays to hold results
    t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
    fractions_full = np.zeros_like(t_vals_full, dtype=float)
    err_lows_full = np.zeros_like(t_vals_full, dtype=float)
    err_highs_full = np.zeros_like(t_vals_full, dtype=float)
    
    contaminated_change_flags = np.zeros_like(t_vals_full, dtype=bool)
    dropped_star_labels = np.empty(len(t_vals_full), dtype=object)

    previous_passing = set(current_indices)

    # 3. Loop over thresholds
    for k, t in enumerate(t_vals_full):
        current_passing = set()
        n_pass = 0
        for idx in current_indices:
            sname = masked.at[idx, "Star"]
            dval = float(masked.at[idx, sort_col_use])
            if _is_significant_binary(sname, line_key, dval, t):
                current_passing.add(idx)
                n_pass += 1
        
        # Detect drops
        dropped = previous_passing - current_passing
        if len(dropped) > 0:
            is_dirty_change = any(not clean_map[idx] for idx in dropped)
            contaminated_change_flags[k] = is_dirty_change
            names = [masked.at[idx, "Star"] for idx in dropped]
            dropped_star_labels[k] = ", ".join(names)
        else:
            contaminated_change_flags[k] = False
            dropped_star_labels[k] = ""
            
        previous_passing = current_passing

        # Wilson Score
        num = n_pass + EXTRA_ABOVE
        den = len(current_indices) + EXTRA_ABOVE
        f = (num / den) if den > 0 else 0.0
        l, h = wilson_score_interval(num, den, z=1.0)
        
        fractions_full[k] = f
        err_lows_full[k] = f - l
        err_highs_full[k] = h - f

    # 4. Filter for changes only (if enabled)
    if filter_changes_only and len(fractions_full) > 0:
        keep = [0]
        prev_f = fractions_full[0]
        for k in range(1, len(fractions_full)):
            if abs(fractions_full[k] - prev_f) > 1e-9:
                keep.append(k)
                prev_f = fractions_full[k]
        
        res = {
            't': t_vals_full[keep],
            'f': fractions_full[keep],
            'el': err_lows_full[keep],
            'eh': err_highs_full[keep],
            'contam': contaminated_change_flags[keep],
            'labels': dropped_star_labels[keep],
            'indices': current_indices
        }
    else:
        res = {
            't': t_vals_full,
            'f': fractions_full,
            'el': err_lows_full,
            'eh': err_highs_full,
            'contam': contaminated_change_flags,
            'labels': dropped_star_labels,
            'indices': current_indices
        }
        
    return res

# ============================================================
# 3) PLOTTING FUNCTION
# ============================================================
def plot_interactive(show_labels, show_fit, exclude_dirty, show_all_blue):
    
    # --- GET DATA ---
    data = recalc_data(exclude_dirty)
    if data is None:
        print("No stars remaining after filter.")
        return

    t_vals = data['t']
    f_arr = data['f']
    sigma = (data['el'] + data['eh']) / 2.0
    sigma[~np.isfinite(sigma)] = 0.01
    sigma[sigma <= 0] = 0.01

    # --- FIT ---
    has_valid_fit = False
    fit_res = None
    if show_fit and fit_elbow and len(f_arr) > 5:
        fit_res = _fit_two_segment_linear_weighted(t_vals, f_arr, sigma)
        if fit_res:
            has_valid_fit = True

    # --- SETUP CANVAS ---
    if has_valid_fit:
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=S['FIG_SIZE_FIT'], sharex=True,
            gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.08}
        )
    else:
        fig, ax1 = plt.subplots(1, 1, figsize=S['FIG_SIZE_NOFIT'])
        ax2 = None

    # --- PLOT MAIN DATA ---
    ms = 8 if presentation else 6

    if show_all_blue:
        # Plot ALL points as blue
        ax1.errorbar(
            t_vals, f_arr,
            yerr=[data['el'], data['eh']],
            fmt='o', markersize=ms, linestyle='none',
            color='blue', ecolor='blue', capsize=4,
            label="Binary Fraction"
        )
    else:
        # Plot split colors (Blue vs Red)
        mask_red = data['contam']
        mask_blue = ~data['contam']

        if np.any(mask_blue):
            ax1.errorbar(
                t_vals[mask_blue], f_arr[mask_blue],
                yerr=[data['el'][mask_blue], data['eh'][mask_blue]],
                fmt='o', markersize=ms, linestyle='none',
                color='blue', ecolor='blue', capsize=4,
                label="Clean Stars"
            )

        if np.any(mask_red):
            ax1.errorbar(
                t_vals[mask_red], f_arr[mask_red],
                yerr=[data['el'][mask_red], data['eh'][mask_red]],
                fmt='o', markersize=ms, linestyle='none',
                color='red', ecolor='red', capsize=4,
                label="Contaminated Stars"
            )

    # Labels
    if show_labels:
        for t_p, f_p, lbl in zip(t_vals, f_arr, data['labels']):
            if lbl and isinstance(lbl, str) and len(lbl) > 0:
                ax1.annotate(
                    lbl, xy=(t_p, f_p), xytext=(0, 10),
                    textcoords='offset points', ha='center', va='bottom',
                    fontsize=S['FS_TINY'], rotation=45, alpha=0.8
                )

    # --- PLOT FIT ---
    if has_valid_fit and ax2 is not None:
        lw = 2.5 if presentation else 2
        elbow_x = fit_res['elbow_x']
        
        # Real data fraction at elbow
        current_indices = data['indices']
        n_pass_real = 0
        for idx in current_indices:
            sname = masked.at[idx, "Star"]
            dval = float(masked.at[idx, sort_col_use])
            if _is_significant_binary(sname, line_key, dval, elbow_x):
                n_pass_real += 1
        
        num = n_pass_real + EXTRA_ABOVE
        den = len(current_indices) + EXTRA_ABOVE
        real_y = (num / den) if den > 0 else 0.0

        # Segments
        x1_r = np.linspace(fit_res['range1'][0], elbow_x, 50)
        ax1.plot(x1_r, np.polyval(fit_res['p1'], x1_r), color='orange', linewidth=lw, label="2-Seg Fit")
        x2_r = np.linspace(elbow_x, fit_res['range2'][1], 50)
        ax1.plot(x2_r, np.polyval(fit_res['p2'], x2_r), color='orange', linewidth=lw)
        
        # Lines
        ax1.axvline(elbow_x, color='orange', linestyle='--', linewidth=lw, 
                    label=f"Fit X ({elbow_x:.1f} km/s)")
        ax2.axvline(elbow_x, color='orange', linestyle='--', linewidth=lw*0.7, alpha=0.7)
        
        ax1.axhline(real_y, color='gray', linestyle='--', linewidth=lw, alpha=0.6,
                    label=f"Concluded Binary Fraction ($f={real_y:.2f}$)")

        # Residuals
        y_fit = np.where(
            t_vals <= elbow_x,
            np.polyval(fit_res['p1'], t_vals),
            np.polyval(fit_res['p2'], t_vals)
        )
        resid = f_arr - y_fit
        
        chi2_val = float(np.sum((resid / sigma) ** 2))
        dof = int(max(len(resid) - 4, 1))
        chi2_red = chi2_val / dof
        p_val = float(chi2.sf(chi2_val, dof))
        
        stats_text = (f"$\\chi^2_\\nu$={chi2_red:.2f}\n$p$={p_val:.2g}\nDoF={dof}")
        ax2.text(0.02, 0.95, stats_text, transform=ax2.transAxes, ha='left', va='top', 
                 fontsize=S["FS_NOTE"], bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
        
        ax2.axhline(0, color='black', linewidth=1)
        ax2.errorbar(t_vals, resid, yerr=sigma, fmt='o', markersize=ms-2, 
                     color='purple', ecolor='purple', capsize=4)
        ax2.set_ylabel("Residuals", fontsize=S['FS_AXIS_LBL'])
        ax2.grid(True, alpha=0.3)
        ax2.set_xticks(np.arange(0, MAX_RV_THRESH + 1, max(10, int(MAX_RV_THRESH / 10))))
        ax2.set_xlabel("Threshold [km/s]", fontsize=S['FS_AXIS_LBL'])
    else:
        ax1.set_xlabel("Threshold [km/s]", fontsize=S['FS_AXIS_LBL'])
        ax1.set_xticks(np.arange(0, MAX_RV_THRESH + 1, max(10, int(MAX_RV_THRESH / 10))))

    if np.isfinite(equiv_thresh):
        ax1.axvline(equiv_thresh, color='green', linestyle='--', linewidth=2, alpha=0.7, label=f'Equiv ({equiv_thresh:.1f})')
        if ax2 is not None:
            ax2.axvline(equiv_thresh, color='green', linestyle='--', linewidth=1.5, alpha=0.7)

    ax1.set_ylabel(r"$f_{\mathrm{bin}}$", fontsize=S['FS_AXIS_LBL'])
    ax1.set_title(f"Binary Fraction vs Threshold: {line_key}", fontsize=S['FS_AXIS_LBL']*1.1)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.15)
    
    # --- DRAGGABLE LEGEND ---
    handles, labels = ax1.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    leg = ax1.legend(by_label.values(), by_label.keys(), loc='best', fontsize=S["FS_NOTE"])
    leg.set_draggable(True)
    
    plt.tight_layout()
    plt.show()

# ============================================================
# 4) WIDGETS
# ============================================================
chk_labels = widgets.Checkbox(value=False, description='Show Star Names')
chk_fit = widgets.Checkbox(value=True, description='Show Linear Fit')
chk_clean = widgets.Checkbox(value=False, description='Exclude Contaminated')
chk_force_blue = widgets.Checkbox(value=True, description='Show All Blue')

ui = widgets.HBox([chk_labels, chk_fit, chk_clean, chk_force_blue])
out = widgets.interactive_output(plot_interactive, {
    'show_labels': chk_labels, 
    'show_fit': chk_fit, 
    'exclude_dirty': chk_clean,
    'show_all_blue': chk_force_blue
})

display(ui, out)

In [ ]:
# Cell 3 — Plot 2: Binary Fraction vs Threshold (Interactive)
# [Updated: Draggable Legends + Best Location + Real Data Horizontal Line]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- USER SETTINGS ---
line_key = CIV_KEY
force_show_low_detection = False
filter_changes_only = True
fit_elbow = True
use_equiv_thresh = False

# --- PRESENTATION MODE ---
presentation = True 

# --- STYLE HELPERS ---
def get_style_dict(is_presentation):
    if is_presentation:
        plt.rcParams.update({
            'font.size': 14,
            'axes.titlesize': 18,
            'axes.labelsize': 16,
            'xtick.labelsize': 14,
            'ytick.labelsize': 14,
            'legend.fontsize': 12,
            'lines.markersize': 8,
            'lines.linewidth': 2.5
        })
        return dict(
            FS_NOTE=14,
            FS_TINY=12,
            FS_AXIS_LBL=16,
            FIG_SIZE_FIT=(12, 8),
            FIG_SIZE_NOFIT=(12, 6)
        )
    else:
        plt.rcParams.update({
            'font.size': 10,
            'axes.titlesize': 12,
            'axes.labelsize': 10,
            'xtick.labelsize': 10,
            'ytick.labelsize': 10,
            'legend.fontsize': 10,
            'lines.markersize': 6,
            'lines.linewidth': 1.5
        })
        return dict(
            FS_NOTE=10,
            FS_TINY=10,
            FS_AXIS_LBL=10,
            FIG_SIZE_FIT=(10, 7),
            FIG_SIZE_NOFIT=(10, 5)
        )

S = get_style_dict(presentation)

# ============================================================
# 1) STATIC DATA LOADING (Run once)
# ============================================================
sort_col = f"dRV | {line_key}"
masked, sort_col_use = build_masked_df(
    df, sort_col=sort_col, ascending=False,
    force_show_low_detection=force_show_low_detection
)

if not sort_col_use.startswith("dRV | "):
    raise ValueError("Plot 2 expects sort_col_use to be a dRV column like 'dRV | <line>'")

# Identify all valid indices initially
all_valid_indices = []
for i in range(len(masked)):
    sname = masked.at[i, "Star"]
    _, _, frac = ew_fail_stats.get((sname, line_key), (0, 0, 0.0))
    if frac > FAIL_FRAC_THRESH:
        continue
    v = masked.at[i, sort_col_use]
    if pd.notna(v) and np.isfinite(float(v)):
        all_valid_indices.append(i)

# Map indices to cleanliness
clean_map = {}
for idx in all_valid_indices:
    sname = masked.at[idx, "Star"]
    try:
        star_obj = obs.load_star_instance(sname, to_print=False)
        is_clean = _is_clean_star(star_obj)
    except Exception:
        is_clean = True
    clean_map[idx] = is_clean

equiv_thresh = np.nan
if use_equiv_thresh:
    t_eq = equiv_thresholds_map.get(line_key, np.nan)
    if np.isfinite(t_eq):
        equiv_thresh = float(t_eq)

# ============================================================
# 2) DYNAMIC CALCULATION FUNCTION
# ============================================================
def recalc_data(exclude_dirty):
    """Re-calculates the binary fractions based on the filter."""
    
    # 1. Filter indices
    if exclude_dirty:
        current_indices = [i for i in all_valid_indices if clean_map[i]]
    else:
        current_indices = all_valid_indices

    if not current_indices:
        return None # No data

    # 2. Arrays to hold results
    t_vals_full = np.arange(1, MAX_RV_THRESH + 1, 1, dtype=float)
    fractions_full = np.zeros_like(t_vals_full, dtype=float)
    err_lows_full = np.zeros_like(t_vals_full, dtype=float)
    err_highs_full = np.zeros_like(t_vals_full, dtype=float)
    
    contaminated_change_flags = np.zeros_like(t_vals_full, dtype=bool)
    dropped_star_labels = np.empty(len(t_vals_full), dtype=object)

    previous_passing = set(current_indices)

    # 3. Loop over thresholds
    for k, t in enumerate(t_vals_full):
        current_passing = set()
        n_pass = 0
        for idx in current_indices:
            sname = masked.at[idx, "Star"]
            dval = float(masked.at[idx, sort_col_use])
            if _is_significant_binary(sname, line_key, dval, t):
                current_passing.add(idx)
                n_pass += 1
        
        # Detect drops
        dropped = previous_passing - current_passing
        if len(dropped) > 0:
            is_dirty_change = any(not clean_map[idx] for idx in dropped)
            contaminated_change_flags[k] = is_dirty_change
            names = [masked.at[idx, "Star"] for idx in dropped]
            dropped_star_labels[k] = ", ".join(names)
        else:
            contaminated_change_flags[k] = False
            dropped_star_labels[k] = ""
            
        previous_passing = current_passing

        # Wilson Score
        num = n_pass + EXTRA_ABOVE
        den = len(current_indices) + EXTRA_ABOVE
        f = (num / den) if den > 0 else 0.0
        l, h = wilson_score_interval(num, den, z=1.0)
        
        fractions_full[k] = f
        err_lows_full[k] = f - l
        err_highs_full[k] = h - f

    # 4. Filter for changes only (if enabled)
    if filter_changes_only and len(fractions_full) > 0:
        keep = [0]
        prev_f = fractions_full[0]
        for k in range(1, len(fractions_full)):
            if abs(fractions_full[k] - prev_f) > 1e-9:
                keep.append(k)
                prev_f = fractions_full[k]
        
        res = {
            't': t_vals_full[keep],
            'f': fractions_full[keep],
            'el': err_lows_full[keep],
            'eh': err_highs_full[keep],
            'contam': contaminated_change_flags[keep],
            'labels': dropped_star_labels[keep],
            'indices': current_indices
        }
    else:
        res = {
            't': t_vals_full,
            'f': fractions_full,
            'el': err_lows_full,
            'eh': err_highs_full,
            'contam': contaminated_change_flags,
            'labels': dropped_star_labels,
            'indices': current_indices
        }
        
    return res

# ============================================================
# 3) PLOTTING FUNCTION
# ============================================================
def plot_interactive(show_labels, show_fit, exclude_dirty):
    
    # --- GET DATA ---
    data = recalc_data(exclude_dirty)
    if data is None:
        print("No stars remaining after filter.")
        return

    t_vals = data['t']
    f_arr = data['f']
    sigma = (data['el'] + data['eh']) / 2.0
    sigma[~np.isfinite(sigma)] = 0.01
    sigma[sigma <= 0] = 0.01

    # --- FIT ---
    has_valid_fit = False
    fit_res = None
    if show_fit and fit_elbow and len(f_arr) > 5:
        fit_res = _fit_two_segment_linear_weighted(t_vals, f_arr, sigma)
        if fit_res:
            has_valid_fit = True

    # --- SETUP CANVAS ---
    if has_valid_fit:
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=S['FIG_SIZE_FIT'], sharex=True,
            gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.08}
        )
    else:
        fig, ax1 = plt.subplots(1, 1, figsize=S['FIG_SIZE_NOFIT'])
        ax2 = None

    # --- PLOT MAIN DATA ---
    mask_red = data['contam']
    mask_blue = ~data['contam']
    ms = 8 if presentation else 6

    if np.any(mask_blue):
        ax1.errorbar(
            t_vals[mask_blue], f_arr[mask_blue],
            yerr=[data['el'][mask_blue], data['eh'][mask_blue]],
            fmt='o', markersize=ms, linestyle='none',
            color='blue', ecolor='blue', capsize=4,
            label="Data (Clean)"
        )

    if np.any(mask_red):
        ax1.errorbar(
            t_vals[mask_red], f_arr[mask_red],
            yerr=[data['el'][mask_red], data['eh'][mask_red]],
            fmt='o', markersize=ms, linestyle='none',
            color='red', ecolor='red', capsize=4,
            label="Data (Contaminated)"
        )

    # Labels
    if show_labels:
        for t_p, f_p, lbl in zip(t_vals, f_arr, data['labels']):
            if lbl and isinstance(lbl, str) and len(lbl) > 0:
                ax1.annotate(
                    lbl, xy=(t_p, f_p), xytext=(0, 10),
                    textcoords='offset points', ha='center', va='bottom',
                    fontsize=S['FS_TINY'], rotation=45, alpha=0.8
                )

    # --- PLOT FIT ---
    if has_valid_fit and ax2 is not None:
        lw = 2.5 if presentation else 2
        elbow_x = fit_res['elbow_x']
        
        # Real data fraction at elbow
        current_indices = data['indices']
        n_pass_real = 0
        for idx in current_indices:
            sname = masked.at[idx, "Star"]
            dval = float(masked.at[idx, sort_col_use])
            if _is_significant_binary(sname, line_key, dval, elbow_x):
                n_pass_real += 1
        
        num = n_pass_real + EXTRA_ABOVE
        den = len(current_indices) + EXTRA_ABOVE
        real_y = (num / den) if den > 0 else 0.0

        # Segments
        x1_r = np.linspace(fit_res['range1'][0], elbow_x, 50)
        ax1.plot(x1_r, np.polyval(fit_res['p1'], x1_r), color='orange', linewidth=lw, label="2-Seg Fit")
        x2_r = np.linspace(elbow_x, fit_res['range2'][1], 50)
        ax1.plot(x2_r, np.polyval(fit_res['p2'], x2_r), color='orange', linewidth=lw)
        
        # Lines
        ax1.axvline(elbow_x, color='orange', linestyle='--', linewidth=lw, 
                    label=f"Fit X ({elbow_x:.1f} km/s)")
        ax2.axvline(elbow_x, color='orange', linestyle='--', linewidth=lw*0.7, alpha=0.7)
        
        ax1.axhline(real_y, color='gray', linestyle='--', linewidth=lw, alpha=0.6,
                    label=f"Real Data Y ($f={real_y:.2f}$)")

        # Residuals
        y_fit = np.where(
            t_vals <= elbow_x,
            np.polyval(fit_res['p1'], t_vals),
            np.polyval(fit_res['p2'], t_vals)
        )
        resid = f_arr - y_fit
        
        chi2_val = float(np.sum((resid / sigma) ** 2))
        dof = int(max(len(resid) - 4, 1))
        chi2_red = chi2_val / dof
        p_val = float(chi2.sf(chi2_val, dof))
        
        stats_text = (f"$\\chi^2_\\nu$={chi2_red:.2f}\n$p$={p_val:.2g}\nDoF={dof}")
        ax2.text(0.02, 0.95, stats_text, transform=ax2.transAxes, ha='left', va='top', 
                 fontsize=S["FS_NOTE"], bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
        
        ax2.axhline(0, color='black', linewidth=1)
        ax2.errorbar(t_vals, resid, yerr=sigma, fmt='o', markersize=ms-2, 
                     color='purple', ecolor='purple', capsize=4)
        ax2.set_ylabel("Residuals", fontsize=S['FS_AXIS_LBL'])
        ax2.grid(True, alpha=0.3)
        ax2.set_xticks(np.arange(0, MAX_RV_THRESH + 1, max(10, int(MAX_RV_THRESH / 10))))
        ax2.set_xlabel("Threshold [km/s]", fontsize=S['FS_AXIS_LBL'])
    else:
        ax1.set_xlabel("Threshold [km/s]", fontsize=S['FS_AXIS_LBL'])
        ax1.set_xticks(np.arange(0, MAX_RV_THRESH + 1, max(10, int(MAX_RV_THRESH / 10))))

    if np.isfinite(equiv_thresh):
        ax1.axvline(equiv_thresh, color='green', linestyle='--', linewidth=2, alpha=0.7, label=f'Equiv ({equiv_thresh:.1f})')
        if ax2 is not None:
            ax2.axvline(equiv_thresh, color='green', linestyle='--', linewidth=1.5, alpha=0.7)

    ax1.set_ylabel(r"$f_{\mathrm{bin}}$", fontsize=S['FS_AXIS_LBL'])
    ax1.set_title(f"Binary Fraction vs Threshold: {line_key}", fontsize=S['FS_AXIS_LBL']*1.1)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.15)
    
    # --- DRAGGABLE LEGEND ---
    handles, labels = ax1.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    leg = ax1.legend(by_label.values(), by_label.keys(), loc='best', fontsize=S["FS_NOTE"])
    leg.set_draggable(True)
    
    plt.tight_layout()
    plt.show()

# ============================================================
# 4) WIDGETS
# ============================================================
chk_labels = widgets.Checkbox(value=False, description='Show Star Names')
chk_fit = widgets.Checkbox(value=True, description='Show Linear Fit')
chk_clean = widgets.Checkbox(value=False, description='Exclude Contaminated')

ui = widgets.HBox([chk_labels, chk_fit, chk_clean])
out = widgets.interactive_output(plot_interactive, {
    'show_labels': chk_labels, 
    'show_fit': chk_fit, 
    'exclude_dirty': chk_clean
})

display(ui, out)

### Graph 3 - Equivalent Thresholds

In [ ]:
# Cell 4 — Plot 3: Equivalent Thresholds across lines

for_presentation = True
S = set_plot_style(for_presentation)

lines_plot = []
thresholds_plot = []
for lk in ordered_lines:
    lines_plot.append(lk)
    thresholds_plot.append(equiv_thresholds_map.get(lk, np.nan))

if len(lines_plot) == 0:
    print("No emission lines found in ordered_lines (LINES_DEFAULT was empty?)")
else:
    plt.figure(figsize=FIGSIZE_THRESH_PLOT)
    plt.plot(lines_plot, thresholds_plot, marker='o', linestyle='-', color='purple')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel("Equivalent Threshold [km/s]")
    plt.title("Thresholds yielding $f_{\\mathrm{bin}}$ ≈ $f_{\\mathrm{bin}}$(CIV, 20 km/s)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


### Graph 4 - Corner Plot Matrix

In [ ]:
# Cell 5 — Plot 4: Corner plot (FIXED)

# --- USER SETTINGS ---
use_equiv_thresh = False
force_show_low_detection = False
use_log_scale = True
show_names = False
for_presentation = True

S = set_plot_style(for_presentation)

# IMPORTANT: prevents "More than 20 figures have been opened" when re-running
plt.close('all')

masked, _ = build_masked_df(
    df,
    sort_col=f"dRV | {CIV_KEY}" if f"dRV | {CIV_KEY}" in df.columns else MEAN_COL,
    ascending=False,
    force_show_low_detection=force_show_low_detection
)

def _safe_log_bins(vals, x_min, x_max, nbins=15):
    """
    Build strictly-increasing log bins safely.
    Returns either an int (let matplotlib decide) or a monotonic array.
    """
    pos = np.array([v for v in vals if np.isfinite(v) and v > 0], dtype=float)
    if pos.size == 0:
        return nbins  # fallback

    vmin = np.min(pos)
    vmax = np.max(pos)

    # Try to respect axis range, but don't let it break monotonicity
    hv_min = max(float(x_min), float(vmin)) if np.isfinite(x_min) else float(vmin)
    hv_max = min(float(x_max), float(vmax)) if np.isfinite(x_max) else float(vmax)

    # Guard against invalid / equal ranges
    if (not np.isfinite(hv_min)) or (not np.isfinite(hv_max)) or (hv_min <= 0) or (hv_max <= 0):
        return nbins
    if hv_max <= hv_min * (1.0 + 1e-9):
        # expand a bit so geomspace is strictly increasing
        hv_max = hv_min * 1.05
        if hv_max <= hv_min:
            return nbins

    bins = np.geomspace(hv_min, hv_max, nbins)
    # Final safety: ensure strictly increasing
    if np.any(np.diff(bins) <= 0):
        return nbins
    return bins


n_lines = len(ordered_lines)
if n_lines <= 1:
    print("Need at least 2 emission lines for a corner plot.")
else:
    # --- Precompute ranges per line ---
    line_ranges = {}
    for lk in ordered_lines:
        col = f"dRV | {lk}"
        vals_rng = []

        # include threshold so it appears in range
        t_val = equiv_thresholds_map.get(lk, THRESH_KMS) if use_equiv_thresh else THRESH_KMS
        if np.isfinite(t_val):
            vals_rng.append(float(t_val))

        if col in masked.columns:
            for i in range(len(masked)):
                sname = masked.at[i, "Star"]
                _, _, frac = ew_fail_stats.get((sname, lk), (0, 0, 0))
                if frac > FAIL_FRAC_THRESH:
                    continue
                v = masked.at[i, col]
                if pd.notna(v) and np.isfinite(float(v)):
                    vals_rng.append(float(v))

        if not vals_rng:
            line_ranges[lk] = (0.1, MAX_RV_THRESH)
            continue

        vmin, vmax = float(np.min(vals_rng)), float(np.max(vals_rng))

        if use_log_scale:
            pos_vals = [x for x in vals_rng if x > 0]
            if pos_vals:
                vmin = float(np.min(pos_vals))
                log_min, log_max = np.log10(vmin), np.log10(vmax if vmax > 0 else vmin * 1.05)
                span = log_max - log_min if log_max > log_min else 0.5
                vmin = 10 ** (log_min - 0.05 * span)
                vmax = 10 ** (log_max + 0.05 * span)
            else:
                vmin, vmax = 0.1, MAX_RV_THRESH
        else:
            span = (vmax - vmin) if (vmax > vmin) else 5.0
            vmin = max(0.0, vmin - 0.05 * span)
            vmax = vmax + 0.05 * span

        # final guard
        if use_log_scale and (vmin <= 0 or vmax <= 0):
            vmin, vmax = 0.1, MAX_RV_THRESH
        if vmax <= vmin:
            vmax = vmin * 1.05 if use_log_scale else (vmin + 1.0)

        line_ranges[lk] = (vmin, vmax)

    dim = max(16, 3 * n_lines)
    fig, axes = plt.subplots(n_lines, n_lines, figsize=(dim, dim), sharex='col')
    plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, wspace=0, hspace=0)

    trans_diagonal = mtransforms.Affine2D().rotate_deg(-45)

    for i in range(n_lines):
        line_y_key = ordered_lines[i]
        y_min, y_max = line_ranges.get(line_y_key, (0.1, MAX_RV_THRESH))
        thresh_y = equiv_thresholds_map.get(line_y_key, THRESH_KMS) if use_equiv_thresh else THRESH_KMS

        for j in range(n_lines):
            ax = axes[i, j]
            line_x_key = ordered_lines[j]
            x_min, x_max = line_ranges.get(line_x_key, (0.1, MAX_RV_THRESH))
            thresh_x = equiv_thresholds_map.get(line_x_key, THRESH_KMS) if use_equiv_thresh else THRESH_KMS

            # labels
            if i == n_lines - 1:
                ax.set_xlabel(format_line_label(line_x_key), fontsize=S["FS_AXIS_LBL"])
            if j == 0 and i != j:
                ax.set_ylabel(format_line_label(line_y_key), fontsize=S["FS_AXIS_LBL"])

            # scales
            if use_log_scale:
                ax.set_xscale('log')
                if i != j:
                    ax.set_yscale('log')

            ax.set_xlim(x_min, x_max)
            if i != j:
                ax.set_ylim(y_min, y_max)

            # tick visibility
            if i < n_lines - 1:
                ax.tick_params(axis='x', labelbottom=False, length=0)
            if j > 0:
                ax.tick_params(axis='y', labelleft=False, length=0)

            # upper triangle off
            if j > i:
                ax.axis('off')
                continue

            # diagonal: histogram
            if i == j:
                col_name = f"dRV | {line_y_key}"
                if col_name in masked.columns:
                    vals_hist = []
                    for row_idx in range(len(masked)):
                        sname = masked.at[row_idx, "Star"]
                        _, _, frac = ew_fail_stats.get((sname, line_y_key), (0, 0, 0))
                        if frac > FAIL_FRAC_THRESH:
                            continue
                        v = masked.at[row_idx, col_name]
                        if pd.notna(v) and np.isfinite(float(v)):
                            vals_hist.append(float(v))

                    if len(vals_hist) > 0:
                        if use_log_scale:
                            bins = _safe_log_bins(vals_hist, x_min, x_max, nbins=15)
                        else:
                            bins = 15

                        ax.hist(vals_hist, bins=bins, color='gray', alpha=0.7,
                                histtype='stepfilled', edgecolor='black')

                ax.yaxis.set_visible(False)
                ax.grid(True, alpha=0.2, which='both')
                continue

            # off-diagonal: scatter with split-color markers
            x_vals, y_vals, x_errs, y_errs = [], [], [], []
            c_topleft, c_botright = [], []
            names_to_plot = []

            col_x = f"dRV | {line_x_key}"
            col_y = f"dRV | {line_y_key}"

            if (col_x in masked.columns) and (col_y in masked.columns):
                for row_idx in range(len(masked)):
                    sname = masked.at[row_idx, "Star"]
                    _, _, frac_x = ew_fail_stats.get((sname, line_x_key), (0, 0, 0))
                    _, _, frac_y = ew_fail_stats.get((sname, line_y_key), (0, 0, 0))
                    if (frac_x > FAIL_FRAC_THRESH) or (frac_y > FAIL_FRAC_THRESH):
                        continue

                    vx = masked.at[row_idx, col_x]
                    vy = masked.at[row_idx, col_y]
                    if pd.notna(vx) and np.isfinite(float(vx)) and pd.notna(vy) and np.isfinite(float(vy)):
                        vx = float(vx); vy = float(vy)
                        if use_log_scale and (vx <= 0 or vy <= 0):
                            continue

                        x_vals.append(vx); y_vals.append(vy)

                        err_x = drverr_map.get((sname, line_x_key), 0.0)
                        err_y = drverr_map.get((sname, line_y_key), 0.0)
                        x_errs.append(err_x if np.isfinite(err_x) else 0.0)
                        y_errs.append(err_y if np.isfinite(err_y) else 0.0)

                        pass_y = _is_significant_binary(sname, line_y_key, vy, thresh_y)
                        pass_x = _is_significant_binary(sname, line_x_key, vx, thresh_x)
                        c_topleft.append('green' if pass_y else 'red')
                        c_botright.append('green' if pass_x else 'red')
                        names_to_plot.append(sname)

            if len(x_vals) > 0:
                x_arr = np.array(x_vals)
                y_arr = np.array(y_vals)
                x_err_arr = np.array(x_errs)
                y_err_arr = np.array(y_errs)

                ax.axvline(thresh_x, color='black', linestyle=':', alpha=0.6, linewidth=1)
                ax.axhline(thresh_y, color='black', linestyle=':', alpha=0.6, linewidth=1)

                diag_min = max(x_min, y_min)
                diag_max = min(x_max, y_max)
                if diag_max > diag_min:
                    ax.plot([diag_min, diag_max], [diag_min, diag_max],
                            ls='--', color='gray', alpha=0.5, linewidth=1, zorder=0)

                fbin_y_str = _get_fbin_simple(masked, line_y_key, use_equiv_thresh)
                ax.text(0.05, 0.95, r"$f_{\mathrm{bin}}$(Y): " + fbin_y_str,
                        transform=ax.transAxes, fontsize=S["FS_NOTE"], va='top', ha='left',
                        bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

                fbin_x_str = _get_fbin_simple(masked, line_x_key, use_equiv_thresh)
                ax.text(0.95, 0.05, r"$f_{\mathrm{bin}}$(X): " + fbin_x_str,
                        transform=ax.transAxes, fontsize=S["FS_NOTE"], va='bottom', ha='right',
                        bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

                ax.hlines(y_arr, x_arr - x_err_arr, x_arr + x_err_arr, colors=c_botright, lw=0.8, zorder=1)
                ax.vlines(x_arr, y_arr - y_err_arr, y_arr + y_err_arr, colors=c_topleft, lw=0.8, zorder=1)

                ax.scatter(x_vals, y_vals, c=c_botright,
                           marker=MarkerStyle('o', fillstyle='right', transform=trans_diagonal),
                           s=80, edgecolors='black', linewidth=0.5, zorder=2)
                ax.scatter(x_vals, y_vals, c=c_topleft,
                           marker=MarkerStyle('o', fillstyle='left', transform=trans_diagonal),
                           s=80, edgecolors='black', linewidth=0.5, zorder=3)

                if len(x_vals) > 1:
                    r_val, p_val = pearsonr(x_vals, y_vals)
                    ax.text(0.05, 0.85, f"r={r_val:.2f}\np={p_val:.2g}\nn={len(x_vals)}",
                            transform=ax.transAxes, fontsize=S["FS_NOTE"], va='top', ha='left',
                            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8))

                if show_names:
                    for k, nm in enumerate(names_to_plot):
                        ax.text(x_vals[k], y_vals[k], nm, fontsize=S["FS_TINY"], alpha=0.8,
                                clip_on=True, rotation=25)

            ax.grid(True, alpha=0.2, which='both')

    plt.show()


### Graph 5 - Agreement Ranking

In [ ]:
# Cell 6 — Plot 5: Agreement Ranking (correlation weighted by N/25)

# --- USER SETTINGS ---
use_equiv_thresh = False
force_show_low_detection = False
show_extra_data = False
for_presentation = True

S = set_plot_style(for_presentation)

masked, _ = build_masked_df(
    df,
    sort_col=f"dRV | {CIV_KEY}" if f"dRV | {CIV_KEY}" in df.columns else MEAN_COL,
    ascending=False,
    force_show_low_detection=force_show_low_detection
)

drv_cols = [f"dRV | {lk}" for lk in ordered_lines if f"dRV | {lk}" in masked.columns]
if len(drv_cols) <= 1:
    print("Need at least 2 dRV columns for Agreement Ranking.")
else:
    cols_names = drv_cols
    adj_corr_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
    adj_pval_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
    adj_n_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)

    for i, c1 in enumerate(cols_names):
        data1 = masked[c1]
        for j, c2 in enumerate(cols_names):
            if i == j:
                continue
            data2 = masked[c2]
            valid = data1.notna() & data2.notna()
            n_dots = int(valid.sum())
            if n_dots < MIN_STARS_FOR_CORR:
                continue
            r, p = pearsonr(data1[valid], data2[valid])
            adj_corr_df.at[c1, c2] = r
            adj_pval_df.at[c1, c2] = p
            adj_n_df.at[c1, c2] = n_dots

    w_df = adj_n_df / MAX_POSSIBLE_STARS
    r_weighted_terms = adj_corr_df * w_df
    avg_scores_user = r_weighted_terms.sum(axis=1) / r_weighted_terms.count(axis=1)

    p_times_n = adj_pval_df * adj_n_df
    sum_n = adj_n_df.sum(axis=1)
    avg_pvals_std_weighted = p_times_n.sum(axis=1) / sum_n.replace(0, np.nan)

    best_friends, best_scores = [], []
    second_best_friends, second_best_scores = [], []
    stats_texts = []

    for col in cols_names:
        line_nm = col.replace("dRV | ", "")
        row_series = adj_corr_df.loc[col].sort_values(ascending=False)
        valid_friends = row_series.dropna()

        def _fmt_friend(friend_col, r_val):
            v = (masked[col].notna() & masked[friend_col].notna())
            n_count = int(v.sum())
            if n_count < 2:
                return "Err", np.nan
            _, p_val = pearsonr(masked[col][v], masked[friend_col][v])
            nm = friend_col.replace("dRV | ", "")
            return f"{nm}\nr={r_val:.2f} (p={p_val:.2g}, n={n_count})", r_val

        if len(valid_friends) > 0:
            b_lbl, b_val = _fmt_friend(valid_friends.index[0], valid_friends.iloc[0])
        else:
            b_val, b_lbl = np.nan, "None"

        if len(valid_friends) > 1:
            s_lbl, s_val = _fmt_friend(valid_friends.index[1], valid_friends.iloc[1])
        else:
            s_val, s_lbl = np.nan, ""

        best_scores.append(b_val)
        best_friends.append(b_lbl)
        second_best_scores.append(s_val)
        second_best_friends.append(s_lbl)

        fbin_txt = _get_fbin_simple(masked, line_nm, use_equiv_thresh)
        ew_succ, ew_mn, ew_sem = _calc_ew_stats_for_line(line_nm)
        stats_texts.append(r"$f_{\mathrm{bin}}$: " + f"{fbin_txt}\nEW%: {ew_succ:.0%}\nEW: {ew_mn:.2f}±{ew_sem:.2f}")

    plot_data = pd.DataFrame({
        "Line": [c.replace("dRV | ", "") for c in cols_names],
        "Score": avg_scores_user.values,
        "Avg_P_Weighted": avg_pvals_std_weighted.values,
        "Best_Score": best_scores,
        "Best_Friend": best_friends,
        "Second_Score": second_best_scores,
        "Second_Friend": second_best_friends,
        "Stats": stats_texts
    }).sort_values("Score", ascending=False, na_position='last').reset_index(drop=True)

    plt.figure(figsize=FIGSIZE_AGREEMENT)
    valid_plot = plot_data.dropna(subset=["Score"])
    if not valid_plot.empty:
        plt.plot(valid_plot.index, valid_plot["Score"], marker='o', linestyle='-',
                 color='teal', linewidth=2, markersize=8,
                 label=f"Avg Score (Weight = N/{MAX_POSSIBLE_STARS})")

    if show_extra_data:
        plt.scatter(plot_data.index, plot_data["Best_Score"], color='orange', s=80,
                    marker='D', zorder=3, label="Max Correlation")
        plt.scatter(plot_data.index, plot_data["Second_Score"], color='lightgreen', s=70,
                    marker='s', edgecolors='green', zorder=2, label="2nd Best Correlation")

    plt.xticks(plot_data.index, plot_data["Line"], rotation=45, ha="right")
    plt.ylabel("Agreement Score")
    plt.title(f"Agreement Ranking (Logic: r * n/25, N >= {MIN_STARS_FOR_CORR})")
    plt.grid(True, alpha=0.3, linestyle='--')

    for i in plot_data.index:
        score = plot_data.at[i, "Score"]
        avg_p_w = plot_data.at[i, "Avg_P_Weighted"]
        best = plot_data.at[i, "Best_Score"]
        friend = plot_data.at[i, "Best_Friend"]
        second = plot_data.at[i, "Second_Score"]
        sec_friend = plot_data.at[i, "Second_Friend"]
        stat_txt = plot_data.at[i, "Stats"]

        if pd.notna(score):
            p_text = f"\n(p_w={avg_p_w:.2g})" if (show_extra_data and pd.notna(avg_p_w)) else ""
            plt.text(i, score - 0.02, f"{score:.3f}{p_text}",
                     ha='center', va='top', fontsize=S["FS_NOTE"],
                     color='teal', fontweight='bold')

        if show_extra_data:
            if pd.notna(best):
                plt.text(i, best + 0.01, friend, ha='center', va='bottom',
                         fontsize=S["FS_TINY"], color='darkorange', rotation=25)
            if pd.notna(second):
                plt.text(i, second - 0.01, sec_friend, ha='center', va='top',
                         fontsize=S["FS_TINY"], color='green', rotation=25)

            ylim_min, ylim_max = plt.ylim()
            y_txt_pos = ylim_min + (ylim_max - ylim_min) * 0.05
            plt.text(i, y_txt_pos, stat_txt, ha='center', va='bottom',
                     fontsize=S["FS_BOX_STAT"],
                     bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="gray", alpha=0.7))

    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

    display(plot_data)


### Graph 6 - Binary Fraction per Line

In [ ]:
# Cell 7 — Plot 6: Binary fraction per emission line

# --- USER SETTINGS ---
use_equiv_thresh = False
force_show_low_detection = False
sort_by_agreement = False
show_hard_and_fit_and_d2 = False
filter_changes_only = True
for_presentation = True

S = set_plot_style(for_presentation)

masked, _ = build_masked_df(
    df,
    sort_col=f"dRV | {CIV_KEY}" if f"dRV | {CIV_KEY}" in df.columns else MEAN_COL,
    ascending=False,
    force_show_low_detection=force_show_low_detection
)

if sort_by_agreement:
    drv_cols = [f"dRV | {lk}" for lk in ordered_lines if f"dRV | {lk}" in masked.columns]
    if len(drv_cols) > 1:
        cols_names = drv_cols
        adj_corr_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)
        adj_n_df = pd.DataFrame(np.nan, index=cols_names, columns=cols_names)

        for i, c1 in enumerate(cols_names):
            data1 = masked[c1]
            for j, c2 in enumerate(cols_names):
                if i == j:
                    continue
                data2 = masked[c2]
                valid = data1.notna() & data2.notna()
                n_dots = int(valid.sum())
                if n_dots < MIN_STARS_FOR_CORR:
                    continue
                r, _ = pearsonr(data1[valid], data2[valid])
                adj_corr_df.at[c1, c2] = r
                adj_n_df.at[c1, c2] = n_dots

        w_df = adj_n_df / MAX_POSSIBLE_STARS
        r_weighted_terms = adj_corr_df * w_df
        avg_scores_user = r_weighted_terms.sum(axis=1) / r_weighted_terms.count(axis=1)

        tmp = pd.DataFrame({
            "Line": [c.replace("dRV | ", "") for c in cols_names],
            "Score": avg_scores_user.values
        }).sort_values("Score", ascending=False, na_position="last")

        sorted_lines_g6 = tmp["Line"].tolist()
    else:
        sorted_lines_g6 = sorted(ordered_lines, key=_get_wavelength)
else:
    sorted_lines_g6 = sorted(ordered_lines, key=_get_wavelength)

x_pos = np.arange(len(sorted_lines_g6))
plt.figure(figsize=FIGSIZE_FBIN_BAR)

if show_hard_and_fit_and_d2:
    hard_vals, hard_el, hard_eh = [], [], []
    fit_vals, fit_el, fit_eh = [], [], []
    d2_vals, d2_el, d2_eh = [], [], []

    for lk in sorted_lines_g6:
        f_h, el_h, eh_h, _, _ = _calc_fbin_stats_at_threshold(masked, lk, THRESH_KMS)
        hard_vals.append(f_h * 100.0); hard_el.append(el_h * 100.0); hard_eh.append(eh_h * 100.0)

        t_fit = get_fit_elbow_threshold_for_line(masked, lk, filter_changes=filter_changes_only)
        if np.isfinite(t_fit):
            f_f, el_f, eh_f, _, _ = _calc_fbin_stats_at_threshold(masked, lk, t_fit)
            fit_vals.append(f_f * 100.0); fit_el.append(el_f * 100.0); fit_eh.append(eh_f * 100.0)
        else:
            fit_vals.append(np.nan); fit_el.append(0.0); fit_eh.append(0.0)

        t_d2 = get_discrete_d2_threshold_for_line(masked, lk, filter_changes=filter_changes_only)
        if np.isfinite(t_d2):
            f_d, el_d, eh_d, _, _ = _calc_fbin_stats_at_threshold(masked, lk, t_d2)
            d2_vals.append(f_d * 100.0); d2_el.append(el_d * 100.0); d2_eh.append(eh_d * 100.0)
        else:
            d2_vals.append(np.nan); d2_el.append(0.0); d2_eh.append(0.0)

    offset = 0.25
    plt.errorbar(x_pos - offset, hard_vals, yerr=[hard_el, hard_eh], fmt='o',
                 color='cornflowerblue', ecolor='gray', capsize=5, elinewidth=2, markersize=7,
                 label=f"Hard ({THRESH_KMS:.0f} km/s)")
    plt.errorbar(x_pos, fit_vals, yerr=[fit_el, fit_eh], fmt='o',
                 color='darkorange', ecolor='gray', capsize=5, elinewidth=2, markersize=7,
                 label="Fit threshold (2-line elbow)")
    plt.errorbar(x_pos + offset, d2_vals, yerr=[d2_el, d2_eh], fmt='o',
                 color='magenta', ecolor='gray', capsize=5, elinewidth=2, markersize=7,
                 label="Max +d^2 threshold (discrete)")

    for i, (hv, fv, dv) in enumerate(zip(hard_vals, fit_vals, d2_vals)):
        if np.isfinite(hv):
            plt.text(i - offset, hv + 2, f"{hv:.0f}%", ha='center', va='bottom',
                     fontsize=max(S["FS_BAR_VAL"] - 2, 8), fontweight='bold')
        if np.isfinite(fv):
            plt.text(i, fv + 2, f"{fv:.0f}%", ha='center', va='bottom',
                     fontsize=max(S["FS_BAR_VAL"] - 2, 8), fontweight='bold')
        if np.isfinite(dv):
            plt.text(i + offset, dv + 2, f"{dv:.0f}%", ha='center', va='bottom',
                     fontsize=max(S["FS_BAR_VAL"] - 2, 8), fontweight='bold')

    plt.title("Binary Fraction per Emission Line (Hard vs Fit vs Max +d^2)")
    plt.legend(loc='upper right')

else:
    fbin_vals, err_lows, err_highs = [], [], []
    for lk in sorted_lines_g6:
        f, el, eh, _, _ = _calc_fbin_stats_internal(masked, lk, use_equiv_thresh)
        fbin_vals.append(f * 100.0); err_lows.append(el * 100.0); err_highs.append(eh * 100.0)

    plt.errorbar(x_pos, fbin_vals, yerr=[err_lows, err_highs], fmt='o',
                 color='cornflowerblue', ecolor='gray', capsize=5, elinewidth=2, markersize=8)

    for i, v in enumerate(fbin_vals):
        plt.text(i, v + 2, f"{v:.0f}%", ha='center', va='bottom',
                 fontsize=S["FS_BAR_VAL"], fontweight='bold')

    plt.title(f"Binary Fraction per Emission Line ({'Equiv Thresholds' if use_equiv_thresh else 'Hard Threshold'})")

plt.xticks(x_pos, sorted_lines_g6, rotation=45, ha='right')
plt.ylabel("Binary Fraction (%)")
plt.ylim(0, 105)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()


### Graph 7: Confidence grading (Golden/Silver/Bronze)

In [ ]:
# Cell 8 — Plot 7: Confidence Grading

use_equiv_thresh = False
highlight_contam = False
for_presentation = True

S = set_plot_style(for_presentation)

star_results = get_star_classification_data(df, use_equiv_thresh)

grades_data = {
    'Golden': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []},
    'Silver': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []},
    'Bronze': {'bin_clean': [], 'bin_contam': [], 'sin_clean': [], 'sin_contam': []}
}

for sname, info in star_results.items():
    grade = info['grade']
    is_bin = info['is_binary']
    is_clean = info['is_clean']
    if is_bin:
        (grades_data[grade]['bin_clean'] if is_clean else grades_data[grade]['bin_contam']).append(sname)
    else:
        (grades_data[grade]['sin_clean'] if is_clean else grades_data[grade]['sin_contam']).append(sname)

fig, axes = plt.subplots(1, 3, figsize=(14, 8))
theme_map = {
    'Golden': (['#FFD700', '#FFFACD'], '#DAA520'),
    'Silver': (['#A9A9A9', '#DCDCDC'], 'gray'),
    'Bronze': (['#CD853F', '#FFE4C4'], '#8B4513')
}

grade_keys = ['Golden', 'Silver', 'Bronze']
titles = ["Golden (>90% Agree)", "Silver (>70% Agree)", "Bronze (Rest)"]

for idx, g in enumerate(grade_keys):
    ax = axes[idx]
    data = grades_data[g]
    n_bin_clean = len(data['bin_clean'])
    n_bin_contam = len(data['bin_contam'])
    n_sin_clean = len(data['sin_clean'])
    n_sin_contam = len(data['sin_contam'])
    total = n_bin_clean + n_bin_contam + n_sin_clean + n_sin_contam

    colors_theme, title_col = theme_map[g]
    c_bin_main, c_sin_main = colors_theme

    if total > 0:
        if highlight_contam:
            n_bin = n_bin_clean + n_bin_contam
            n_sin = n_sin_clean + n_sin_contam
            outer_sizes = [s for s in [n_bin, n_sin] if s > 0]
            outer_labels = [l for s, l in zip([n_bin, n_sin], ["Binary", "Single"]) if s > 0]
            outer_colors = [c for s, c in zip([n_bin, n_sin], [c_bin_main, c_sin_main]) if s > 0]

            ax.pie(outer_sizes, labels=outer_labels, colors=outer_colors,
                   radius=1.0, wedgeprops=dict(width=0.3, edgecolor='w'),
                   startangle=90, textprops={'fontsize': S["FS_PIE_LABEL"], 'weight': 'bold'},
                   autopct='%1.1f%%', pctdistance=0.85)

            inner_sizes = [n_bin_clean, n_bin_contam, n_sin_clean, n_sin_contam]
            inner_colors = [c_bin_main, '#FF4444', c_sin_main, '#FF4444']
            f_in_sizes = [s for s in inner_sizes if s > 0]
            f_in_colors = [c for s, c in zip(inner_sizes, inner_colors) if s > 0]

            if f_in_sizes:
                ax.pie(f_in_sizes, colors=f_in_colors, radius=0.7,
                       wedgeprops=dict(width=0.7, edgecolor='w'),
                       startangle=90, autopct='%1.1f%%', pctdistance=0.6,
                       textprops={'fontsize': S["FS_TINY"]})
        else:
            n_bin = n_bin_clean + n_bin_contam
            n_sin = n_sin_clean + n_sin_contam
            ax.pie([n_bin, n_sin],
                   labels=[f"Binary\n{n_bin}", f"Single\n{n_sin}"],
                   colors=colors_theme, autopct='%1.1f%%', startangle=90,
                   textprops={'fontsize': S["FS_PIE_LABEL"]})
    else:
        ax.text(0.5, 0.5, "No Stars", ha='center', va='center', fontsize=S["FS_PIE_LABEL"])
        ax.axis('off')

    ax.set_title(titles[idx], color=title_col, fontsize=S["FS_PIE_LABEL"] + 2, fontweight='bold')

    def _get_wrapped(lst):
        return "\n".join(textwrap.wrap(", ".join(lst), width=20)) if lst else ""

    bin_clean_txt = _get_wrapped(data['bin_clean'])
    bin_contam_txt = _get_wrapped(data['bin_contam'])
    sin_clean_txt = _get_wrapped(data['sin_clean'])
    sin_contam_txt = _get_wrapped(data['sin_contam'])

    ax.text(0.05, -0.05, r"$\bf{Binary:}$", transform=ax.transAxes,
            ha='left', va='top', fontsize=S["FS_LIST_TEXT"])
    current_y = -0.12
    if bin_clean_txt:
        ax.text(0.05, current_y, bin_clean_txt, transform=ax.transAxes,
                ha='left', va='top', fontsize=S["FS_LIST_TEXT"])
        current_y -= (bin_clean_txt.count('\n') + 2) * 0.05
    if bin_contam_txt:
        ax.text(0.05, current_y, bin_contam_txt, transform=ax.transAxes,
                ha='left', va='top', fontsize=S["FS_LIST_TEXT"],
                color=('red' if highlight_contam else 'black'),
                fontweight=('bold' if highlight_contam else 'normal'))

    ax.text(0.55, -0.05, r"$\bf{Single:}$", transform=ax.transAxes,
            ha='left', va='top', fontsize=S["FS_LIST_TEXT"])
    current_y = -0.12
    if sin_clean_txt:
        ax.text(0.55, current_y, sin_clean_txt, transform=ax.transAxes,
                ha='left', va='top', fontsize=S["FS_LIST_TEXT"])
        current_y -= (sin_clean_txt.count('\n') + 2) * 0.05
    if sin_contam_txt:
        ax.text(0.55, current_y, sin_contam_txt, transform=ax.transAxes,
                ha='left', va='top', fontsize=S["FS_LIST_TEXT"],
                color=('red' if highlight_contam else 'black'),
                fontweight=('bold' if highlight_contam else 'normal'))

plt.tight_layout()
plt.show()


### Graph 8: All/Clean/Contaminated comparison

In [ ]:
# Cell 9 — Plot 8: All / Clean / Contaminated Comparison

use_equiv_thresh = False
for_presentation = True

S = set_plot_style(for_presentation)

star_results = get_star_classification_data(df, use_equiv_thresh)

clean_counts = {'Binary': 0, 'Single': 0}
contam_counts = {'Binary': 0, 'Single': 0}
all_counts = {'Binary': 0, 'Single': 0}

clean_lists = {'Binary': [], 'Single': []}
contam_lists = {'Binary': [], 'Single': []}
all_lists = {'Binary': [], 'Single': []}

for sname, info in star_results.items():
    k = 'Binary' if info['is_binary'] else 'Single'
    all_counts[k] += 1
    all_lists[k].append(sname)
    if info['is_clean']:
        clean_counts[k] += 1
        clean_lists[k].append(sname)
    else:
        contam_counts[k] += 1
        contam_lists[k].append(sname)

fig, axes = plt.subplots(1, 3, figsize=(16, 8))

def plot_pie_with_lists(ax, counts, lists, title):
    n_bin = counts['Binary']
    n_sin = counts['Single']
    if (n_bin + n_sin) > 0:
        ax.pie([n_bin, n_sin],
               labels=[f"Binary\n{n_bin}", f"Single\n{n_sin}"],
               colors=['cornflowerblue', 'lightgray'],
               autopct='%1.1f%%', startangle=90,
               textprops={'fontsize': S["FS_PIE_LABEL"]})
    else:
        ax.text(0.5, 0.5, "No Data", ha='center', va='center', fontsize=S["FS_PIE_LABEL"])

    ax.set_title(title, fontsize=S["FS_PIE_LABEL"] + 2, fontweight='bold')

    def _wrp(l):
        return "\n".join(textwrap.wrap(", ".join(l), width=20)) if l else ""

    bin_txt = _wrp(lists['Binary'])
    sin_txt = _wrp(lists['Single'])

    ax.text(0.05, -0.05, r"$\bf{Binary:}$", transform=ax.transAxes,
            ha='left', va='top', fontsize=S["FS_LIST_TEXT"])
    if bin_txt:
        ax.text(0.05, -0.12, bin_txt, transform=ax.transAxes,
                ha='left', va='top', fontsize=S["FS_LIST_TEXT"])

    ax.text(0.55, -0.05, r"$\bf{Single:}$", transform=ax.transAxes,
            ha='left', va='top', fontsize=S["FS_LIST_TEXT"])
    if sin_txt:
        ax.text(0.55, -0.12, sin_txt, transform=ax.transAxes,
                ha='left', va='top', fontsize=S["FS_LIST_TEXT"])

plot_pie_with_lists(axes[0], all_counts, all_lists, "All Stars Together")
plot_pie_with_lists(axes[1], clean_counts, clean_lists, "Clean Stars")
plot_pie_with_lists(axes[2], contam_counts, contam_lists, "Contaminated Stars")

plt.tight_layout()
plt.show()


## showing the final results with errors and fraction on the nice background of a normalized spectra

In [ ]:
%matplotlib widget
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from itertools import cycle

# ----------------------------------------------------------------------
#  1. SETUP & CONFIG
# ----------------------------------------------------------------------
SETTINGS_PATH = "ccf_settings_with_global_lines.json"
try:
    with open(SETTINGS_PATH, "r") as f:
        SETTINGS = json.load(f)
except FileNotFoundError:
    SETTINGS = {}

STAR_CFG = {s["star_name"]: s for s in SETTINGS.get("stars", [])}
LINES_DEFAULT = SETTINGS.get("emission_lines_default", {}) 

# Analysis Constants
THRESH_KMS = 20.0
EXTRA_ABOVE = 3
FAIL_FRAC_THRESH = 0.40
KNOWN_ERR_KEYS = ("full_RV_err", "full_err", "sigma", "err", "RV_err", "RV_sigma")

# Plotting Constants
line_centre = {tag: 10.0 * np.mean(rng) for tag, rng in LINES_DEFAULT.items()}
epoch_ew_data = {}        
ew_fail_stats = {}   
drverr_map = {} 

# ----------------------------------------------------------------------
#  2. DATA PROCESSING HELPERS
# ----------------------------------------------------------------------

def _extract_full_rv(cell):
    try:
        if isinstance(cell, dict): return cell.get('full_RV', None)
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict): return v.get('full_RV', None)
        if hasattr(cell, "get"): return cell.get('full_RV', None)
    except Exception: pass
    return None

def _extract_full_rv_err(cell):
    try:
        if isinstance(cell, dict):
            for k in KNOWN_ERR_KEYS:
                if k in cell and cell[k] is not None: return float(cell[k])
        if hasattr(cell, "item"):
            v = cell.item()
            if isinstance(v, dict):
                for k in KNOWN_ERR_KEYS:
                    if k in v and v[k] is not None: return float(v[k])
        if hasattr(cell, "get"):
            for k in KNOWN_ERR_KEYS:
                val = cell.get(k, None)
                if val is not None: return float(val)
    except Exception: pass
    return np.nan

def _is_epoch_skipped_for_line(star_name, line_key, ep):
    cfg = STAR_CFG.get(star_name, {})
    if ep in set(cfg.get("skip_epochs", [])): return True
    sl = cfg.get("skip_emission_lines", {})
    if line_key in sl:
        skip_eps = sl[line_key]
        if isinstance(skip_eps, (int, np.integer)): skip_eps = [skip_eps]
        if 0 in skip_eps or ep in skip_eps: return True
    return False

def _ew_record_for(star, epoch_num, line_key):
    try: EWs = star.load_property('EWs', epoch_num, 'COMBINED')
    except Exception: return None
    if EWs is None: return None
    rec = EWs.get(line_key)
    if rec is None: return None
    if hasattr(rec, 'item'): rec = rec.item()
    if not isinstance(rec, dict): return None
    return {"detected": rec.get("detected")}

def _compute_ew_fail_stats(star, line_key):
    sname = star.star_name
    considered = []
    failed = 0
    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep): continue
        considered.append(ep)
        rec = _ew_record_for(star, ep, line_key)
        if rec and rec.get("detected") is False:
            failed += 1
    n_considered = len(considered)
    frac = (failed / n_considered) if n_considered > 0 else 0.0
    return failed, n_considered, frac

def _line_delta_rv_for_star(star, line_key):
    rv_vals = []
    sname = star.star_name
    for ep in star.get_all_epoch_numbers():
        if _is_epoch_skipped_for_line(sname, line_key, ep): continue
        RVs = star.load_property('RVs', ep, 'COMBINED')
        if not RVs or line_key not in RVs: continue
        cell = RVs[line_key]
        rv = _extract_full_rv(cell)
        if rv is None: continue
        try: rv = float(rv)
        except Exception: continue
        err = _extract_full_rv_err(cell)
        rv_vals.append((ep, rv, err))

    if len(rv_vals) < 2:
        drverr_map[(sname, line_key)] = np.nan
        return np.nan, np.nan

    t_min = min(rv_vals, key=lambda t: t[1])
    t_max = max(rv_vals, key=lambda t: t[1])
    dRV = abs(t_max[1] - t_min[1])
    if dRV <= 0: dRV = 1e-3

    if np.isfinite(t_min[2]) and np.isfinite(t_max[2]):
        sigma_A = float(np.sqrt(t_min[2]**2 + t_max[2]**2))
    else:
        sigma_A = np.nan

    drverr_map[(sname, line_key)] = sigma_A
    return dRV, sigma_A

def _is_significant_binary(star_name, line_key, drv_val):
    if not (pd.notna(drv_val) and np.isfinite(drv_val)): return False
    sigma_A = drverr_map.get((star_name, line_key), np.nan)
    if not np.isfinite(sigma_A): return False
    return (float(drv_val) >= THRESH_KMS) and ((float(drv_val) - THRESH_KMS) >= 4.0 * float(sigma_A))

# ----------------------------------------------------------------------
#  3. MAIN PROCESSING
# ----------------------------------------------------------------------
records = []
target_list = list(STAR_CFG.keys())
ordered_lines = list(LINES_DEFAULT.keys())

for star_name in target_list:
    try:
        star = obs.load_star_instance(star_name, to_print=False)
    except Exception: continue
    
    for lk in ordered_lines:
        _, _, fail_frac = _compute_ew_fail_stats(star, lk)
        if fail_frac > FAIL_FRAC_THRESH: continue

        dRV, dRV_err = _line_delta_rv_for_star(star, lk)
        if pd.isna(dRV): continue
        
        is_bin = _is_significant_binary(star_name, lk, dRV)
        records.append({
            'star': star_name, 'emission': lk, 'λ_Å': line_centre.get(lk, np.nan),
            'ΔRV_km_s': dRV, 'ΔRV_err': dRV_err, 'is_binary': is_bin
        })

df = pd.DataFrame(records)

# ----------------------------------------------------------------------
#  4. PLOTTING
# ----------------------------------------------------------------------
try:
    s16_name = target_list[16] if len(target_list) > 16 else target_list[0]
    s16 = obs.load_star_instance(s16_name)
    s16.plot_normalized_spectra(bands='COMBINED', compare=False, separation=0, bin_window=0,
                                bary_correction=False, epoch_nums=[4], add_RV_emission_lines=True,
                                color_combined=True, add_continuum=True)
    ax_spec = plt.gca()
    
    # --- TITLE FIX: Clear the auto-generated title immediately ---
    ax_spec.set_title("")

    # Unit Fix (nm -> Å)
    lines = ax_spec.get_lines()
    unit_fixed = False
    for line in lines:
        x_data = line.get_xdata()
        if len(x_data) > 0 and np.max(x_data) < 3000:
            line.set_xdata(x_data * 10.0)
            unit_fixed = True
    if unit_fixed:
        ax_spec.relim(); ax_spec.autoscale_view()
        
    ax_spec.set_ylim(-1, 6)
    ax_spec.set_ylabel(f"Normalized Flux ({s16_name})", color='gray')
    ax_spec.set_xlabel("Wavelength [Å]")
    for line in lines:
        line.set_alpha(0.3); line.set_zorder(0)
except Exception:
    fig, ax_spec = plt.subplots(figsize=(14,8)); ax_spec.set_ylim(-1, 6)
    ax_spec.set_title("")

# --- Foreground RVs ---
ax_rv = ax_spec.twinx()
ax_rv.set_yscale('log')

for _, row in df.iterrows():
    x, y, yerr, star = row['λ_Å'], row['ΔRV_km_s'], row['ΔRV_err'], row['star']
    c = 'green' if row['is_binary'] else 'red'
    ax_rv.errorbar(x, y, yerr=yerr, fmt='o', color=c, ecolor=c, 
                   elinewidth=1.5, capsize=3, alpha=0.9, zorder=10, ms=5)
    ax_rv.text(x + 50, y, star, fontsize=8, color=c, ha='left', va='center', zorder=11)

ax_rv.axhline(THRESH_KMS, ls='--', lw=1, color='black', alpha=0.5)

# --- Stats & Labels ---
frac_dict = {}
for tag, grp in df.groupby('emission'):
    stars_this_line = grp['star'].unique()
    binary_stars = grp.loc[grp['is_binary'] == True, 'star'].unique()
    n_above = len(binary_stars)
    n_total = len(stars_this_line)
    frac = (n_above + EXTRA_ABOVE) / (n_total + EXTRA_ABOVE) if n_total else 0
    frac_dict[tag] = frac

if not df.empty:
    rv_min, rv_max = df['ΔRV_km_s'].min(), df['ΔRV_km_s'].max()
    log_min, log_max = np.log10(rv_min), np.log10(rv_max)
    pad = (log_max - log_min) * 0.15
    ax_rv.set_ylim(10**(log_min - pad), 10**(log_max + pad * 3))

for tag, frac in frac_dict.items():
    if tag not in line_centre: continue
    x = line_centre[tag]
    col_max = df[df['emission'] == tag]['ΔRV_km_s'].max()
    y_text = max(col_max, THRESH_KMS) * 1.5
    ax_rv.text(x, y_text, f"{frac*100:.0f}%", ha='center', va='bottom', 
               fontsize=10, fontweight='bold', color='black', zorder=12)

all_centers = list(line_centre.values())
if all_centers: ax_spec.set_xlim(min(all_centers)-1000, max(all_centers)+1500)

ax_rv.set_ylabel("|ΔRV| [km/s] (log)", fontsize=10)
# This title will now be the ONLY one
ax_rv.set_title("Peak-to-Peak RVs vs Emission Lines and $f_{bin}$")

plt.tight_layout()
plt.show()

# Plotting simulations to demonstrate high cadence spectroscopy

In [ ]:
# --- 1. Enable Interactive Widget Mode ---
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.patches as patches

# --- 2. System Parameters (WR 137) ---
Period_days = 4766.0      # 13.05 years
e = 0.178                 # Eccentricity
omega = np.radians(108)   # Argument of periastron
M_WR = 8.4                # Solar Masses
M_O = 19.8                # Solar Masses

# Scaling
a_total = 14.0 
a_O = a_total * (M_WR / (M_WR + M_O))    
a_WR = a_total * (M_O / (M_WR + M_O))    

# --- 3. Campaign Data (Corrected) ---
# We keep the Start BJD for the physics (correct orbital phase)
# But we will treat the time array as relative (0 to 15) for the animation
real_start_bjd = 8705
duration = 15 
n_frames = 16  # Exactly 16 points

# Relative days: 0, 1, 2, ..., 15
t_relative = np.linspace(0, duration, n_frames) 

# Absolute orbital time (for physics)
t_physics = real_start_bjd + t_relative

# Orbit path (high resolution for the background line)
t_orbit = np.linspace(0, Period_days, 1000)

# --- 4. Physics (Robust Kepler Solver) ---
def solve_kepler(M, e):
    # Ensure M is within 0 to 2pi to prevent drift
    M = M % (2 * np.pi)
    E = M.copy()
    for _ in range(15):
        E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
    return E

def get_pos(t, P, e, a, omega, offset_phase=0):
    # Mean Anomaly
    M = 2 * np.pi * t / P
    
    # Eccentric Anomaly
    E = solve_kepler(M, e)
    
    # True Anomaly (Robust Method using arctan2)
    # This prevents the "Jumping" by handling all quadrants correctly
    cos_nu = (np.cos(E) - e) / (1 - e * np.cos(E))
    sin_nu = (np.sqrt(1 - e**2) * np.sin(E)) / (1 - e * np.cos(E))
    nu = np.arctan2(sin_nu, cos_nu)
    
    r = a * (1 - e**2) / (1 + e * np.cos(nu))
    theta = nu + omega + offset_phase
    
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return x, y

# Calculate full orbit paths
wx_full, wy_full = get_pos(t_orbit, Period_days, e, a_WR, omega)
ox_full, oy_full = get_pos(t_orbit, Period_days, e, a_O, omega, np.pi)

# Calculate the 16 specific campaign points
wx_anim, wy_anim = get_pos(t_physics, Period_days, e, a_WR, omega)
ox_anim, oy_anim = get_pos(t_physics, Period_days, e, a_O, omega, np.pi)

# --- 5. Interactive Plot Setup ---
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(12, 10))

# Presentation Labels
ax.set_title(f"WR 137 Campaign (16 Spectra)", fontsize=16, color='white')
ax.set_xlabel("AU", fontsize=14)
ax.set_ylabel("AU", fontsize=14)
ax.set_aspect('equal')

# Static Elements (Full Orbit)
ax.plot(wx_full, wy_full, '--', color='magenta', alpha=0.2, lw=1, label='WR Orbit')
ax.plot(ox_full, oy_full, '--', color='cyan', alpha=0.2, lw=1, label='O Orbit')
ax.plot(0, 0, '+', color='white', markersize=10, label='Center of Mass')

# Highlight Box (Zoom Area)
x_min, x_max = min(wx_anim), max(wx_anim)
y_min, y_max = min(wy_anim), max(wy_anim)
pad = 0.5
rect = patches.Rectangle((x_min-pad, y_min-pad), (x_max-x_min)+2*pad, (y_max-y_min)+2*pad,
                         edgecolor='yellow', facecolor='none', linestyle=':', label='Campaign Window')
ax.add_patch(rect)

# Dynamic Elements
wr_star, = ax.plot([], [], 'o', color='magenta', markersize=12, markeredgecolor='white', label='WR 137')
o_star, = ax.plot([], [], 'o', color='cyan', markersize=15, markeredgecolor='white', label='O Companion')
spectra_trail, = ax.plot([], [], 'x', color='lime', markersize=10, markeredgewidth=2, label='Spectra Taken')
text_label = ax.text(0.05, 0.9, '', transform=ax.transAxes, color='lime', fontsize=14, fontweight='bold')

# --- 6. Animation Logic ---
def init():
    wr_star.set_data([], [])
    o_star.set_data([], [])
    spectra_trail.set_data([], [])
    text_label.set_text('')
    return wr_star, o_star, spectra_trail, text_label

def update(frame):
    # Current positions
    wx = wx_anim[frame]
    wy = wy_anim[frame]
    ox = ox_anim[frame]
    oy = oy_anim[frame]

    # Update Stars
    wr_star.set_data([wx], [wy])
    o_star.set_data([ox], [oy])
    
    # Update Spectra Trail (Historical points up to current frame)
    spectra_trail.set_data(wx_anim[:frame+1], wy_anim[:frame+1])
    
    # Update Text (using relative days 0-15)
    current_day = t_relative[frame]
    text_label.set_text(f"Day: {current_day:.0f}\nSpectra: {frame + 1}/16")
    
    return wr_star, o_star, spectra_trail, text_label

# Interval=500ms: Slower pace to see each day clearly
ani = FuncAnimation(fig, update, frames=n_frames, init_func=init, interval=500, repeat=True)

plt.legend(loc='lower right', fontsize=12)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- 1. System Parameters (V444 Cygni approximation) ---
M_WR = 9.3    # Solar Masses (Lighter)
M_O = 27.9    # Solar Masses (Heavier)
Period = 4.2  # Days
q = M_WR / M_O  # Mass ratio

# Distances from Center of Mass (CoM at 0,0)
# Arbitrary units where separation a = 1
a_total = 1.0 
a_O = a_total * (M_WR / (M_WR + M_O))  # Heavier star is closer to CoM
a_WR = a_total * (M_O / (M_WR + M_O))  # Lighter star is further

# --- 2. Observation Settings ---
# We simulate a high-cadence "burst" of observations
phase_start = 0.70          # Start at phase 0.7 (approaching quadrature)
phase_duration = 0.05       # Observe for 5% of the orbit (~5 hours)
n_exposures = 60            # 60 spectra taken in that time

# Time arrays
t_orbit = np.linspace(0, 1, 500) # Full orbit for background lines
t_obs = np.linspace(phase_start, phase_start + phase_duration, n_exposures)

# --- 3. Mechanics ---
def get_binary_positions(t):
    # Circular orbit approximation
    theta = 2 * np.pi * t
    # WR positions
    wx = a_WR * np.cos(theta)
    wy = a_WR * np.sin(theta)
    # O positions (opposite side, phase shift of pi)
    ox = a_O * np.cos(theta + np.pi)
    oy = a_O * np.sin(theta + np.pi)
    return wx, wy, ox, oy

# Pre-calculate full orbit paths (for context)
wx_full, wy_full, ox_full, oy_full = get_binary_positions(t_orbit)

# Pre-calculate specific observation moments
wx_obs, wy_obs, ox_obs, oy_obs = get_binary_positions(t_obs)

# --- 4. Plotting ---
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(8, 8))

# Fix the view to see the WHOLE system
limit = a_WR * 1.2
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_aspect('equal')
ax.set_title("V444 Cygni: High-Cadence Spectroscopy\n(Lighter WR Star vs Heavy O-Star)", fontsize=14, color='white')
ax.set_xlabel("AU (Arbitrary)", fontsize=12)

# Static Background Elements
# Center of Mass
ax.plot(0, 0, '+', color='white', markersize=10, markeredgewidth=2, label='Center of Mass')
# Orbit Trails
ax.plot(wx_full, wy_full, '--', color='magenta', alpha=0.3, label='WR Orbit')
ax.plot(ox_full, oy_full, '--', color='cyan', alpha=0.3, label='O Orbit')

# Dynamic Elements (initially empty)
wr_star, = ax.plot([], [], 'o', color='magenta', markersize=12, label='WR Star')
o_star, = ax.plot([], [], 'o', color='cyan', markersize=18, label='O Star') # O star is bigger/brighter
spectra_lines, = ax.plot([], [], 'x', color='lime', markersize=6, alpha=0.8, label='Spectra Taken')
bar_connector, = ax.plot([], [], '-', color='white', alpha=0.2, lw=1) # Visual bar connecting stars

def init():
    wr_star.set_data([], [])
    o_star.set_data([], [])
    spectra_lines.set_data([], [])
    bar_connector.set_data([], [])
    return wr_star, o_star, spectra_lines, bar_connector

def update(frame):
    # Get current positions
    wx = wx_obs[frame]
    wy = wy_obs[frame]
    ox = ox_obs[frame]
    oy = oy_obs[frame]
    
    # Update Stars
    wr_star.set_data([wx], [wy])
    o_star.set_data([ox], [oy])
    
    # Draw faint line connecting them to prove binary structure
    bar_connector.set_data([wx, ox], [wy, oy])
    
    # Update "Spectra" (Accumulate marks on the WR path)
    spectra_lines.set_data(wx_obs[:frame+1], wy_obs[:frame+1])
    
    return wr_star, o_star, spectra_lines, bar_connector

ani = FuncAnimation(fig, update, frames=n_exposures, init_func=init, blit=True, interval=100)

plt.legend(loc='upper right')
plt.close()

HTML(ani.to_jshtml())

# Plotting Binary Fraction vs Z

In [ ]:
# -----------------------------
# Metallicity anchors (as you requested)
# -----------------------------
Z = {
    "SMC": 0.2,
    "LMC": 0.5,
    "MW":  1.0,
}

# -----------------------------
# WC/WO (and related subtype) observed binary fractions
#   f_obs = k / n
# -----------------------------
wcwo_and_related = [
    # SMC: only WC/WO object is AB8, which is a binary
    {"galaxy": "SMC", "Z": Z["SMC"], "sample": "WC/WO (AB8 only)", "k": 1, "n": 1,
     "ref": "Schootemeijer+2024", "label": "SMC WC/WO"},

    # LMC WC/WO baseline from Bartzakos+01
    {"galaxy": "LMC", "Z": Z["LMC"], "sample": "WC/WO (definite binaries)", "k": 3, "n": 28,
     "ref": "Bartzakos+2001", "label": "LMC WC/WO (Bartzakos+01)"},

    # LMC WC/WO updated by you ("this work")
    # IMPORTANT: you stated f_obs = 0.46 and +10 new binaries, but (k, n) not uniquely defined from that alone.
    # Fill in k_thiswork and n_thiswork with YOUR final accounting used in the thesis.
    {"galaxy": "LMC", "Z": Z["LMC"], "sample": "WC/WO (this work)", "k": 13, "n": 28,
     "f_obs_reported": 0.46, "delta_new_binaries": 10,
     "ref": "this work (update to Bartzakos+2001)", "label": "LMC WC/WO (this work)"},

    # MW WC survey
    {"galaxy": "MW", "Z": Z["MW"], "sample": "WC", "k": 7, "n": 12,
     "ref": "Dsilva+2020", "label": "MW WC (Dsilva+20)"},
]

# -----------------------------
# MW WN subtypes from Dsilva surveys (all at Z=1)
# -----------------------------
mw_wn_subtypes = [
    {"galaxy": "MW", "Z": Z["MW"], "sample": "WNE", "k": 7, "n": 16,
     "ref": "Dsilva+2022", "label": "MW WNE (Dsilva+22)"},
    {"galaxy": "MW", "Z": Z["MW"], "sample": "WNL", "k": 4, "n": 11,
     "ref": "Dsilva+2023", "label": "MW WNL (Dsilva+23)"},
    {"galaxy": "MW", "Z": Z["MW"], "sample": "WN (WNE+WNL combined)", "k": 11, "n": 27,
     "ref": "Dsilva+2023", "label": "MW WN (combined, Dsilva+23)"},
    # Dsilva+2024 is a summary/compilation; no new (k,n) beyond the above for these lines
]

# -----------------------------
# Entire WR population (observed binaries / total known WR)
# -----------------------------
all_wr_population = [
    {"galaxy": "SMC", "Z": Z["SMC"], "sample": "All WR", "k": 5, "n": 12,
     "ref": "Schootemeijer+2024", "label": "SMC all WR"},
    {"galaxy": "LMC", "Z": Z["LMC"], "sample": "All WR", "k": 41, "n": 154,
     "ref": "Shenar+2019 (binaries)", "label": "LMC all WR"},
    {"galaxy": "MW",  "Z": Z["MW"],  "sample": "All WR (VIIth catalogue sample)", "k": 86, "n": 227,
     "ref": "van der Hucht+2001", "label": "MW all WR"},
]

# -----------------------------
# Convenience: compute f_obs where possible
# -----------------------------
def add_f_obs(points):
    out = []
    for p in points:
        q = dict(p)
        if q.get("k") is not None and q.get("n") not in (None, 0):
            q["f_obs"] = q["k"] / q["n"]
        out.append(q)
    return out

wcwo_and_related = add_f_obs(wcwo_and_related)
mw_wn_subtypes   = add_f_obs(mw_wn_subtypes)
all_wr_population = add_f_obs(all_wr_population)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import beta

# ==========================================
#        USER CONFIGURATION SECTION
# ==========================================

# 1. Figure & Axes
CONFIG_FIG = {
    "figsize": (10, 7.5),
    "ylim": (0, 120),
    "xlim": (0, 1.45),
    "title": "Binary Fraction of WC vs. WR Stars",
    "xlabel": "Metallicity ($Z/Z_{\odot}$)",
    "ylabel": "Observed Binary Fraction (%)"
}

# 2. Fonts
CONFIG_FONTS = {
    "title": 20,
    "axis_label": 18,
    "tick_label": 16,
    "label_big": 14,   # For headers like "LMC (This Work)"
    "label_small": 14, # For references
}

# 3. Error Bars (The "Caps")
CONFIG_ERRORS = {
    "capsize": 5,      # Width of the horizontal lines
    "capthick": 1.5,   # Thickness of the horizontal lines
    "elinewidth": 1.5, # Thickness of the vertical line
}

# 4. Markers
CONFIG_MARKERS = {
    "WR":        {"symbol": "s", "color": "red", "size": 10},
    "WC_Main":   {"symbol": "o", "color": "black",    "size": 12},
    "WC_Bartz":  {"symbol": "X", "color": "grey",     "size": 10, "edge_width": 1.5},
    "WC_MyWork": {"symbol": "o", "color": "black",    "size": 12},
}

# 5. Text Offsets (Adjust these to fix overlaps!)
# Format: (x_shift, y_shift)
CONFIG_OFFSETS = {
    "ratio_text":      (0.03, 2.0),   # "7/12" etc.
    "ref_wr":          (0.01, 3),   # WR references (right of square)
    
    # Specific Galaxy Headers
    "SMC_header":      (-0.08, -5.0), # Top Left
    "MW_header":       (0.05, 8.0),   # Top Right (pushed up)
    "MyWork_header":   (0.05, 2.0),   # Right
    "Bartzakos_label": (0.05, -2.0),  # Right (pushed down)
}

# 6. Green Arrow
CONFIG_ARROW = {
    "color": "green",
    "width": 2.5,
    "curvature": -0.3, # Negative curves left
    "text": "New RV Analysis"
}

# ==========================================
#           END CONFIGURATION
# ==========================================


# -----------------------------
# Helper: Error Calculation
# -----------------------------
def get_errors_percentage(k, n):
    """Calculate Clopper-Pearson 1-sigma error bars in PERCENTAGE."""
    if n is None or n == 0: return 0, 0
    confidence = 0.683
    alpha = 1 - confidence
    f = k / n
    lower = beta.ppf(alpha / 2, k, n - k + 1) if k > 0 else 0
    upper = beta.ppf(1 - alpha / 2, k + 1, n - k) if k < n else 1
    return (f - lower)*100, (upper - f)*100

# -----------------------------
# Plotting Logic
# -----------------------------
fig, ax = plt.subplots(figsize=CONFIG_FIG["figsize"])

# Variables to store arrow coordinates
bartzakos_coords = None
my_work_coords = None

# --- 1. Plot WR Population (Grey Squares) ---
for d in all_wr_population:
    z = d["Z"]
    
    if d.get("k") is not None and d.get("n") is not None:
        f_pct = (d["k"] / d["n"]) * 100
        err_low, err_high = get_errors_percentage(d["k"], d["n"])
        
        style = CONFIG_MARKERS["WR"]
        
        ax.errorbar(z, f_pct, yerr=[[err_low], [err_high]], 
                    fmt=style["symbol"], 
                    color=style["color"],      
                    ecolor=style["color"],     
                    capsize=CONFIG_ERRORS["capsize"],             
                    capthick=CONFIG_ERRORS["capthick"],          
                    elinewidth=CONFIG_ERRORS["elinewidth"],       
                    markersize=style["size"], 
                    markeredgecolor=style["color"]) 
        
        # Reference Label
        # Special check to shift LMC WR text slightly if needed
        x_off, y_off = CONFIG_OFFSETS["ref_wr"]
        if z == 0.5: x_off += 0.01 
        
        ax.text(z + x_off, f_pct + y_off, f"WR: {d['ref']}", 
                fontsize=CONFIG_FONTS["label_small"], color='grey', va='center', ha='left')

# --- 2. Plot WC Population ---
for d in wcwo_and_related:
    z = d["Z"]
    is_bartzakos = "Bartzakos" in d.get("ref", "") and "this work" not in d.get("ref", "").lower()
    is_my_work = "this work" in d.get("ref", "").lower()
    
    # Setup data based on config
    if is_my_work:
        f_pct = d.get("f_obs_reported", 0.46) * 100
        err_low, err_high = 9.0, 9.0
        my_work_coords = (z, f_pct)
        style = CONFIG_MARKERS["WC_MyWork"]
        edge_w = 0 # standard dot
    elif d.get("k") is not None:
        f_pct = (d["k"] / d["n"]) * 100
        err_low, err_high = get_errors_percentage(d["k"], d["n"])
        if is_bartzakos:
            style = CONFIG_MARKERS["WC_Bartz"]
            edge_w = style["edge_width"]
            bartzakos_coords = (z, f_pct)
        else:
            style = CONFIG_MARKERS["WC_Main"]
            edge_w = 0 # standard dot
    else:
        continue

    # Plot
    ax.errorbar(z, f_pct, yerr=[[err_low], [err_high]], 
                fmt=style["symbol"], 
                color=style["color"], 
                ecolor=style["color"],
                capsize=CONFIG_ERRORS["capsize"],             
                capthick=CONFIG_ERRORS["capthick"],          
                elinewidth=CONFIG_ERRORS["elinewidth"],        
                markersize=style["size"], 
                markeredgecolor=style["color"],
                markeredgewidth=edge_w if edge_w > 0 else None)   

    # --- Annotations ---
    
    # Ratio Labels (e.g. "1/1")
    if d.get("k") is not None:
        ratio_str = f"{d['k']}/{d['n']}"
        rx, ry = CONFIG_OFFSETS["ratio_text"]
        # MW Ratio Exception: Push it higher
        if z == 1.0: ry += 1.0 
        
        ax.text(z + rx, f_pct + ry, ratio_str, 
                fontsize=CONFIG_FONTS["label_big"], color='black', weight='bold')

    # Galaxy Headers
    if z == 0.2 and "SMC" in d.get("label", ""):
        ox, oy = CONFIG_OFFSETS["SMC_header"]
        ax.text(z + ox, f_pct + oy, "SMC\n($Z \sim 0.2 Z_{\odot}$)", 
                fontsize=CONFIG_FONTS["label_big"], ha='right', va='top', color='black')
                
    if z == 1.0 and "MW" in d.get("label", ""):
        ox, oy = CONFIG_OFFSETS["MW_header"]
        ax.text(z + ox, f_pct + oy, "MW\n($Z \sim 1.0 Z_{\odot}$)\nDsilva+20", 
                fontsize=CONFIG_FONTS["label_big"], ha='left', va='bottom', color='black')
                
    if is_my_work:
        ox, oy = CONFIG_OFFSETS["MyWork_header"]
        ax.text(z + ox, f_pct + oy, "LMC (This Work)\n($Z \sim 0.5 Z_{\odot}$)", 
                fontsize=CONFIG_FONTS["label_big"], ha='left', va='top', color='black')
                
    if is_bartzakos:
        ox, oy = CONFIG_OFFSETS["Bartzakos_label"]
        ax.text(z + ox, f_pct + oy, f"Bartzakos+01 (~{int(f_pct)}%)", 
                fontsize=CONFIG_FONTS["label_small"], ha='left', va='top', color='grey')


# --- 3. Green Arrow ---
if bartzakos_coords and my_work_coords:
    arrow_props = dict(arrowstyle="-|>", color=CONFIG_ARROW["color"], 
                       lw=CONFIG_ARROW["width"], 
                       connectionstyle=f"arc3,rad={CONFIG_ARROW['curvature']}")
    
    ax.annotate("", 
                xy=my_work_coords, 
                xytext=bartzakos_coords,
                arrowprops=arrow_props)
    
    mid_x = (bartzakos_coords[0] + my_work_coords[0]) / 2
    mid_y = (bartzakos_coords[1] + my_work_coords[1]) / 2
    
    ax.text(mid_x - 0.06, mid_y, CONFIG_ARROW["text"], 
            color=CONFIG_ARROW["color"], rotation=90, fontsize=CONFIG_FONTS["label_big"], 
            ha='center', va='center', weight='bold')

# --- 4. Final Formatting ---
ax.set_title(CONFIG_FIG["title"], fontsize=CONFIG_FONTS["title"], pad=15)
ax.set_ylabel(CONFIG_FIG["ylabel"], fontsize=CONFIG_FONTS["axis_label"])
ax.set_xlabel(CONFIG_FIG["xlabel"], fontsize=CONFIG_FONTS["axis_label"])

ax.set_ylim(CONFIG_FIG["ylim"])
ax.set_xlim(CONFIG_FIG["xlim"])

ax.grid(True, linestyle='--', alpha=0.5, color='lightgrey')
ax.tick_params(labelsize=CONFIG_FONTS["tick_label"])

# Custom Legend
legend_elements = [
    Line2D([0], [0], marker=CONFIG_MARKERS["WR"]["symbol"], color='w', 
           markerfacecolor=CONFIG_MARKERS["WR"]["color"], 
           markersize=CONFIG_MARKERS["WR"]["size"], label='WR Population'),
           
    Line2D([0], [0], marker=CONFIG_MARKERS["WC_Main"]["symbol"], color='w', 
           markerfacecolor=CONFIG_MARKERS["WC_Main"]["color"], 
           markersize=CONFIG_MARKERS["WC_Main"]["size"], label='WC Population (Current)'),
           
    Line2D([0], [0], marker=CONFIG_MARKERS["WC_Bartz"]["symbol"], color='w', 
           markeredgecolor=CONFIG_MARKERS["WC_Bartz"]["color"], 
           markeredgewidth=CONFIG_MARKERS["WC_Bartz"]["edge_width"],
           markersize=CONFIG_MARKERS["WC_Bartz"]["size"], label='LMC WC (Bartzakos+01)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=CONFIG_FONTS["tick_label"], framealpha=1.0)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import beta

# -----------------------------
# Helper: Error Calculation
# -----------------------------
def get_errors_percentage(k, n):
    """Calculate Clopper-Pearson 1-sigma error bars in PERCENTAGE."""
    if n is None or n == 0: return 0, 0
    confidence = 0.683
    alpha = 1 - confidence
    f = k / n
    lower = beta.ppf(alpha / 2, k, n - k + 1) if k > 0 else 0
    upper = beta.ppf(1 - alpha / 2, k + 1, n - k) if k < n else 1
    return (f - lower)*100, (upper - f)*100

# ==========================================
#        USER CONFIGURATION SECTION
# ==========================================

# 1. Figure & Axes
CONFIG_FIG = {
    "figsize": (10, 7.5),
    "ylim": (0, 120),
    "xlim": (0, 1.45),
    "title": "Binary Fraction of WC Stars",
    "xlabel": "Metallicity ($Z/Z_{\odot}$)",
    "ylabel": "Observed Binary Fraction (%)"
}

# 2. Fonts
CONFIG_FONTS = {
    "title": 20,
    "axis_label": 18,
    "tick_label": 16,
    "label_big": 14,   
    "label_small": 14, 
}

# 3. Error Bars
CONFIG_ERRORS = {
    "capsize": 5,      
    "capthick": 1.5,   
    "elinewidth": 1.5, 
}

# 4. Markers
CONFIG_MARKERS = {
    "WC_Main":   {"symbol": "o", "color": "black",    "size": 12},
    "WC_Bartz":  {"symbol": "X", "color": "grey",     "size": 10, "edge_width": 1.5},
    "WC_MyWork": {"symbol": "o", "color": "black",    "size": 12},
}

# 5. Text Offsets (x_shift, y_shift)
CONFIG_OFFSETS = {
    "ratio_text":      (0.03, 2.0),   
    "SMC_header":      (-0.08, -5.0), 
    "MW_header":       (0.05, 5.0),   
    "MyWork_header":   (0.05, 2.0),   
    "Bartzakos_label": (0.05, -2.0),  
}

# 6. Green Arrow
CONFIG_ARROW = {
    "color": "green",
    "width": 2.5,
    "curvature": -0.3, 
    "text": "New RV Analysis"
}

# ==========================================
#        PLOT GENERATION
# ==========================================

fig, ax = plt.subplots(figsize=CONFIG_FIG["figsize"])
bartzakos_coords = None
my_work_coords = None

# --- Plot WC Population Only ---
for d in wcwo_and_related:
    z = d["Z"]
    is_bartzakos = "Bartzakos" in d.get("ref", "") and "this work" not in d.get("ref", "").lower()
    is_my_work = "this work" in d.get("ref", "").lower()
    
    # 1. Determine Style & Data
    if is_my_work:
        f_pct = d.get("f_obs_reported", 0.46) * 100
        err_low, err_high = 9.0, 9.0
        my_work_coords = (z, f_pct)
        style = CONFIG_MARKERS["WC_MyWork"]
        edge_w = 0 
    elif d.get("k") is not None:
        f_pct = (d["k"] / d["n"]) * 100
        err_low, err_high = get_errors_percentage(d["k"], d["n"])
        if is_bartzakos:
            style = CONFIG_MARKERS["WC_Bartz"]
            edge_w = style["edge_width"]
            bartzakos_coords = (z, f_pct)
        else:
            style = CONFIG_MARKERS["WC_Main"]
            edge_w = 0 
    else:
        continue

    # 2. Plot Error Bar & Marker
    ax.errorbar(z, f_pct, yerr=[[err_low], [err_high]], 
                fmt=style["symbol"], 
                color=style["color"], 
                ecolor=style["color"],
                capsize=CONFIG_ERRORS["capsize"],             
                capthick=CONFIG_ERRORS["capthick"],          
                elinewidth=CONFIG_ERRORS["elinewidth"],        
                markersize=style["size"], 
                markeredgecolor=style["color"],
                markeredgewidth=edge_w if edge_w > 0 else None)   

    # 3. Add Ratio Text (e.g., "1/1")
    if d.get("k") is not None:
        ratio_str = f"{d['k']}/{d['n']}"
        rx, ry = CONFIG_OFFSETS["ratio_text"]
        # MW Ratio adjustment
        if z == 1.0: ry += 1.0 
        ax.text(z + rx, f_pct + ry, ratio_str, 
                fontsize=CONFIG_FONTS["label_big"], color='black', weight='bold')

    # 4. Add Galaxy/Source Headers
    # SMC
    if z == 0.2 and "SMC" in d.get("label", ""):
        ox, oy = CONFIG_OFFSETS["SMC_header"]
        ax.text(z + ox, f_pct + oy, "SMC\n($Z \sim 0.2 Z_{\odot}$)", 
                fontsize=CONFIG_FONTS["label_big"], ha='right', va='top', color='black')
    
    # MW
    if z == 1.0 and "MW" in d.get("label", ""):
        ox, oy = CONFIG_OFFSETS["MW_header"]
        ax.text(z + ox, f_pct + oy, "MW\n($Z \sim 1.0 Z_{\odot}$)\nDsilva+20", 
                fontsize=CONFIG_FONTS["label_big"], ha='left', va='bottom', color='black')
    
    # This Work (LMC)
    if is_my_work:
        ox, oy = CONFIG_OFFSETS["MyWork_header"]
        ax.text(z + ox, f_pct + oy, "LMC (This Work)\n($Z \sim 0.5 Z_{\odot}$)", 
                fontsize=CONFIG_FONTS["label_big"], ha='left', va='top', color='black')
    
    # Bartzakos (LMC)
    if is_bartzakos:
        ox, oy = CONFIG_OFFSETS["Bartzakos_label"]
        ax.text(z + ox, f_pct + oy, f"Bartzakos+01 (~{int(f_pct)}%)", 
                fontsize=CONFIG_FONTS["label_small"], ha='left', va='top', color='grey')

# --- Add Green Arrow ---
if bartzakos_coords and my_work_coords:
    arrow_props = dict(arrowstyle="-|>", color=CONFIG_ARROW["color"], 
                       lw=CONFIG_ARROW["width"], 
                       connectionstyle=f"arc3,rad={CONFIG_ARROW['curvature']}")
    
    ax.annotate("", xy=my_work_coords, xytext=bartzakos_coords, arrowprops=arrow_props)
    
    mid_x = (bartzakos_coords[0] + my_work_coords[0]) / 2
    mid_y = (bartzakos_coords[1] + my_work_coords[1]) / 2
    ax.text(mid_x - 0.06, mid_y, CONFIG_ARROW["text"], 
            color=CONFIG_ARROW["color"], rotation=90, fontsize=CONFIG_FONTS["label_big"], 
            ha='center', va='center', weight='bold')

# --- Final Formatting ---
ax.set_title(CONFIG_FIG["title"], fontsize=CONFIG_FONTS["title"], pad=15)
ax.set_ylabel(CONFIG_FIG["ylabel"], fontsize=CONFIG_FONTS["axis_label"])
ax.set_xlabel(CONFIG_FIG["xlabel"], fontsize=CONFIG_FONTS["axis_label"])
ax.set_ylim(CONFIG_FIG["ylim"])
ax.set_xlim(CONFIG_FIG["xlim"])
ax.grid(True, linestyle='--', alpha=0.5, color='lightgrey')
ax.tick_params(labelsize=CONFIG_FONTS["tick_label"])

# Custom Legend
legend_elements = [
    Line2D([0], [0], marker=CONFIG_MARKERS["WC_Main"]["symbol"], color='w', 
           markerfacecolor=CONFIG_MARKERS["WC_Main"]["color"], 
           markersize=CONFIG_MARKERS["WC_Main"]["size"], label='WC Population (Current)'),
    Line2D([0], [0], marker=CONFIG_MARKERS["WC_Bartz"]["symbol"], color='w', 
           markeredgecolor=CONFIG_MARKERS["WC_Bartz"]["color"], 
           markeredgewidth=CONFIG_MARKERS["WC_Bartz"]["edge_width"],
           markersize=CONFIG_MARKERS["WC_Bartz"]["size"], label='LMC WC (Bartzakos+01)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=CONFIG_FONTS["tick_label"], framealpha=1.0)

plt.tight_layout()
plt.show()

# Plotting the normalized flux when trying to take the 2D image and throw the top and bottom panes, sun along the vertical axis and normalized using what I chose for the 1D flux. And comparing to my normalization

In [ ]:
obs = obsm()
star_name = specs.star_names[12]
print(star_name)
star = obs.load_star_instance(star_name)
epoch_num = 4

In [ ]:
data = []
for star_name in specs.star_names:
    star = obs.load_star_instance(star_name)
    data.append([star_name])

df = pd.DataFrame(data, columns=["Star Name"])
print(df)

In [ ]:
%matplotlib widget
obs = obsm()
star_name = specs.star_names[0]
star = obs.load_star_instance(star_name)
try:
    for epoch_num in range(1,8):
        for band in ['UVB','VIS','NIR']:
            star.clean_flux_and_normalize_interactive(epoch_num,band)

In [ ]:
%matplotlib widget
star.clean_flux_and_normalize(epoch_num,'VIS')

In [ ]:
%matplotlib widget
star.clean_flux_and_normalize(epoch_num,'NIR')